<a href="https://colab.research.google.com/github/PapLion/Akemi/blob/main/NeoTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tokenizer




### Libreria

In [ ]:
"""NeoToken: Librería de Inferencia (Versión Mejorada)
=====================================================

Librería optimizada para inferencia y razonamiento multiescala.
Combina las mejores características de todas las versiones:
- Base: train/train.py (arquitectura completa)
- Mejoras: library/library.py (fallback seguro, lossless decode)
- Correcciones: Bug fixes y patrones AST completos

Soporta enrutamiento dinámico entre Texto, Código (AST) y Ciencia (LaTeX).

Cambios respecto a la versión anterior:
- BUG FIX: _encode_ast ahora usa _extract_ast_patterns() (igual que en entrenamiento),
  recuperando los patrones enriquecidos FUNC:Nargs, LOOP:for, COND:if.
  Antes se usaba ast.walk directo → muchos [UNK] en código.
- BUG FIX: _encode_science_latex tenía backslash+/ en el regex (inválido). Corregido a '/'.
- BUG FIX: _get_safe_char_tokens retornaba temprano para espacio/▁ sin hacer el
  fallback a bytes si [C: ] tampoco existía en el vocab.
- BUG FIX: _encode_dynamic_stream usaba re.split sin re.DOTALL, fallando en
  ecuaciones LaTeX multilinea ($$...$$). Corregido con flag re.DOTALL.
- BUG FIX: Detección de código mejorada en _encode_dynamic_stream para evitar
  falsos positivos (e.g. "return" en texto natural).
- MEJORA: Añadido vocab_size como property (necesario para capas de embedding del LM).
- MEJORA: Añadido __len__, __repr__ y método get_scale_name() para inspección.
- MEJORA: _extract_ast_patterns extraído como método de clase (no duplicado).
"""

import os
import pickle
import re
import ast
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
from collections import Counter


@dataclass
class TokenScale:
    """Representa una escala de tokenización."""
    level: int  # 0=byte, 1=char, 2=subword, 3=word, 4=AST, 5=LATEX
    granularity: str
    vocab_size: int
    compression_ratio: float


class PatternMemory:
    """Memoria de patrones frecuentes para optimización de contexto."""

    def __init__(self, max_patterns: int = 10000):
        self.max_patterns = max_patterns
        self.patterns = Counter()
        self.context_cache = {}

    def observe(self, tokens: List[int], context: str):
        pattern = tuple(tokens[:10])
        self.patterns[pattern] += 1
        if self.patterns[pattern] > 10:
            self.context_cache[pattern] = context

    def get_frequent_patterns(self, top_k: int = 100) -> List[Tuple]:
        return self.patterns.most_common(top_k)

    def evolve(self):
        if len(self.patterns) > self.max_patterns:
            keep = dict(self.patterns.most_common(self.max_patterns))
            self.patterns = Counter(keep)


class NeoToken:
    """
    Motor de Tokenización Multi-Escala para inferencia.
    Soporta 6 escalas: byte, char, subword (BPE), word, AST, LaTeX.
    """

    def __init__(self,
                 target_vocab_size: int = 50000,
                 enable_ast_mode: bool = True,
                 enable_memory: bool = True):

        self.target_vocab_size = target_vocab_size
        self.enable_ast = enable_ast_mode
        self.enable_memory = enable_memory

        self.scales = {
            0: TokenScale(0, 'byte', 256, 1.0),
            1: TokenScale(1, 'char', 512, 1.0),
            2: TokenScale(2, 'subword', 25000, 3.5),
            3: TokenScale(3, 'word', 12000, 5.0),
            4: TokenScale(4, 'ast_node', 5000, 8.0),
            5: TokenScale(5, 'latex_math', 8000, 6.0),
        }

        self.special_tokens = {
            '[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4,
            '[CODE_START]': 5, '[CODE_END]': 6, '[STRUCT_START]': 7, '[STRUCT_END]': 8,
            '[MATH_START]': 9, '[MATH_END]': 10,
            '[SCALE_0]': 11, '[SCALE_1]': 12, '[SCALE_2]': 13,
            '[SCALE_3]': 14, '[SCALE_4]': 15, '[SCALE_5]': 16,
        }

        self.pattern_memory = PatternMemory() if enable_memory else None
        self.token_to_id: Dict[str, int] = {}
        self.id_to_token: Dict[int, str] = {}
        self.vocabs: Dict[int, dict] = {}
        self._is_trained = False

        # Caché de rendimiento (construidas post-load, no se persisten)
        self._bpe_cache: Dict[str, List[int]] = {}   # texto_segmento → ids
        self._subword_prefixes: set = set()           # prefijos válidos del vocab BPE

    # ------------------------------------------------------------------
    # Propiedades de utilidad
    # ------------------------------------------------------------------

    @property
    def vocab_size(self) -> int:
        """Tamaño real del vocabulario unificado. Necesario para capas de embedding."""
        return len(self.token_to_id)

    def __len__(self) -> int:
        return self.vocab_size

    def __repr__(self) -> str:
        status = "entrenado" if self._is_trained else "sin cargar"
        return f"NeoToken(vocab={self.vocab_size}, ast={self.enable_ast}, status={status})"

    def get_scale_name(self, scale_id: int) -> str:
        """Devuelve el nombre legible de una escala."""
        return self.scales.get(scale_id, TokenScale(-1, 'unknown', 0, 0.0)).granularity

    def _build_bpe_prefix_index(self):
        """
        Pre-computa el set de todos los prefijos válidos del vocabulario BPE.
        Permite cortar el loop greedy en cuanto ningún prefijo del texto actual
        puede matchear — elimina el spike de P95 en textos con muchos OOV.
        Coste: O(V * L_max) una sola vez al cargar. Prácticamente instantáneo.
        """
        self._subword_prefixes = set()
        self._bpe_cache = {}
        for key in self.token_to_id:
            if key.startswith('[S:'):
                subword = key[3:-1]
                # Añadir todos los prefijos de este subword al índice
                for length in range(1, len(subword) + 1):
                    self._subword_prefixes.add(subword[:length])

    # ------------------------------------------------------------------
    # Persistencia
    # ------------------------------------------------------------------

    def load(self, path: str):
        """Carga el binario del modelo (.pkl)."""
        if not os.path.exists(path):
            raise FileNotFoundError(f"❌ Archivo de modelo no encontrado: {path}")

        with open(path, 'rb') as f:
            state = pickle.load(f)

        self.__dict__.update(state)

        # Reconstruir TokenScale desde tuplas (compatibilidad con modelos antiguos)
        reconstructed_scales = {}
        for k, v in self.scales.items():
            if isinstance(v, TokenScale):
                reconstructed_scales[k] = v
            elif isinstance(v, tuple):
                reconstructed_scales[k] = TokenScale(*v)
            else:
                raise TypeError(
                    f"Tipo inesperado para scales[{k}]: {type(v)}. Se esperaba TokenScale o tuple."
                )
        self.scales = reconstructed_scales

        # Reconstruir PatternMemory desde dict o instancia (compatibilidad)
        if self.pattern_memory is not None:
            if isinstance(self.pattern_memory, dict):
                pm_state = self.pattern_memory
            elif isinstance(self.pattern_memory, PatternMemory):
                pm_state = self.pattern_memory.__dict__
            else:
                print(f"⚠️ Tipo inesperado para pattern_memory: {type(self.pattern_memory)}. Se omite.")
                pm_state = None

            if pm_state is not None:
                new_pm = PatternMemory()
                new_pm.__dict__.update(pm_state)
                self.pattern_memory = new_pm

        self._is_trained = True
        self._build_bpe_prefix_index()
        print(f"✅ NeoToken cargado: {self.vocab_size} tokens en vocabulario.")

    def save(self, path: str):
        """Guarda el estado actual del tokenizador."""
        state = {
            'target_vocab_size': self.target_vocab_size,
            'enable_ast': self.enable_ast,
            'enable_memory': self.enable_memory,
            # Guardamos como tuplas para máxima compatibilidad entre versiones
            'scales': {k: (v.level, v.granularity, v.vocab_size, v.compression_ratio)
                       for k, v in self.scales.items()},
            'vocabs': self.vocabs,
            'special_tokens': self.special_tokens,
            'token_to_id': self.token_to_id,
            'id_to_token': self.id_to_token,
            'pattern_memory': self.pattern_memory.__dict__ if self.pattern_memory else None,
        }
        with open(path, 'wb') as f:
            pickle.dump(state, f)
        print(f"💾 NeoToken guardado exitosamente en {path}")

    # ------------------------------------------------------------------
    # Núcleo de codificación
    # ------------------------------------------------------------------

    def encode(self, text: str,
               preferred_scale: Optional[int] = None,
               adaptive: bool = True) -> List[int]:
        """
        Codifica texto en IDs.

        Args:
            text: Texto a codificar.
            preferred_scale: Si se especifica (0-5), fuerza esa escala.
            adaptive: Si True (default), usa enrutamiento dinámico por segmento.
                      Si False, analiza el texto completo y elige una escala global.
        """
        if not self._is_trained:
            raise ValueError("NeoToken no ha sido cargado o entrenado. Llama a .load() primero.")

        if preferred_scale is not None:
            return self._dispatch_encoder(text, preferred_scale)

        if adaptive:
            return self._encode_dynamic_stream(text)

        # Modo estático: elige escala global según complejidad del texto completo
        complexity = self._analyze_complexity(text)
        scale = self._select_optimal_scale(text, complexity)
        return self._dispatch_encoder(text, scale)

    def _dispatch_encoder(self, text: str, scale: int) -> List[int]:
        if scale == 0: return self._encode_bytes(text)
        elif scale == 1: return self._encode_chars(text)
        elif scale == 2: return self._encode_subwords(text)
        elif scale == 3: return self._encode_words(text)
        elif scale == 4: return self._encode_ast(text)
        elif scale == 5: return self._encode_science_latex(text)
        return []

    def _encode_dynamic_stream(self, text: str) -> List[int]:
        """
        Segmentación inteligente: cambia de escala en tiempo real por segmento.

        Orden de prioridad por segmento:
          1. LaTeX inline/display ($...$ o $$...$$)  → Escala 5
          2. Código Python parseable (heurística estricta) → Escala 4
          3. Texto general                             → Escala 2 (subword)

        FIX: re.DOTALL para capturar bloques $$...$$ multilinea.
        FIX: Heurística de código más estricta para evitar falsos positivos.
        """
        tokens = []

        # re.DOTALL para que . capture saltos de línea dentro de $$ bloques
        math_pattern = r'(\$\$.*?\$\$|\$[^\$\n]+?\$)'
        segments = re.split(math_pattern, text, flags=re.DOTALL)

        for segment in segments:
            if not segment:
                continue

            # --- Segmento LaTeX ---
            if segment.startswith('$'):
                tokens.extend(self._encode_science_latex(segment))
                continue

            # --- Detección de código (heurística más estricta) ---
            # Requiere al menos 2 indicadores fuertes o una definición explícita
            code_signals = [
                bool(re.search(r'^\s*(def |class |async def )\w', segment, re.MULTILINE)),
                bool(re.search(r'^\s*import\s+\w', segment, re.MULTILINE)),
                bool(re.search(r'^\s*from\s+\w+\s+import\s+', segment, re.MULTILINE)),
                bool(re.search(r'^\s*for\s+\w+\s+in\s+', segment, re.MULTILINE)),
                bool(re.search(r'^\s*if\s+.+:\s*$', segment, re.MULTILINE)),
                bool(re.search(r'^\s*return\s+', segment, re.MULTILINE)),
            ]
            is_code = code_signals[0] or sum(code_signals) >= 2

            if is_code and self.enable_ast:
                try:
                    ast_tokens = self._encode_ast(segment)
                    # Verificar que realmente se parseó como AST (no cayó a subword)
                    if ast_tokens and ast_tokens[0] == self.special_tokens['[SCALE_4]']:
                        tokens.extend(ast_tokens)
                        continue
                except Exception:
                    pass

            # --- Texto general: subword BPE ---
            tokens.extend(self._encode_subwords(segment))

        return tokens

    # ------------------------------------------------------------------
    # Decodificación
    # ------------------------------------------------------------------

    def _ast_pattern_to_code(self, pattern: str) -> str:
        """
        Convierte un patrón AST a código Python legible con nombres genéricos.
        Ingeniería inversa parcial: recupera estructura sintáctica pero no léxico.

        Principio de diseño:
          - Nodos ESTRUCTURALES (sentencias, definiciones) → emiten línea completa con newline
          - Nodos INTERNOS (operadores, contextos, tipos) → emiten '' para no ensuciar output
          - Nodos EXPRESIÓN suelta (Call, BinOp etc.) solo emiten si el estado indica
            que estamos dentro de una sentencia Expr (tracked via 'in_expr')

        Estado reseteado por decode() al inicio de cada llamada.
        """
        state = self._ast_decode_state
        ind   = '    ' * state['indent']  # indentación actual

        # ── Patrones enriquecidos ──────────────────────────────────────────────

        if pattern.startswith('FUNC:'):
            try:
                nargs = int(pattern.split(':')[1].replace('args', ''))
            except (IndexError, ValueError):
                nargs = 1
            fname = f"func_{state['func_count']}"
            state['func_count'] += 1
            args  = ', '.join(f"arg_{i}" for i in range(nargs))
            state['indent'] += 1
            return f"\n{ind}def {fname}({args}):\n"

        if pattern == 'LOOP:for':
            var = f"item_{state['var_count']}"
            state['var_count'] += 1
            state['indent'] += 1
            return f"\n{ind}for {var} in iterable:\n"

        if pattern == 'COND:if':
            state['indent'] += 1
            return f"\n{ind}if condition:\n"

        # ── Nodos estructurales (sentencias completas) ─────────────────────────

        stmt_ind = '    ' * state['indent']   # indentación dentro del bloque actual

        structural = {
            # Módulo — no emite nada, es el nodo raíz
            'NODE:Module':        '',
            'NODE:Interactive':   '',
            'NODE:Expression':    '',

            # Definiciones — FunctionDef ya manejado por FUNC:Nargs
            'NODE:FunctionDef':   '',
            'NODE:AsyncFunctionDef': '',
            'NODE:ClassDef':      f"\n{ind}class Class_{state['func_count']}:\n",

            # Sentencias simples
            'NODE:Return':        f"{stmt_ind}return result\n",
            'NODE:Pass':          f"{stmt_ind}pass\n",
            'NODE:Break':         f"{stmt_ind}break\n",
            'NODE:Continue':      f"{stmt_ind}continue\n",
            'NODE:Raise':         f"{stmt_ind}raise Exception()\n",
            'NODE:Assert':        f"{stmt_ind}assert condition\n",
            'NODE:Delete':        f"{stmt_ind}del var_{state['var_count']}\n",
            'NODE:Global':        f"{stmt_ind}global var\n",
            'NODE:Nonlocal':      f"{stmt_ind}nonlocal var\n",

            # Asignaciones
            'NODE:Assign':        f"{stmt_ind}var_{state['var_count']} = value_{state['var_count']}\n",
            'NODE:AugAssign':     f"{stmt_ind}var_{state['var_count']} += value_{state['var_count']}\n",
            'NODE:AnnAssign':     f"{stmt_ind}var_{state['var_count']}: Any = value_{state['var_count']}\n",

            # Imports
            'NODE:Import':        f"{stmt_ind}import module_{state['var_count']}\n",
            'NODE:ImportFrom':    f"{stmt_ind}from module_{state['var_count']} import name\n",

            # Control flow con bloques — For/If manejados por LOOP:/COND:
            'NODE:For':           '',
            'NODE:If':            '',
            'NODE:While':         f"\n{ind}while condition:\n",
            'NODE:AsyncFor':      f"\n{ind}async for item in iterable:\n",
            'NODE:With':          f"\n{ind}with context() as ctx:\n",
            'NODE:AsyncWith':     f"\n{ind}async with context() as ctx:\n",
            'NODE:Try':           f"\n{ind}try:\n",
            'NODE:ExceptHandler': f"\n{ind}except Exception as e:\n",

            # Expr es una sentencia que envuelve una expresión suelta (ej: llamada a función)
            'NODE:Expr':          f"{stmt_ind}func_{state['func_count']}()\n",
        }

        if pattern in structural:
            # Incrementar var_count para nodos que implican nuevas variables
            if pattern in ('NODE:Assign', 'NODE:AugAssign', 'NODE:AnnAssign',
                           'NODE:Delete', 'NODE:Import', 'NODE:ImportFrom'):
                state['var_count'] += 1
            return structural[pattern]

        # ── Nodos internos — no emiten texto visible ───────────────────────────
        # Estos son subnodos de expresiones, operadores, contextos, etc.
        # Emitirlos genera la sopa de "obj.attr_0a+bvar_1" que veías antes.
        silent = {
            # Acceso / nombres — son subnodos de Assign/Call/etc., no sentencias
            'NODE:Name', 'NODE:Load', 'NODE:Store', 'NODE:Del',
            'NODE:Attribute', 'NODE:Subscript', 'NODE:Starred',
            # Expresiones — subnodos, contexto ya emitido por NODE:Expr/Assign
            'NODE:Call', 'NODE:BinOp', 'NODE:UnaryOp', 'NODE:BoolOp',
            'NODE:Compare', 'NODE:IfExp', 'NODE:Lambda',
            'NODE:Await', 'NODE:Yield', 'NODE:YieldFrom',
            'NODE:JoinedStr', 'NODE:FormattedValue',
            # Literales — subnodos
            'NODE:Constant', 'NODE:List', 'NODE:Tuple',
            'NODE:Set', 'NODE:Dict',
            # Comprehensions
            'NODE:ListComp', 'NODE:SetComp', 'NODE:DictComp',
            'NODE:GeneratorExp', 'NODE:comprehension',
            # Argumentos / parámetros
            'NODE:arguments', 'NODE:arg', 'NODE:keyword',
            # Todos los operadores
            'NODE:Add',    'NODE:Sub',    'NODE:Mult',   'NODE:Div',
            'NODE:Mod',    'NODE:Pow',    'NODE:FloorDiv','NODE:MatMult',
            'NODE:BitAnd', 'NODE:BitOr',  'NODE:BitXor',
            'NODE:LShift', 'NODE:RShift',
            'NODE:And',    'NODE:Or',     'NODE:Not',
            'NODE:Invert', 'NODE:UAdd',   'NODE:USub',
            'NODE:Eq',     'NODE:NotEq',  'NODE:Lt',     'NODE:LtE',
            'NODE:Gt',     'NODE:GtE',    'NODE:Is',     'NODE:IsNot',
            'NODE:In',     'NODE:NotIn',
            # Misc internos
            'NODE:alias', 'NODE:withitem', 'NODE:match_case',
        }

        if pattern in silent:
            return ''

        # Patrón desconocido — silencioso para no corromper el output
        return ''

    def decode(self, ids: List[int]) -> str:
        """
        Decodificación lossless con soporte para emojis y formato exacto.
        No aplica .strip() para preservar saltos de línea y espacios iniciales/finales.
        Maneja correctamente múltiples marcadores [SCALE_X] del modo dinámico.
        """
        # Reset estado del decoder AST para cada llamada a decode()
        self._ast_decode_state = {'func_count': 0, 'var_count': 0, 'indent': 0}

        result = []
        byte_buffer = bytearray()

        for id_ in ids:
            token = self.id_to_token.get(id_, '[UNK]')

            # Ignorar todos los marcadores de escala (modo dinámico emite varios)
            if token.startswith('[SCALE_') or token in self.special_tokens:
                # Vaciar buffer de bytes pendiente antes de continuar
                if byte_buffer:
                    result.append(byte_buffer.decode('utf-8', errors='replace'))
                    byte_buffer = bytearray()
                continue

            # Acumulación de bytes (para emojis y caracteres multibyte)
            if token.startswith('[B:'):
                try:
                    byte_buffer.extend(bytes.fromhex(token[3:-1]))
                except ValueError:
                    pass
                continue

            # Vaciar buffer de bytes si llega un token de otra escala
            if byte_buffer:
                result.append(byte_buffer.decode('utf-8', errors='replace'))
                byte_buffer = bytearray()

            if token.startswith('[C:'):
                content = token[3:-1]
                if content == '▁':
                    content = ' '
                result.append(content)
            elif token.startswith('[S:'):
                result.append(token[3:-1].replace('▁', ' '))
            elif token.startswith('[W:'):
                result.append(token[3:-1])
            elif token.startswith('[A:'):
                pattern = token[3:-1]
                result.append(self._ast_pattern_to_code(pattern))
            elif token.startswith('[L:'):
                result.append(token[3:-1])
            # [UNK] y tokens especiales no reconocidos: se omiten silenciosamente

        # Vaciar cualquier byte que quedó al final
        if byte_buffer:
            result.append(byte_buffer.decode('utf-8', errors='replace'))

        return "".join(result)

    # ------------------------------------------------------------------
    # Encoders internos
    # ------------------------------------------------------------------

    def _get_safe_char_tokens(self, char: str) -> List[int]:
        """
        Busca el carácter en el vocabulario de chars.
        Si no existe, recurre silenciosamente a bytes (Escala 0).

        FIX: Ya no retorna temprano para espacio/▁ sin haber intentado el fallback a bytes.
        """
        # Normalizar ▁ a espacio para buscar en vocab
        lookup_char = ' ' if char == '▁' else char
        token_key = f"[C:{lookup_char}]"

        if token_key in self.token_to_id:
            return [self.token_to_id[token_key]]

        # Fallback a bytes (maneja emojis, caracteres especiales, etc.)
        return [
            self.token_to_id.get(f"[B:{bytes([b]).hex()}]", self.special_tokens['[UNK]'])
            for b in char.encode('utf-8')
        ]

    def _encode_bytes(self, text: str) -> List[int]:
        return [self.special_tokens['[SCALE_0]']] + [
            self.token_to_id.get(f"[B:{bytes([b]).hex()}]", self.special_tokens['[UNK]'])
            for b in text.encode('utf-8')
        ]

    def _encode_chars(self, text: str) -> List[int]:
        tokens = [self.special_tokens['[SCALE_1]']]
        for char in text:
            tokens.extend(self._get_safe_char_tokens(char))
        return tokens

    def _encode_subwords(self, text: str) -> List[int]:
        """
        BPE greedy optimizado con dos aceleraciones:

        1. Prefix index: antes de iterar longitudes decrecientes, verifica si
           text[i] siquiera existe como prefijo en el vocab. Si no, salta
           directo al fallback sin hacer ningún lookup. Elimina el O(20)
           inner loop en posiciones OOV — que es el origen del spike P95.

        2. Segment cache: cachea el resultado de segmentos ya vistos.
           Durante el entrenamiento del LM, los mismos fragmentos aparecen
           miles de veces. El cache los resuelve en O(1).
        """
        # Cache hit en el segmento completo
        cache_key = text
        if cache_key in self._bpe_cache:
            return [self.special_tokens['[SCALE_2]']] + self._bpe_cache[cache_key]

        inner_tokens: List[int] = []
        text_bpe = text.replace(' ', '▁')
        i = 0
        prefixes = self._subword_prefixes  # referencia local para velocidad

        while i < len(text_bpe):
            char = text_bpe[i]

            # Atajo rápido: si el carácter ni siquiera es prefijo de nada en BPE,
            # ir directo al fallback sin iterar longitudes
            if char not in prefixes:
                inner_tokens.extend(self._get_safe_char_tokens(char))
                i += 1
                continue

            found = False
            max_len = min(20, len(text_bpe) - i)
            for length in range(max_len, 0, -1):
                subword = text_bpe[i:i + length]
                # Poda adicional: si el subword completo no es prefijo posible, skip
                if length > 1 and subword not in prefixes:
                    continue
                token_key = f"[S:{subword}]"
                if token_key in self.token_to_id:
                    inner_tokens.append(self.token_to_id[token_key])
                    i += length
                    found = True
                    break
            if not found:
                inner_tokens.extend(self._get_safe_char_tokens(text_bpe[i]))
                i += 1

        # Guardar en cache (máximo 50k entradas para no inflar memoria)
        if len(self._bpe_cache) < 50000:
            self._bpe_cache[cache_key] = inner_tokens

        return [self.special_tokens['[SCALE_2]']] + inner_tokens

    def _encode_words(self, text: str) -> List[int]:
        """Codificación a nivel palabra con fallback a chars para OOV."""
        tokens = [self.special_tokens['[SCALE_3]']]
        matches = re.findall(r'(\w+)|(\s+)|([^\w\s])', text)
        for word, space, punct in matches:
            if word:
                token_key = f"[W:{word}]"
                if token_key in self.token_to_id:
                    tokens.append(self.token_to_id[token_key])
                else:
                    # OOV: descomponer en chars con fallback a bytes
                    for char in word:
                        tokens.extend(self._get_safe_char_tokens(char))
            elif space:
                for s in space:
                    tokens.extend(self._get_safe_char_tokens(s))
            elif punct:
                tokens.extend(self._get_safe_char_tokens(punct))
        return tokens

    def _extract_ast_patterns(self, tree) -> List[str]:
        """
        Extrae patrones enriquecidos del AST.
        CRÍTICO: Debe ser idéntico al usado durante el entrenamiento para que
        los patrones coincidan con el vocabulario del modelo (.pkl).

        Patrones generados:
          - NODE:<ClassName>      → todos los nodos
          - FUNC:<N>args          → funciones con N argumentos
          - LOOP:for              → bucles for
          - COND:if               → condicionales if
        """
        patterns = []
        for node in ast.walk(tree):
            patterns.append(f"NODE:{type(node).__name__}")
            if isinstance(node, ast.FunctionDef):
                patterns.append(f"FUNC:{len(node.args.args)}args")
            elif isinstance(node, ast.For):
                patterns.append("LOOP:for")
            elif isinstance(node, ast.If):
                patterns.append("COND:if")
        return patterns

    def _encode_ast(self, text: str) -> List[int]:
        """
        Codifica código Python usando nodos AST.
        FIX: Ahora usa _extract_ast_patterns() (igual que el entrenamiento),
        recuperando FUNC:Nargs, LOOP:for, COND:if. Antes se usaba ast.walk
        directo con solo NODE:X, generando muchos [UNK].
        Si el parsing falla, cae silenciosamente a subword.
        """
        tokens = [self.special_tokens['[SCALE_4]']]
        try:
            tree = ast.parse(text)
            patterns = self._extract_ast_patterns(tree)
            for pattern in patterns:
                tokens.append(
                    self.token_to_id.get(f"[A:{pattern}]", self.special_tokens['[UNK]'])
                )
        except SyntaxError:
            return self._encode_subwords(text)
        return tokens

    def _encode_science_latex(self, text: str) -> List[int]:
        """
        Codifica expresiones LaTeX/matemáticas/científicas.
        FIX: Regex corregido — se eliminó el operador backslash+/ (inválido) y se reemplazó por '/'.
        """
        tokens = [self.special_tokens['[SCALE_5]']]
        # Regex que captura: comandos LaTeX, operadores, dígitos, letras, espacios y resto
        pattern = r'(\\[a-zA-Z]+)|(\^)|(_)|([{}])|([=+\-*/])|(\d+)|([a-zA-Z]+)|(\s+)|(.)'
        matches = re.findall(pattern, text)
        for groups in matches:
            token_str = next((g for g in groups if g), None)
            if token_str is None:
                continue
            token_key = f"[L:{token_str}]"
            if token_key in self.token_to_id:
                tokens.append(self.token_to_id[token_key])
            else:
                for char in token_str:
                    tokens.extend(self._get_safe_char_tokens(char))
        return tokens

    # ------------------------------------------------------------------
    # Selección de escala (modo no-adaptativo)
    # ------------------------------------------------------------------

    def _analyze_complexity(self, text: str) -> Dict[str, float]:
        return {
            'length': len(text),
            'has_code': bool(re.search(
                r'(def |class |import |return |for .+ in |if .+:\s*$)', text
            )),
            'is_science': bool(re.search(
                r'(\$.+\$)|(\\frac)|(\\sum)|(\\int)|(H2O)|(\[.*\])', text
            )),
        }

    def _select_optimal_scale(self, text: str, complexity: Dict[str, float]) -> int:
        has_code = complexity['has_code']
        is_science = complexity.get('is_science', False)
        if has_code and self.enable_ast:
            # Requiere al menos una definición o múltiples líneas de código
            if sum([text.count('\n') >= 2, bool(re.search(r'\bdef\s+\w', text))]) >= 1:
                return 4
        if is_science:
            return 5
        if complexity['length'] >= 100:
            return 3
        return 2

### Benchmark libreria

In [ ]:
"""
NeoToken — Benchmark Real y Justo
===================================
Metodología para comparación honesta:

  PROBLEMA con "ratio chars/tokens" como métrica principal:
  Un tokenizador con 1M de vocab siempre gana en ratio porque puede memorizar
  frases enteras. Eso no indica calidad — indica sobreajuste de vocab.

  MÉTRICAS REALES usadas aquí:
  1. Fertility (tokens/word): cuántos tokens necesita por palabra. Menor = mejor compresión real.
  2. Compression ratio normalizado por vocab: ratio / log2(vocab_size).
     Mide eficiencia por "bit de vocabulario invertido".
  3. OOV rate: % de tokens que son [UNK] o fallback a bytes. Mide cobertura.
  4. Latencia P50/P95 (ms): velocidad real bajo carga, no solo promedio.
  5. Integridad encode→decode: el texto se recupera exactamente (o normalizado).

  CONDICIONES DE IGUALDAD:
  - Mismos textos para todos.
  - Mismo hardware (single thread, sin paralelismo).
  - Categorías variadas: texto natural español, inglés, código, LaTeX, emojis,
    texto largo, JSON, mezcla multidominio.
  - Se incluyen 2-3 modelos indie y 2-3 "ballenas" de industria.

  MODELOS:
  Ballenas:  GPT-4o (tiktoken o200k_base), Llama-3 o GPT-2 (transformers)
  Indies:    BPE simple (implementación mínima aquí mismo), SentencePiece si disponible
  NeoToken:  Tu modelo entrenado

  USO:
    python benchmark.py
    python benchmark.py --model path/to/neotoken.pkl
    python benchmark.py --no-external   # solo NeoToken + baselines internos
"""

import os
import re
import sys
import time
import math
import argparse
import statistics
from typing import List, Dict, Any, Optional, Callable

# ── importar NeoToken desde main.py en el mismo directorio ──────────────────
# NeoToken se asume disponible en el entorno (ejecuta la celda de main.py antes)
NEOTOKEN_AVAILABLE = True

# ── dependencias opcionales ──────────────────────────────────────────────────
try:
    import tiktoken
    TIKTOKEN_AVAILABLE = True
except ImportError:
    TIKTOKEN_AVAILABLE = False

try:
    from transformers import AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False

try:
    import sentencepiece as spm
    SPM_AVAILABLE = True
except ImportError:
    SPM_AVAILABLE = False


# ==============================================================================
# SUITE DE TEXTOS: variada, representativa, con dificultad real
# ==============================================================================

TEST_SUITE: Dict[str, Dict[str, str]] = {

    # ── Texto natural ──────────────────────────────────────────────────────
    "ES: Texto corto": {
        "text": "El procesamiento de lenguaje natural permite a las máquinas comprender el habla humana.",
        "category": "text_es",
    },
    "ES: Acentos y ñ": {
        "text": "¡Ñoño! La información güera está en el menú de configuración del móvil. Año tras año.",
        "category": "text_es",
    },
    "EN: Texto corto": {
        "text": "Natural language processing enables machines to understand and generate human speech.",
        "category": "text_en",
    },
    "EN: Texto largo (300w)": {
        "text": (
            "Artificial intelligence is no longer a concept confined to science fiction. "
            "Modern machine learning systems can recognize speech, translate languages, "
            "generate images from text descriptions, and even write code. The transformer "
            "architecture, introduced in 2017, revolutionized the field by enabling models "
            "to process entire sequences in parallel rather than one token at a time. "
            "This parallelism unlocked the ability to train on massive datasets, leading "
            "to the emergence of large language models that exhibit surprising capabilities "
            "across a wide range of tasks, from summarization to reasoning to creative writing. "
            "However, these models are not without limitations. They can hallucinate facts, "
            "struggle with precise arithmetic, and reflect biases present in their training data. "
            "The tokenization step that precedes all of this processing is often overlooked, "
            "yet it fundamentally shapes what the model can and cannot learn efficiently."
        ),
        "category": "text_en",
    },

    # ── Emojis y caracteres especiales ────────────────────────────────────
    "Emojis y Unicode": {
        "text": "¡Hola! 🚀 ¿Cómo vas? 😅 Necesito revisar el servidor 🖥️ urgente. → ← ∀x∈ℝ",
        "category": "special",
    },
    "Símbolos mixtos": {
        "text": "Precio: $99.99 USD | Descuento: 15% | Código: A-B_C#42 | © 2024 ™ ® §",
        "category": "special",
    },

    # ── Código Python ──────────────────────────────────────────────────────
    "Código: función simple": {
        "text": "def fibonacci(n):\n    if n < 2:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
        "category": "code",
    },
    "Código: clase completa": {
        "text": (
            "class BinaryTree:\n"
            "    def __init__(self, val=0, left=None, right=None):\n"
            "        self.val = val\n"
            "        self.left = left\n"
            "        self.right = right\n\n"
            "    def insert(self, val):\n"
            "        if val < self.val:\n"
            "            if self.left is None:\n"
            "                self.left = BinaryTree(val)\n"
            "            else:\n"
            "                self.left.insert(val)\n"
            "        else:\n"
            "            if self.right is None:\n"
            "                self.right = BinaryTree(val)\n"
            "            else:\n"
            "                self.right.insert(val)\n"
        ),
        "category": "code",
    },
    "Código: imports y decoradores": {
        "text": (
            "from typing import List, Optional, Dict\n"
            "import numpy as np\n\n"
            "@staticmethod\n"
            "def normalize(arr: np.ndarray, axis: int = 0) -> np.ndarray:\n"
            "    return (arr - arr.mean(axis=axis)) / (arr.std(axis=axis) + 1e-8)\n"
        ),
        "category": "code",
    },

    # ── Ciencia / LaTeX ────────────────────────────────────────────────────
    "LaTeX: ecuación de campo": {
        "text": r"La ecuación de campo de Einstein: $G_{\mu\nu} + \Lambda g_{\mu\nu} = \frac{8\pi G}{c^4} T_{\mu\nu}$",
        "category": "science",
    },
    "LaTeX: mecánica cuántica": {
        "text": r"El principio de incertidumbre: $\Delta x \cdot \Delta p \geq \frac{\hbar}{2}$ donde $\hbar = \frac{h}{2\pi}$",
        "category": "science",
    },
    "LaTeX: fórmula compleja": {
        "text": r"$\int_{-\infty}^{\infty} e^{-x^2} dx = \sqrt{\pi}$ y $\sum_{n=0}^{\infty} \frac{x^n}{n!} = e^x$",
        "category": "science",
    },

    # ── Estructurado / JSON ────────────────────────────────────────────────
    "JSON simple": {
        "text": '{"id": 101, "name": "NeoToken", "tags": ["AI", "NLP", "tokenizer"], "score": 0.9987}',
        "category": "structured",
    },
    "JSON anidado": {
        "text": (
            '{"model": {"name": "neotoken_22k", "vocab": 22619, "scales": 6}, '
            '"config": {"bpe_iters": 2000, "enable_ast": true, "enable_memory": true}, '
            '"metrics": {"ratio": 2.36, "density": 10.44, "latency_ms": 0.33}}'
        ),
        "category": "structured",
    },

    # ── Texto multidominio (el más difícil) ────────────────────────────────
    "Multidominio: texto+código+LaTeX": {
        "text": (
            "Para entrenar el modelo usé la función $L(\\theta) = -\\sum_i \\log p(x_i|\\theta)$.\n"
            "La implementación en Python fue:\n"
            "def loss(logits, targets):\n"
            "    return -torch.log_softmax(logits, dim=-1)[range(len(targets)), targets].mean()\n"
            "Con lr=3e-4 converge en ~50 epochs. ¡Resultados prometedores! 🎯"
        ),
        "category": "mixed",
    },
}


# ==============================================================================
# WRAPPER UNIVERSAL: interfaz común para todos los tokenizadores
# ==============================================================================

class TokenizerWrapper:
    """Interfaz unificada para comparar cualquier tokenizador."""

    def __init__(self, name: str, encode_fn: Callable, decode_fn: Callable, vocab_size: int):
        self.name = name
        self.vocab_size = vocab_size
        self._encode = encode_fn
        self._decode = decode_fn

    def encode(self, text: str) -> List[int]:
        return self._encode(text)

    def decode(self, ids: List[int]) -> str:
        return self._decode(ids)


def build_tokenizers(neo_model_path: str) -> Dict[str, TokenizerWrapper]:
    """Construye todos los tokenizadores disponibles."""
    tokenizers = {}

    # ── NeoToken ────────────────────────────────────────────────────────
    if NEOTOKEN_AVAILABLE and os.path.exists(neo_model_path):
        try:
            nt = NeoToken()
            nt.load(neo_model_path)
            tokenizers["NeoToken (tuyo)"] = TokenizerWrapper(
                name="NeoToken (tuyo)",
                encode_fn=lambda t: nt.encode(t, adaptive=True),
                decode_fn=nt.decode,
                vocab_size=nt.vocab_size,
            )
            print(f"  ✅ NeoToken cargado: {nt.vocab_size} tokens")
        except Exception as e:
            print(f"  ⚠️ Error cargando NeoToken: {e}")

    # ── GPT-4o / tiktoken ───────────────────────────────────────────────
    if TIKTOKEN_AVAILABLE:
        try:
            enc_gpt4 = tiktoken.get_encoding("o200k_base")
            tokenizers["GPT-4o (OpenAI)"] = TokenizerWrapper(
                name="GPT-4o (OpenAI)",
                encode_fn=enc_gpt4.encode,
                decode_fn=enc_gpt4.decode,
                vocab_size=200019,
            )
            print("  ✅ GPT-4o (tiktoken o200k_base) cargado")
        except Exception as e:
            print(f"  ⚠️ Error GPT-4o: {e}")

        # ── GPT-2 (indie clásico, vocab 50k) ────────────────────────────
        try:
            enc_gpt2 = tiktoken.get_encoding("gpt2")
            tokenizers["GPT-2 (OpenAI, 50k)"] = TokenizerWrapper(
                name="GPT-2 (OpenAI, 50k)",
                encode_fn=enc_gpt2.encode,
                decode_fn=enc_gpt2.decode,
                vocab_size=50257,
            )
            print("  ✅ GPT-2 (tiktoken gpt2) cargado")
        except Exception as e:
            print(f"  ⚠️ Error GPT-2: {e}")

    # ── Modelos via transformers ─────────────────────────────────────────
    if TRANSFORMERS_AVAILABLE:
        hf_models = [
            # Ballenas
            ("meta-llama/Meta-Llama-3-8B",     "Llama-3 (Meta, 128k)"),
            ("mistralai/Mistral-7B-v0.1",       "Mistral-7B (Mistral, 32k)"),
            # Indies / modelos más pequeños
            ("microsoft/phi-2",                 "Phi-2 (Microsoft, 51k)"),
        ]
        for model_id, label in hf_models:
            try:
                tok = AutoTokenizer.from_pretrained(
                    model_id, use_fast=True, trust_remote_code=True
                )
                vs = len(tok)
                tokenizers[label] = TokenizerWrapper(
                    name=label,
                    encode_fn=lambda t, _tok=tok: _tok.encode(t, add_special_tokens=False),
                    decode_fn=lambda ids, _tok=tok: _tok.decode(ids, skip_special_tokens=True),
                    vocab_size=vs,
                )
                print(f"  ✅ {label} cargado (vocab={vs:,})")
            except Exception as e:
                print(f"  ⚠️ {label} no disponible: {e}")

    return tokenizers


# ==============================================================================
# MÉTRICAS
# ==============================================================================

def count_words(text: str) -> int:
    """Cuenta palabras de forma simple pero consistente."""
    return len(re.findall(r'\S+', text))


def compute_oov_rate(ids: List[int], tokenizer: TokenizerWrapper) -> float:
    """
    Estima la tasa de OOV/fallback.
    Para NeoToken: cuenta tokens [UNK] o tokens de byte-fallback.
    Para otros: no aplicable directamente → retorna 0.0 (ellos no exponen esto).
    """
    if not hasattr(tokenizer, '_encode'):
        return 0.0
    # Intentamos acceder al NeoToken subyacente
    try:
        nt_instance = tokenizer._encode.__self__  # bound method → instancia
        if not isinstance(nt_instance, NeoToken):
            return 0.0
        unk_id = nt_instance.special_tokens.get('[UNK]', -1)
        byte_ids = {v for k, v in nt_instance.token_to_id.items() if k.startswith('[B:')}
        oov_count = sum(1 for i in ids if i == unk_id or i in byte_ids)
        return oov_count / max(len(ids), 1)
    except Exception:
        return 0.0


def measure_latency(encode_fn: Callable, text: str, runs: int = 50) -> Dict[str, float]:
    """Mide latencia con múltiples runs para P50 y P95 estables."""
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        encode_fn(text)
        times.append((time.perf_counter() - t0) * 1000)
    times.sort()
    return {
        "p50": statistics.median(times),
        "p95": times[int(len(times) * 0.95)],
        "mean": statistics.mean(times),
    }


def check_integrity(original: str, decoded: str, category: str) -> bool:
    """Verifica que encode→decode sea fiel al original."""
    if original == decoded:
        return True
    # Normalización de espacios (aceptable)
    norm = lambda t: re.sub(r'\s+', ' ', t).strip()
    if norm(original) == norm(decoded):
        return True
    # Para AST: la decodificación produce <NODE:...>, es correcto por diseño
    if category == "code" and "<NODE:" in decoded:
        return True
    return False


# ==============================================================================
# EVALUACIÓN
# ==============================================================================

def evaluate_tokenizer(wrapper: TokenizerWrapper,
                        suite: Dict[str, Dict[str, str]]) -> Dict[str, Any]:
    """Evalúa un tokenizador en todas las métricas sobre toda la suite."""

    total_tokens = 0
    total_words = 0
    total_chars = 0
    integrity_failures = []
    oov_rates = []
    latencies_all = []

    per_category: Dict[str, Dict[str, list]] = {}

    for test_name, data in suite.items():
        text = data["text"]
        cat = data["category"]

        if cat not in per_category:
            per_category[cat] = {"fertility": [], "oov": []}

        # Encode
        try:
            ids = wrapper.encode(text)
        except Exception as e:
            integrity_failures.append(f"{test_name}: encode error ({e})")
            continue

        # Decode + integridad
        try:
            decoded = wrapper.decode(ids)
            if not check_integrity(text, decoded, cat):
                integrity_failures.append(test_name)
        except Exception as e:
            integrity_failures.append(f"{test_name}: decode error ({e})")
            decoded = ""

        n_tokens = len(ids)
        n_words = max(count_words(text), 1)
        n_chars = len(text)
        fertility = n_tokens / n_words
        oov = count_oov_neo(ids, wrapper)

        total_tokens += n_tokens
        total_words += n_words
        total_chars += n_chars
        oov_rates.append(oov)
        per_category[cat]["fertility"].append(fertility)
        per_category[cat]["oov"].append(oov)

        # Latencia (solo sobre textos cortos/medios para no inflar con textos largos)
        if n_chars <= 500:
            lat = measure_latency(wrapper.encode, text, runs=30)
            latencies_all.append(lat["p50"])

    # Métricas globales
    global_fertility = total_tokens / max(total_words, 1)
    global_ratio = total_chars / max(total_tokens, 1)  # chars per token (para contexto)
    mean_oov = statistics.mean(oov_rates) if oov_rates else 0.0
    n_failures = len(integrity_failures)

    # Compression efficiency: ratio normalizado por vocabulario
    # Mide cuánta compresión logras por "bit de vocabulario gastado"
    vocab_bits = math.log2(max(wrapper.vocab_size, 1))
    compression_efficiency = global_ratio / vocab_bits

    lat_p50 = statistics.median(latencies_all) if latencies_all else 0.0
    lat_p95 = statistics.quantiles(latencies_all, n=20)[-1] if len(latencies_all) >= 20 else (max(latencies_all) if latencies_all else 0.0)

    # Fertility por categoría
    cat_fertility = {
        cat: statistics.mean(vals["fertility"]) if vals["fertility"] else 0.0
        for cat, vals in per_category.items()
    }

    return {
        "vocab_size": wrapper.vocab_size,
        "global_fertility": global_fertility,       # tokens/word (↓ mejor)
        "global_ratio": global_ratio,               # chars/token (↑ mejor, pero sesgado por vocab)
        "compression_efficiency": compression_efficiency,  # ratio/log2(vocab) (↑ mejor, MÉTRICA JUSTA)
        "oov_rate_pct": mean_oov * 100,             # % tokens OOV/fallback (↓ mejor, solo NeoToken)
        "lat_p50_ms": lat_p50,                      # latencia mediana (↓ mejor)
        "lat_p95_ms": lat_p95,                      # latencia P95 (↓ mejor)
        "integrity_failures": n_failures,            # 0 = perfecto
        "integrity_fail_list": integrity_failures,
        "cat_fertility": cat_fertility,
    }


def count_oov_neo(ids: List[int], wrapper: TokenizerWrapper) -> float:
    """Helper para OOV: solo funciona con NeoToken, resto retorna 0."""
    try:
        fn = wrapper._encode
        if hasattr(fn, '__self__') and isinstance(fn.__self__, NeoToken):
            nt = fn.__self__
            unk_id = nt.special_tokens.get('[UNK]', -1)
            byte_ids = {v for k, v in nt.token_to_id.items() if k.startswith('[B:')}
            oov = sum(1 for i in ids if i == unk_id or i in byte_ids)
            return oov / max(len(ids), 1)
    except Exception:
        pass
    return 0.0


# ==============================================================================
# PRESENTACIÓN
# ==============================================================================

CATEGORY_LABELS = {
    "text_es": "Texto ES",
    "text_en": "Texto EN",
    "special":  "Emojis/Unicode",
    "code":     "Código Python",
    "science":  "LaTeX/Ciencia",
    "structured": "JSON/Struct",
    "mixed":    "Multidominio",
}


def print_results(results: Dict[str, Dict[str, Any]]):
    """Imprime el reporte completo con todas las métricas."""

    if not results:
        print("❌ Sin resultados que mostrar.")
        return

    # ── Tabla principal ──────────────────────────────────────────────────
    col_w = 26
    sep = "─" * 130

    print()
    print("=" * 130)
    print("  🏆  BENCHMARK REAL: NeoToken vs Industry  —  Métricas honestas, condiciones iguales")
    print("=" * 130)
    print()
    print(
        f"  {'MODELO':<{col_w}} {'VOCAB':>9}  {'FERTILITY':>10}  {'RATIO c/t':>10}  "
        f"{'EFFIC*':>8}  {'LAT P50':>9}  {'LAT P95':>9}  {'INTEGRIDAD':>12}"
    )
    print(f"  {'':─<{col_w}} {'':─>9}  {'':─>10}  {'':─>10}  {'':─>8}  {'':─>9}  {'':─>9}  {'':─>12}")

    # Ordenar por compression_efficiency descendente
    ranked = sorted(results.items(), key=lambda x: x[1]["compression_efficiency"], reverse=True)

    for name, m in ranked:
        vocab_str = f"{m['vocab_size']:,}"
        fert_str  = f"{m['global_fertility']:.2f} t/w"
        ratio_str = f"{m['global_ratio']:.2f} c/t"
        eff_str   = f"{m['compression_efficiency']:.3f}"
        p50_str   = f"{m['lat_p50_ms']:.3f} ms"
        p95_str   = f"{m['lat_p95_ms']:.3f} ms"
        fails     = m["integrity_failures"]
        integ_str = f"✅ 100%" if fails == 0 else f"⚠️  {fails} fallo{'s' if fails>1 else ''}"

        print(
            f"  {name:<{col_w}} {vocab_str:>9}  {fert_str:>10}  {ratio_str:>10}  "
            f"{eff_str:>8}  {p50_str:>9}  {p95_str:>9}  {integ_str:>12}"
        )

    print()
    print(
        "  * EFFIC = ratio / log2(vocab)  →  mide compresión real por unidad de vocabulario invertida."
    )
    print(
        "    Es la métrica más justa: penaliza tener 200k vocab para comprimir bien."
    )

    # ── Fertility por categoría ──────────────────────────────────────────
    print()
    print("─" * 130)
    print("  FERTILITY POR CATEGORÍA (tokens por palabra)  —  menor es mejor compresión en esa categoría")
    print("─" * 130)

    all_cats = sorted({
        cat for m in results.values()
        for cat in m["cat_fertility"].keys()
    })

    header = f"  {'MODELO':<{col_w}}"
    for cat in all_cats:
        label = CATEGORY_LABELS.get(cat, cat)[:10]
        header += f"  {label:>12}"
    print(header)
    print(f"  {'':─<{col_w}}" + "  " + "─" * (14 * len(all_cats)))

    for name, m in ranked:
        row = f"  {name:<{col_w}}"
        for cat in all_cats:
            val = m["cat_fertility"].get(cat)
            row += f"  {val:>10.2f}  " if val is not None else f"  {'N/A':>10}  "
        print(row)

    # ── Fallos de integridad ─────────────────────────────────────────────
    print()
    print("─" * 130)
    print("  DETALLE DE FALLOS DE INTEGRIDAD (encode→decode)")
    print("─" * 130)
    any_failure = False
    for name, m in ranked:
        if m["integrity_fail_list"]:
            any_failure = True
            print(f"  ⚠️  {name}:")
            for f in m["integrity_fail_list"]:
                print(f"       - {f}")
    if not any_failure:
        print("  ✅ Todos los modelos pasaron integridad al 100%.")

    # ── Nota sobre OOV ───────────────────────────────────────────────────
    print()
    print("─" * 130)
    print("  OOV / BYTE-FALLBACK RATE  (solo disponible para NeoToken — otros no exponen esto)")
    print("─" * 130)
    for name, m in ranked:
        oov = m["oov_rate_pct"]
        if oov > 0:
            print(f"  {name:<{col_w}}  {oov:.2f}% de tokens son byte-fallback o [UNK]")
        else:
            print(f"  {name:<{col_w}}  N/A (métrica interna no expuesta por este tokenizador)")

    print()
    print("=" * 130)
    print()


def print_legend():
    print("""
  GLOSARIO DE MÉTRICAS
  ─────────────────────────────────────────────────────────────────────────────
  FERTILITY (t/w)   Tokens por palabra del texto original.
                    Menor = más compresión real, independiente del vocab.
                    GPT-4o suele dar ~1.2-1.5 en inglés, ~1.5-2.5 en español.

  RATIO (c/t)       Caracteres por token. Mayor = tokens más "grandes".
                    SESGADO por tamaño de vocab: no uses esto solo para comparar.

  EFFIC*            ratio / log2(vocab_size).
                    Cuánta compresión logras por bit de vocabulario gastado.
                    Métrica JUSTA: penaliza tener 200k vocab para comprimir bien.
                    Un modelo con 22k vocab que logra EFFIC similar a uno de 200k
                    es arquitectónicamente más eficiente.

  LAT P50/P95       Latencia de encode en milisegundos (percentil 50 y 95).
                    P95 revela comportamiento en el peor caso frecuente.

  INTEGRIDAD        ¿Se recupera el texto original después de encode→decode?
                    Para NeoToken en modo AST, la decodificación produce <NODE:X>
                    en lugar del código original — esto es correcto por diseño.
  ─────────────────────────────────────────────────────────────────────────────
""")


# ==============================================================================
# MAIN
# ==============================================================================

# ── Configuracion para Colab: edita estos valores directamente ──────────────────
NEO_MODEL_PATH = "neotoken_22k.pkl"   # ruta al .pkl (relativa a /content/)
ONLY_NEOTOKEN  = False                 # True = omitir modelos externos

def main():
    parser = argparse.ArgumentParser(description="NeoToken — Benchmark Real y Justo")
    parser.add_argument(
        "--model", default=NEO_MODEL_PATH,
        help="Ruta al .pkl de NeoToken"
    )
    parser.add_argument(
        "--no-external", action="store_true",
        help="Omitir modelos externos (solo NeoToken)"
    )
    # parse_known_args ignora los args extra del kernel de Jupyter/Colab
    args, _ = parser.parse_known_args()
    if ONLY_NEOTOKEN:
        args.no_external = True

    print()
    print("  🔬 NeoToken — Benchmark Real")
    print("  Cargando tokenizadores...")
    print()

    tokenizers = build_tokenizers(args.model)

    if args.no_external:
        tokenizers = {k: v for k, v in tokenizers.items() if "NeoToken" in k}

    if not tokenizers:
        print("❌ No hay tokenizadores disponibles. Verifica las dependencias.")
        return

    print(f"\n  Evaluando {len(tokenizers)} tokenizador(es) sobre {len(TEST_SUITE)} textos...\n")

    results = {}
    for name, wrapper in tokenizers.items():
        print(f"  ⏳ Evaluando: {name}")
        try:
            results[name] = evaluate_tokenizer(wrapper, TEST_SUITE)
        except Exception as e:
            print(f"  ❌ Error evaluando {name}: {e}")

    print_results(results)
    print_legend()


if __name__ == "__main__":
    main()


  🔬 NeoToken — Benchmark Real
  Cargando tokenizadores...

✅ NeoToken cargado: 22619 tokens en vocabulario.
  ✅ NeoToken cargado: 22619 tokens
  ✅ GPT-4o (tiktoken o200k_base) cargado
  ✅ GPT-2 (tiktoken gpt2) cargado
  ⚠️ Llama-3 (Meta, 128k) no disponible: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B.
401 Client Error. (Request ID: Root=1-69a89637-1bbcea4450cdada8508333e1;ed4509f5-279f-4476-ae00-cbc8a93959c1)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B is restricted. You must have access to it and be authenticated to access it. Please log in.
  ✅ Mistral-7B (Mistral, 32k) cargado (vocab=32,000)
  ✅ Phi-2 (Microsoft, 51k) cargado (vocab=50,295)

  Evaluando 5 tokenizador(es) sobre 15 textos...

  ⏳ Evaluando: NeoToken (tuyo)
  ⏳ Evaluando: GPT-4o (OpenAI)
  ⏳ Evaluando: GPT-2 (OpenAI, 50k)
  ⏳ Evalua

# Data


### Instalacion de lo necesario para el dataset

In [ ]:
# ================================================================
# EJECUTAR ESTA CELDA SOLA ANTES DE NADA — luego reiniciar runtime
# ================================================================
# FIX: "cannot import name 'logging' from 'huggingface_hub'"
# Causa: Colab trae datasets==4.0.0 + huggingface_hub con hf_xet
# incompatibles entre sí. Forzamos versiones que funcionan juntas.

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "--quiet"] + list(args))

print("Paso 1/3: Desinstalando versiones rotas de Colab...")
pip("uninstall", "-y", "huggingface_hub", "datasets", "hf_xet", "hf-xet",
    "tokenizers", "multiprocess", "dill")

print("Paso 2/3: Instalando versiones compatibles...")
pip("install",
    "huggingface_hub==0.23.4",   # pre-xet, sin el bug de telemetry/logging
    "datasets==2.20.0",          # compatible con hf_hub 0.23.x
    "tokenizers==0.19.1",
    "tqdm>=4.65.0",
    "numpy>=1.24.0",
    "multiprocess==0.70.16",
    "dill==0.3.8",
)

print("Paso 3/3: Verificando...")
import importlib
for lib in ["huggingface_hub", "datasets"]:
    mod = importlib.import_module(lib)
    print(f"  ✅ {lib} {mod.__version__}")

print()
print("=" * 50)
print("✅ LISTO. Ahora:")
print("   Runtime → Restart session  (⚠️ obligatorio)")
print("   Luego ejecutar Dataset.py normalmente")
print("=" * 50)

Paso 1/3: Desinstalando versiones rotas de Colab...
Paso 2/3: Instalando versiones compatibles...
Paso 3/3: Verificando...
  ✅ huggingface_hub 1.5.0
  ✅ datasets 2.20.0

✅ LISTO. Ahora:
   Runtime → Restart session  (⚠️ obligatorio)
   Luego ejecutar Dataset.py normalmente


### Instalacion de DataSet

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
import os
import hashlib
import numpy as np
from datasets import load_dataset
from tqdm.auto import tqdm
import random
import gc
import warnings
import shutil
from datetime import datetime
import threading

# --- AUTENTICACIÓN HUGGING FACE ---
try:
    from huggingface_hub import login
except ImportError:
    os.system("pip install -q huggingface_hub")
    from huggingface_hub import login

# TOKEN desde variable de entorno (configúralo en Colab: Secrets → HF_TOKEN)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except: pass

if HF_TOKEN:
    print("🔐 Iniciando sesión en Hugging Face...")
    try:
        login(token=HF_TOKEN)
        print("✅ Login exitoso.")
    except Exception as e:
        print(f"⚠️ Error login: {e}")

warnings.filterwarnings("ignore")

# ==============================================================================
# LOGGING
# ==============================================================================

log_lock = threading.Lock()

def log(msg):
    timestamp = datetime.now().strftime("%H:%M:%S")
    with log_lock:
        tqdm.write(f"[{timestamp}] {msg}")

# ==============================================================================
# TOKENIZADOR GLOBAL
# Usamos MAX_WORKERS=1 para evitar race condition con _bpe_cache de NeoToken.
# ==============================================================================

tokenizer = None
struct_ids = None

def load_global_tokenizer(model_path="neotoken_22k.pkl"):
    global tokenizer, struct_ids
    if tokenizer is None:
        log(f"⏳ Cargando NeoTokenizer desde {model_path}...")
        tokenizer = NeoToken()
        tokenizer.load(model_path)
        struct_ids = {
            "start": tokenizer.special_tokens['[STRUCT_START]'],
            "end":   tokenizer.special_tokens['[STRUCT_END]'],
            "sep":   tokenizer.special_tokens['[SEP]'],
        }
        log(f"✅ Tokenizer listo — vocab: {tokenizer.vocab_size} tokens.")

# ==============================================================================
#
#  ANÁLISIS DE DISTRIBUCIÓN (por qué este dataset es así)
#  ───────────────────────────────────────────────────────
#
#  NeoToken tiene 6 escalas activas:
#    Escala 0/1 (byte/char)   — fallback, siempre activa
#    Escala 2   (subword BPE) — texto general ES+EN (entrenado en opus-100)
#    Escala 3   (word)        — texto fluido ES+EN
#    Escala 4   (AST Python)  — código (entrenado en code-search-net-python)
#    Escala 5   (LaTeX/math)  — fórmulas (entrenado en MetaMathQA+arxiv)
#
#  CryoCCM V4.2 tiene:
#    Torre L (Trapos ×6)  — percepción rápida: necesita texto variado, fluido
#    Torre H (Hielo ×15)  — razonamiento profundo: necesita texto denso/factual
#    Memoria diferenciable — necesita conversación multi-turno para aprender a leer
#    PonderNet / ACT       — aprende a parar antes en inputs fáciles, pensar más
#                            en difíciles → necesita mezcla de fácil+difícil
#
#  Objetivo del modelo: asistente bilingüe ES/EN, conversacional, con código y
#  matemáticas básicas.
#
#  DISTRIBUCIÓN RESULTANTE:
#    Narrativa ES+EN  38%  — base del idioma, activa escalas 2+3
#    Chat/Instruct    29%  — tarea principal del asistente
#    Wiki ES+EN       14%  — activa torre Hielo (razonamiento factual)
#    Código Python    10%  — activa Escala 4 (AST)
#    Código otro       3%  — JS/SQL/bash, útil para asistente general
#    Matemáticas       3%  — activa Escala 5 (LaTeX)
#    Ciencia/LaTeX     2%  — activa Escala 5 con contexto real
#
#  El dataset anterior tenía narrativa_es al 76% → el modelo aprendería a
#  "contar cuentos" pero la torre Hielo nunca vería suficiente texto factual
#  para aprender razonamiento real. Chat era solo 6.9% para una tarea que es
#  el 100% del objetivo. LaTeX/math era 0% → Escala 5 nunca se activaba.
#
# ==============================================================================

# ==============================================================================
# CONFIGURACIÓN DE FUENTES (con fallbacks robustos)
# ==============================================================================

# ==============================================================================
#  CRITERIO DE SELECCIÓN DE FUENTES: SOLO PARQUET NATIVO
#  ───────────────────────────────────────────────────────
#  HuggingFace datasets>=4.0 eliminó soporte para dataset scripts (.py).
#  Los datasets que usan scripts propios fallan con:
#    RuntimeError: Dataset scripts are no longer supported
#  o requieren trust_remote_code=True (que Colab bloquea silenciosamente).
#
#  REGLA: toda fuente aquí debe existir como archivos .parquet directos en HF.
#  No se usa ningún dataset con loading script Python custom.
#
#  FUENTES DESCARTADAS (scripts, inestables):
#    ❌ allenai/c4               → script Python, falla en datasets>=4.0
#    ❌ bigcode/the-stack-smol   → script + trust_remote_code, inestable
#    ❌ OpenAssistant/oasst1     → script Python, trust_remote_code
#    ❌ google-research-datasets/wiki40b → requiere TensorFlow, siempre falla en Colab
#    ❌ somosnlp/somos-clean-alpaca-es   → script custom, inestable
#    ❌ bertin-project/alpaca-spanish    → script custom, inestable
#    ❌ AI-MO/NuminaMath-CoT            → script custom, inestable
#    ❌ ccdv/arxiv-summarization        → requiere auth especial + script
#    ❌ bacalhau-project/*              → repo oscuro, inestable
#    ❌ msamogh/multilingual_tinystories → repo no oficial, sin garantía
#    ❌ Cohere/wikipedia-22-12-simple-embeddings → embeddings, no texto
#
#  FUENTES SELECCIONADAS (parquet nativo, 100% estables):
#    ✅ uonlp/CulturaX            — 6.3T tokens, 167 idiomas, parquet puro
#    ✅ wikimedia/wikipedia       — convertido a parquet por wikimedia
#    ✅ roneneldan/TinyStories    — parquet nativo
#    ✅ HuggingFaceH4/ultrachat_200k — HF house dataset, siempre parquet
#    ✅ timdettmers/openassistant-guanaco — CSV simple, siempre funciona
#    ✅ vicgalle/alpaca-gpt4      — JSON/parquet, siempre funciona
#    ✅ tatsu-lab/alpaca          — JSON simple
#    ✅ somosnlp/alpaca_plus_gpt4_es — versión parquet del mismo grupo somosnlp
#    ✅ lvwerra/codeparrot-clean  — parquet nativo
#    ✅ flytech/python-codes-25k  — parquet nativo
#    ✅ mbpp                      — benchmark clásico, parquet
#    ✅ meta-math/MetaMathQA      — parquet nativo
#    ✅ lighteval/MATH             — parquet nativo, problemas con solución
# ==============================================================================

DATASETS_CONFIG = {

    # ══════════════════════════════════════════════════════════════════════════
    # REGLA DE DISEÑO: cada fuente aparece en UNA SOLA categoría.
    # Sin reuso entre categorías → dedup no vacía categorías posteriores.
    #
    # FUENTES POR CATEGORÍA Y SUS TAMAÑOS REALES:
    #
    # narrative_es  → CulturaX ES (200M)  + cosmopedia wikihow (31M)
    # narrative_en  → TinyStories (2.1M)  + cosmopedia stories + CulturaX EN
    # chat_es       → OpenHermes ES (1M)  → 400k es fácil
    # chat_en       → OpenHermes EN (1M)  + smoltalk (400k) + ultrachat (207k)
    # alpaca_instruct→ yahma-cleaned(51k) + c-s-ale/alpaca-gpt4(52k) = 103k > 100k
    # wiki_es       → wikipedia ES (1.9M) — más que suficiente
    # wiki_en       → wikipedia simple(200k) + wikipedia EN completa
    # code_python   → codeparrot-clean (18M) + smollm-corpus/python-edu
    # code_other    → smollm-corpus/python-edu (15 lenguajes reales)
    # math          → finemath-4plus (6.7M, LaTeX real) + MetaMathQA
    # science       → finemath-3plus (21.4M, LaTeX real de web) + finemath-4plus
    # ══════════════════════════════════════════════════════════════════════════

    # ──────────────────────────────────────────────────────────────────────────
    # 1. NARRATIVA ESPAÑOL (26%) — base idiomática ES, escalas 2+3 de NeoToken
    #    Solo CulturaX y Cosmopedia. Wikipedia ES queda RESERVADA para wiki_es.
    # ──────────────────────────────────────────────────────────────────────────
    "narrative_es": [
        # PRIMARIO: CulturaX ES — 200M docs web, parquet nativo, gated con acceso
        {"path": "uonlp/CulturaX", "name": "es", "split": "train", "type": "text_raw"},
        # FALLBACK: Cosmopedia wikihow — 31M docs sintéticos de alta calidad
        {"path": "HuggingFaceTB/cosmopedia", "name": "wikihow", "split": "train", "type": "text_raw"},
    ],

    # ──────────────────────────────────────────────────────────────────────────
    # 2. NARRATIVA INGLÉS (12%) — mantiene vocab EN activo en NeoToken escalas 2+3
    #    TinyStories exclusivo aquí. CulturaX EN como gran fallback.
    # ──────────────────────────────────────────────────────────────────────────
    "narrative_en": [
        # PRIMARIO: TinyStories — 2.1M historias cortas EN, parquet nativo
        {"path": "roneneldan/TinyStories", "split": "train", "type": "text_raw"},
        # SECUNDARIO: Cosmopedia stories — narrativa sintética de calidad
        {"path": "HuggingFaceTB/cosmopedia", "name": "stories", "split": "train", "type": "text_raw"},
        # FALLBACK inagotable: CulturaX EN — 200M docs
        {"path": "uonlp/CulturaX", "name": "en", "split": "train", "type": "text_raw"},
    ],

    # ──────────────────────────────────────────────────────────────────────────
    # 3. CHAT ESPAÑOL (14%) — tarea principal, entrena la memoria HRM
    #    OpenHermes ES tiene 1M instrucciones reales en español.
    #    400k meta es trivial para esta fuente. Sin fallback a texto crudo.
    # ──────────────────────────────────────────────────────────────────────────
    "chat_es": [
        # PRIMARIO: OpenHermes 2.5 traducido a ES — 1M instrucciones ES reales
        {"path": "Iker/OpenHermes-2.5-Spanish", "split": "train", "type": "openhermes_es"},
        # FALLBACK: smoltalk multilingüe con ES (HF house dataset)
        {"path": "HuggingFaceTB/smoltalk", "name": "smol-magpie-ultra", "split": "train", "type": "smoltalk"},
    ],

    # ──────────────────────────────────────────────────────────────────────────
    # 4. CHAT INGLÉS (12%) — conversación EN de alta calidad
    #    OpenHermes EN 1M cubre 350k con margen enorme. Sin problema de volumen.
    # ──────────────────────────────────────────────────────────────────────────
    "chat_en": [
        # PRIMARIO: OpenHermes 2.5 EN — 1M instrucciones, parquet nativo
        {"path": "teknium/OpenHermes-2.5", "split": "train", "type": "openhermes"},
        # SECUNDARIO: smoltalk smol-magpie-ultra — 400k conversaciones HF house
        {"path": "HuggingFaceTB/smoltalk", "name": "smol-magpie-ultra", "split": "train", "type": "smoltalk"},
        # FALLBACK: UltraChat — 207k, ya probado que funciona
        {"path": "HuggingFaceH4/ultrachat_200k", "split": "train_sft", "type": "ultrachat"},
    ],

    # ──────────────────────────────────────────────────────────────────────────
    # 5. INSTRUCT MIXTO (3.5%) — formato instrucción→respuesta explícito
    #    EXCLUSIVO: yahma y c-s-ale NO aparecen en ninguna otra categoría.
    #    yahma/alpaca-cleaned (51k) + c-s-ale/alpaca-gpt4-data (52k) = 103k > 100k
    # ──────────────────────────────────────────────────────────────────────────
    "alpaca_instruct": [
        # PRIMARIO: Alpaca Cleaned — 51k instrucciones limpias, parquet
        {"path": "yahma/alpaca-cleaned", "split": "train", "type": "alpaca"},
        # SECUNDARIO: WizardLM evol-instruct 70k — instrucciones evolucionadas, parquet
        # DISTINTO a OpenHermes y alpaca-cleaned → no se vacía por dedup
        {"path": "WizardLM/WizardLM_evol_instruct_70k", "split": "train", "type": "wizardlm"},
        # TERCIARIO: tatsu-lab/alpaca original — 52k, parquet, distinto a yahma-cleaned
        {"path": "tatsu-lab/alpaca", "split": "train", "type": "alpaca"},
        # FALLBACK: c-s-ale como último recurso
        {"path": "c-s-ale/alpaca-gpt4-data", "split": "train", "type": "alpaca_csa"},
    ],

    # ──────────────────────────────────────────────────────────────────────────
    # 6. WIKIPEDIA ESPAÑOL (7%) — texto factual denso, Torre Hielo
    #    Wikipedia ES tiene 1.9M artículos — 200k meta es fácil.
    # ──────────────────────────────────────────────────────────────────────────
    "wiki_es": [
        # PRIMARIO: Wikipedia ES oficial, parquet wikimedia
        {"path": "wikimedia/wikipedia", "name": "20231101.es", "split": "train", "type": "wiki"},
        # FALLBACK: CulturaX ES si wikimedia falla por red
        {"path": "uonlp/CulturaX", "name": "es", "split": "train", "type": "text_raw"},
    ],

    # ──────────────────────────────────────────────────────────────────────────
    # 7. WIKIPEDIA INGLÉS (7%) — texto factual EN
    #    Simple Wiki tiene ~200k artículos, exacto para la meta.
    # ──────────────────────────────────────────────────────────────────────────
    "wiki_en": [
        # PRIMARIO: Wikipedia Simple EN — frases cortas, ideal para razonamiento
        {"path": "wikimedia/wikipedia", "name": "20231101.simple", "split": "train", "type": "wiki"},
        # FALLBACK: Wikipedia EN completa — mucho más grande
        {"path": "wikimedia/wikipedia", "name": "20231101.en", "split": "train", "type": "wiki"},
    ],

    # ──────────────────────────────────────────────────────────────────────────
    # 8. CÓDIGO PYTHON (10%) — activa Escala 4 AST de NeoToken
    #    codeparrot-clean EXCLUSIVO aquí. smollm-corpus python-edu como refuerzo.
    # ──────────────────────────────────────────────────────────────────────────
    "code_python": [
        # PRIMARIO: CodeParrot Clean — 18M docs Python deduplicado, parquet
        {"path": "lvwerra/codeparrot-clean", "split": "train", "type": "codeparrot"},
        # SECUNDARIO: smollm-corpus python-edu — Python educativo curado
        {"path": "HuggingFaceTB/smollm-corpus", "name": "python-edu", "split": "train", "type": "text_raw"},
    ],

    # ──────────────────────────────────────────────────────────────────────────
    # 8b. CÓDIGO OTROS LENGUAJES (3.5%)
    #     smollm-corpus/stack-edu tiene 15 lenguajes (JS, TS, Java, C, C++, Go,
    #     Rust, SQL, bash, etc.) — parquet nativo HF house, no es script.
    #     ESTO RESUELVE el problema anterior donde github-code era un script roto.
    # ──────────────────────────────────────────────────────────────────────────
    "code_other": [
        # PRIMARIO: codeparrot/codeparrot-clean-valid — multi-lenguaje real, parquet
        # Contiene Python, JS, Java, C, C++, Go, Ruby, PHP, etc.
        # NOTA: stack-edu-dedup NO existe en smollm-corpus (verificado en output real)
        # Nombres válidos: 'cosmopedia-v2', 'fineweb-edu-dedup', 'python-edu'
        {"path": "HuggingFaceTB/smollm-corpus", "name": "fineweb-edu-dedup", "split": "train", "type": "text_raw"},
        # SECUNDARIO: flytech/python-codes-25k — Python con instrucciones, parquet
        {"path": "flytech/python-codes-25k", "split": "train", "type": "codeparrot"},
        # FALLBACK: codeparrot como último recurso
        {"path": "lvwerra/codeparrot-clean", "split": "train", "type": "codeparrot"},
    ],

    # ──────────────────────────────────────────────────────────────────────────
    # 9. MATEMÁTICAS (2.8%) — activa Escala 5 LaTeX de NeoToken
    #    FineMath-4+ es MEJOR que MetaMathQA: 6.7M docs, LaTeX real de CommonCrawl
    #    con step-by-step explanations. HF paper: mejora GSM8K +20.5pts sobre Llama.
    #    finemath y MetaMathQA son datasets DISTINTOS — no hay conflicto con science.
    # ──────────────────────────────────────────────────────────────────────────
    "math": [
        # PRIMARIO: FineMath-4+ — 6.7M docs matemática de alta calidad, LaTeX real
        {"path": "HuggingFaceTB/finemath", "name": "finemath-4plus", "split": "train", "type": "text_raw"},
        # FALLBACK: MetaMathQA — 395k, el original que entrenó NeoToken Escala 5
        {"path": "meta-math/MetaMathQA", "split": "train", "type": "math"},
    ],

    # ──────────────────────────────────────────────────────────────────────────
    # 10. CIENCIA / LATEX (1.7%) — activa Escala 5 con LaTeX de papers reales
    #     FineMath-3+ tiene 21.4M docs de páginas web con contenido matemático
    #     real (no sintético), incluyendo papers, libros, foros de matemáticas.
    #     Es DISTINTO a finemath-4plus (que es un subconjunto más pequeño y curado).
    # ──────────────────────────────────────────────────────────────────────────
    "science": [
        # PRIMARIO: FineMath-3+ — 21.4M docs LaTeX/Markdown de CommonCrawl
        {"path": "HuggingFaceTB/finemath", "name": "finemath-3plus", "split": "train", "type": "text_raw"},
        # FALLBACK: FineMath-4+ (subconjunto de mayor calidad)
        {"path": "HuggingFaceTB/finemath", "name": "finemath-4plus", "split": "train", "type": "text_raw"},
    ],
}


# ==============================================================================
# SERIALIZADORES — convierten items crudos a dict normalizado
# ==============================================================================

def serialize_item(item, ds_type):
    """Convierte el item crudo del dataset a un dict normalizado y serializable."""
    try:
        if ds_type == "text_raw":
            return {"text": str(item.get('text', '') or item.get('content', ''))}
        elif ds_type == "wiki":
            return {"title": str(item.get('title', '')), "text": str(item.get('text', ''))}
        elif ds_type == "wiki40b":
            return {"text": str(item.get('text', ''))}
        elif ds_type == "wiki_cohere":
            return {"title": str(item.get('title', '')), "text": str(item.get('text', ''))}
        elif ds_type == "alpaca":
            return {
                "instruction": str(item.get('instruction', '')),
                "input":       str(item.get('input', '')),
                "output":      str(item.get('output', '')),
            }
        elif ds_type == "orca":
            return {
                "question":      str(item.get('question', '')),
                "response":      str(item.get('response', '')),
                "system_prompt": str(item.get('system_prompt', '')),
            }
        elif ds_type in ("ultrachat", "oasst1", "guanaco"):
            if 'messages' in item:
                return {"messages": [{"content": str(m.get('content', ''))} for m in item['messages'][:4]]}
            return {"text": str(item.get('text', ''))}
        elif ds_type == "codeparrot":
            val = item.get('content', item.get('code', item.get('text', '')))
            return {"content": str(val)} if val else None
        elif ds_type == "mbpp":
            return {"text": str(item.get('text', '')), "code": str(item.get('code', ''))}
        elif ds_type == "math":
            return {"query": str(item.get('query', '')), "response": str(item.get('response', ''))}
        elif ds_type == "openhermes":
            # teknium/OpenHermes-2.5: campo "conversations" lista de {from, value}
            convs = item.get('conversations', [])
            if len(convs) >= 2:
                human = next((c['value'] for c in convs if c.get('from') == 'human'), '')
                gpt   = next((c['value'] for c in convs if c.get('from') == 'gpt'), '')
                return {"instruction": str(human), "output": str(gpt)}
            return None
        elif ds_type == "openhermes_es":
            # Iker/OpenHermes-2.5-Spanish: campo "spanish_conversation" lista de {from, value}
            convs = item.get('spanish_conversation', item.get('conversations', []))
            if len(convs) >= 2:
                human = next((c.get('value','') for c in convs if c.get('from') == 'human'), '')
                gpt   = next((c.get('value','') for c in convs if c.get('from') == 'gpt'), '')
                return {"instruction": str(human), "output": str(gpt)}
            return None
        elif ds_type == "smoltalk":
            # HuggingFaceTB/smoltalk: campo "messages" lista de {role, content}
            msgs = item.get('messages', [])
            if len(msgs) >= 2:
                user = next((m['content'] for m in msgs if m.get('role') == 'user'), '')
                asst = next((m['content'] for m in msgs if m.get('role') == 'assistant'), '')
                return {"instruction": str(user), "output": str(asst)}
            return None
        elif ds_type == "alpaca_csa":
            # c-s-ale/alpaca-gpt4-data: campos instruction, input, output
            return {
                "instruction": str(item.get('instruction', '')),
                "input":       str(item.get('input', '')),
                "output":      str(item.get('output', '')),
            }
        elif ds_type == "wizardlm":
            # WizardLM/WizardLM_evol_instruct_70k: campos instruction + output
            return {
                "instruction": str(item.get('instruction', '')),
                "input":       str(item.get('input', '')),
                "output":      str(item.get('output', '')),
            }
        elif ds_type == "math_lighteval":
            return {"problem": str(item.get('problem', '')), "solution": str(item.get('solution', ''))}
        elif ds_type == "aya":
            return {"inputs": str(item.get('inputs', '')), "targets": str(item.get('targets', ''))}
        elif ds_type in ("openhermes", "slimorca"):
            convs = item.get('conversations', [])
            if len(convs) >= 2:
                return {"conversations": [{"from": str(c.get('from','')), "value": str(c.get('value',''))} for c in convs[:6]]}
            return None
        elif ds_type == "github_code":
            code = str(item.get('content', ''))
            lang = str(item.get('file_language', item.get('repo_primary_language', 'unknown')))
            return {"content": code, "language": lang} if code else None
    except:
        return None
    return None

# ==============================================================================
# FORMATEADORES — convierten dict normalizado a par (instrucción, respuesta)
# La estructura [STRUCT_START] inst [STRUCT_END][STRUCT_START] resp [STRUCT_END][SEP]
# es la que entiende CryoCCM según los special_tokens de NeoToken.
# ==============================================================================

def format_item(item, ds_type):
    """Devuelve (instrucción, respuesta) o None si el item no sirve."""
    try:
        # ── Narrativa ─────────────────────────────────────────────────────────
        if ds_type == "text_raw":
            text = item.get('text', '')
            if len(text) < 80: return None
            return ("Continúa el siguiente texto:", text[:1500])

        elif ds_type in ("wiki", "wiki_cohere"):
            title = item.get('title', 'Información')
            text  = item.get('text', '')
            if len(text) < 60 or len(text) > 2000: return None
            # Filtrar páginas de desambiguación
            if any(w in text.lower() for w in ("refer ", "refiere ", "disambig")): return None
            prompts_es = [f"¿Qué es {title}?", f"Explica brevemente {title}."]
            prompts_en = [f"What is {title}?", f"Tell me about {title}."]
            # Detectar idioma por caracteres
            prompt = random.choice(prompts_es if any(c in text for c in 'áéíóúñ¿¡') else prompts_en)
            return (prompt, text)

        elif ds_type == "wiki40b":
            text = (item.get('text', '')
                    .replace('_START_ARTICLE_', '')
                    .replace('_START_PARAGRAPH_', '')
                    .replace('_NEWLINE_', '\n'))
            if len(text) < 80: return None
            return ("Wikipedia:", text[:1500])

        # ── Chat / Instruct ────────────────────────────────────────────────────
        elif ds_type == "alpaca":
            inst = item.get('instruction', '')
            inp  = item.get('input', '')
            out  = item.get('output', '')
            if not inst or not out: return None
            if inp: inst = f"{inst}\n{'Contexto' if any(c in inst for c in 'áéíóúñ') else 'Context'}: {inp}"
            return (inst, out)

        elif ds_type == "orca":
            q   = item.get('question', '')
            sys = item.get('system_prompt', '')
            r   = item.get('response', '')
            if not q or not r: return None
            if sys: q = f"System: {sys}\nUser: {q}"
            return (q, r)

        elif ds_type == "ultrachat":
            msgs = item.get('messages', [])
            if len(msgs) < 2: return None
            return (msgs[0]['content'], msgs[1]['content'])

        elif ds_type == "oasst1":
            text = item.get('text', '')
            if len(text) < 10: return None
            # OASST1 no viene en pares — usamos el mensaje como respuesta a "Chat:"
            return ("Chat:", text)

        elif ds_type == "guanaco":
            text = item.get('text', '')
            try:
                parts = text.split('### Assistant:')
                if len(parts) < 2: return None
                human = parts[0].split('### Human:')[-1].strip()
                bot   = parts[1].strip()
                if not human or not bot: return None
                return (human, bot)
            except:
                return None

        # ── Código ────────────────────────────────────────────────────────────
        # IMPORTANTE: el tokenizer usará Escala 4 (AST) automáticamente cuando
        # detecte señales de código Python (_encode_dynamic_stream).
        elif ds_type == "codeparrot":
            code = item.get('content', '')
            if len(code) < 20: return None
            return ("Write Python code:", code[:2000])

        elif ds_type == "mbpp":
            desc = item.get('text', '')
            code = item.get('code', '')
            if not desc or not code: return None
            return (f"Write a Python function to: {desc}", code)

        # ── Matemáticas — activa Escala 5 (LaTeX) de NeoToken ────────────────
        elif ds_type == "math":
            q = item.get('query', '')
            r = item.get('response', '')
            if not q or not r: return None
            prefix = random.choice(["Resuelve:", "Solve:", "Calcula:", "Compute:"])
            return (f"{prefix} {q}", r)

        elif ds_type in ("openhermes", "openhermes_es"):
            inst = item.get('instruction', '')
            out  = item.get('output', '')
            if not inst or not out: return None
            return (inst, out)

        elif ds_type == "smoltalk":
            inst = item.get('instruction', '')
            out  = item.get('output', '')
            if not inst or not out: return None
            return (inst, out)

        elif ds_type == "alpaca_csa":
            inst = item.get('instruction', '')
            inp  = item.get('input', '')
            out  = item.get('output', '')
            if not inst or not out: return None
            if inp: inst = f"{inst}\nContext: {inp}"
            return (inst, out)

        elif ds_type == "wizardlm":
            # WizardLM evol-instruct: mismo formato que alpaca
            inst = item.get('instruction', '')
            out  = item.get('output', '')
            if not inst or not out: return None
            return (inst, out)

        elif ds_type == "math_lighteval":
            q = item.get('problem', '')
            r = item.get('solution', '')
            if not q or not r: return None
            prefix = random.choice(["Solve:", "Resuelve:", "Find:", "Encuentra:"])
            return (f"{prefix} {q}", r)

        elif ds_type == "aya":
            inp = item.get('inputs', '').strip()
            tgt = item.get('targets', '').strip()
            if len(inp) < 5 or len(tgt) < 5: return None
            return (inp[:1500], tgt[:1500])

        elif ds_type in ("openhermes", "slimorca"):
            convs = item.get('conversations', [])
            if len(convs) < 2: return None
            human_msg, gpt_msg = None, None
            for c in convs:
                role = c.get('from', '').lower()
                val  = c.get('value', '').strip()
                if not val: continue
                if role in ('human', 'user') and human_msg is None:
                    human_msg = val
                elif role in ('gpt', 'assistant') and human_msg is not None and gpt_msg is None:
                    gpt_msg = val
                    break
            if not human_msg or not gpt_msg: return None
            return (human_msg[:1500], gpt_msg[:1500])

        elif ds_type == "github_code":
            code = item.get('content', '').strip()
            lang = item.get('language', 'code')
            if len(code) < 20: return None
            if lang.lower() in ('python', 'jupyter notebook'): return None
            prompt = random.choice([f"Write {lang} code:", f"Implement in {lang}:", f"{lang} example:"])
            return (prompt, code[:2000])

    except:
        return None
    return None

# ==============================================================================
# MOTOR DE GENERACIÓN (single-thread para thread-safety con _bpe_cache)
# ==============================================================================

def process_chunk(chunk_data):
    """
    Procesa una lista de (item, ds_type) → lista de token IDs.
    Ejecuta en single-thread para evitar race condition con _bpe_cache de NeoToken.
    La estructura de tokens respeta el formato CryoCCM:
      [STRUCT_START] ids_instrucción [STRUCT_END][STRUCT_START] ids_respuesta [STRUCT_END][SEP]
    """
    if tokenizer is None:
        return [], 0

    buffer_ids = []
    valid_count = 0
    s_start = struct_ids["start"]
    s_end   = struct_ids["end"]
    s_sep   = struct_ids["sep"]

    for item, ds_type in chunk_data:
        res = format_item(item, ds_type)
        if not res:
            continue
        inst, resp = res
        if not inst or not resp:
            continue
        try:
            # adaptive=True → NeoToken elige escala automáticamente por segmento:
            #   texto normal → Escala 2/3 (subword/word)
            #   código Python → Escala 4 (AST)
            #   fórmulas LaTeX → Escala 5
            ids_inst = tokenizer.encode(str(inst), adaptive=True)
            ids_resp = tokenizer.encode(str(resp), adaptive=True)

            if not ids_inst or not ids_resp:
                continue

            full_ids = [s_start] + ids_inst + [s_end, s_start] + ids_resp + [s_end, s_sep]
            buffer_ids.extend(full_ids)
            valid_count += 1
        except:
            continue

    return buffer_ids, valid_count


def encode_dataset(final_path: str, limits: dict, seen_hashes: set = None):
    """
    Motor principal de generación del dataset.
    Escribe directamente en /content (SSD de Colab) y hace backup progresivo a Drive.

    Args:
        final_path:   Ruta final en Drive donde guardar el .bin
        limits:       Dict {categoria: num_muestras}
        seen_hashes:  Set compartido entre train y val para evitar duplicados globales.
                      Si es None se crea uno nuevo (deduplicación solo dentro del archivo).
    """
    print(f"\n⚡ GENERANDO DATASET: {final_path}")
    temp_local_path = os.path.join("/content", f"temp_{os.path.basename(final_path)}")

    if os.path.exists(temp_local_path):
        os.remove(temp_local_path)
    open(temp_local_path, 'wb').close()

    if os.path.exists(final_path):
        log("♻️  Eliminando versión antigua en Drive...")
        os.remove(final_path)

    # Set de hashes para deduplicación. Usa el externo si se pasa (comparte entre val+train).
    if seen_hashes is None:
        seen_hashes = set()

    total_tokens      = 0
    total_samples     = 0
    total_duplicates  = 0
    CHUNK_SIZE        = 500   # items por chunk

    with open(temp_local_path, "ab", buffering=16 * 1024 * 1024) as f_out:

        for category, target in limits.items():
            if target <= 0:
                continue

            print("\n" + "=" * 60)
            log(f"🎯 CATEGORÍA: {category.upper()} → meta {target:,} muestras")

            datasets_list = DATASETS_CONFIG.get(category, [])
            if not datasets_list:
                log(f"⚠️ Categoría '{category}' no encontrada en DATASETS_CONFIG.")
                continue

            current = 0

            for ds_idx, ds_cfg in enumerate(datasets_list):
                if current >= target:
                    break

                remaining = target - current
                log(f"👉 Fuente {ds_idx+1}/{len(datasets_list)}: {ds_cfg['path']} (necesito {remaining:,})")

                # Carga robusta con 2 intentos
                ds = None
                try:
                    kw = {"streaming": True}
                    if "name" in ds_cfg:
                        kw["name"] = ds_cfg["name"]
                    ds = load_dataset(ds_cfg["path"], split=ds_cfg["split"], **kw)
                    log("   ✅ Conectado (modo Parquet/nativo).")
                except Exception as e1:
                    try:
                        kw["trust_remote_code"] = True
                        ds = load_dataset(ds_cfg["path"], split=ds_cfg["split"], **kw)
                        log("   ✅ Conectado (modo script/legacy).")
                    except Exception as e2:
                        log(f"   ❌ Fallo: {e1}. Saltando fuente...")
                        continue

                chunk  = []
                pbar   = tqdm(total=remaining, desc=f"   {ds_cfg['path'].split('/')[-1]}", unit="docs")

                try:
                    for item in ds:
                        if current >= target:
                            break

                        serialized = serialize_item(item, ds_cfg['type'])
                        if not serialized:
                            continue

                        # ── DEDUPLICACIÓN ─────────────────────────────────────
                        # Hash del texto crudo (antes de tokenizar) para evitar
                        # procesar el mismo contenido dos veces, incluso si viene
                        # de fuentes diferentes o de CulturaX como fallback.
                        # Usamos los primeros 500 chars para que sea rápido.
                        raw_text = str(list(serialized.values())[:2])[:500]
                        item_hash = hashlib.md5(raw_text.encode('utf-8', errors='ignore')).hexdigest()

                        if item_hash in seen_hashes:
                            total_duplicates += 1
                            continue
                        seen_hashes.add(item_hash)
                        # ──────────────────────────────────────────────────────

                        chunk.append((serialized, ds_cfg['type']))

                        if len(chunk) >= CHUNK_SIZE:
                            ids, count = process_chunk(chunk)
                            chunk = []
                            if ids:
                                np.array(ids, dtype=np.int32).tofile(f_out)
                                total_tokens  += len(ids)
                                current       += count
                                total_samples += count
                                pbar.update(count)

                    # Flush del chunk final
                    if chunk:
                        ids, count = process_chunk(chunk)
                        if ids:
                            np.array(ids, dtype=np.int32).tofile(f_out)
                            total_tokens  += len(ids)
                            current       += count
                            total_samples += count
                            pbar.update(count)

                except Exception as e:
                    log(f"   ⚠️ Error iterando: {e}")

                pbar.close()
                gc.collect()

                if current >= target:
                    log(f"   🎉 Meta cumplida ({current:,}/{target:,}). Dupes saltados: {total_duplicates:,}")
                else:
                    log(f"   ⚠️ Fuente agotada ({current:,}/{target:,}). Probando siguiente...")

            # Backup progresivo a Drive después de cada categoría
            f_out.flush()
            os.fsync(f_out.fileno())
            try:
                if os.path.exists(final_path):
                    os.remove(final_path)
                shutil.copy(temp_local_path, final_path)
                log(f"   ☁️  Backup a Drive OK ({total_tokens:,} tokens hasta ahora).")
            except Exception as e:
                log(f"   ⚠️  Backup Drive falló: {e}. Archivo seguro en Colab.")

    # Mover definitivamente a Drive
    log("🚀 Movimiento final a Drive...")
    if os.path.exists(final_path):
        os.remove(final_path)
    shutil.move(temp_local_path, final_path)

    print(f"\n✅ FINALIZADO")
    print(f"   Muestras únicas:  {total_samples:,}")
    print(f"   Duplicados saltados: {total_duplicates:,}")
    print(f"   Tokens:           {total_tokens:,}")
    print(f"   Hashes en memoria: {len(seen_hashes):,}")
    print(f"   Archivo:          {final_path}")


# ==============================================================================
# PUNTO DE ENTRADA
# ==============================================================================

if __name__ == "__main__":
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_FOLDER = "/content/drive/MyDrive/CryoDataset"
    os.makedirs(DRIVE_FOLDER, exist_ok=True)  # crear carpeta si no existe
    os.makedirs(DRIVE_FOLDER, exist_ok=True)

    load_global_tokenizer("neotoken_22k.pkl")

    # ──────────────────────────────────────────────────────────────────────────
    #  DISTRIBUCIÓN FINAL (justificada arriba en el bloque de análisis)
    #
    #  Categoría          Train      Val    % Train
    #  narrative_es      750,000   7,500    26.0%
    #  narrative_en      350,000   3,500    12.2%
    #  chat_es           400,000   4,000    13.9%
    #  chat_en           350,000   3,500    12.2%
    #  alpaca_instruct   100,000   1,000     3.5%
    #  wiki_es           200,000   2,000     6.9%
    #  wiki_en           200,000   2,000     6.9%
    #  code_python       300,000   3,000    10.4%
    #  code_other        100,000   1,000     3.5%
    #  math               80,000     800     2.8%
    #  science            50,000     500     1.7%
    #  ──────────────────────────────────────────
    #  TOTAL           2,880,000  28,800   100.0%
    # ──────────────────────────────────────────────────────────────────────────

    # ── Chinchilla: 20 × 189M params = 3.78B tokens óptimos ──────────────────
    # Con ~586 tokens/muestra → necesitamos ~6.4M muestras
    # alpaca_instruct capped en 100k (yahma 51k + c-s-ale 52k = 103k máximo)
    # Los tokens extra se cubren expandiendo wiki + code (fuentes inagotables)
    # Total estimado: 6.7M muestras → ~3.93B tokens ≈ Chinchilla ✅
    TRAIN_LIMITS = {
        "narrative_es":  1_745_000,   # CulturaX 200M — inagotable
        "narrative_en":    814_000,   # TinyStories 2.1M + CulturaX EN — OK
        "chat_es":         931_000,   # OpenHermes ES 1M — 93% de la fuente
        "chat_en":         814_000,   # OpenHermes EN 1M — 81% de la fuente
        "alpaca_instruct": 100_000,   # CAPPED: yahma+c-s-ale = 103k máximo
        "wiki_es":         530_000,   # Wikipedia ES 1.9M — OK
        "wiki_en":         530_000,   # Wikipedia Simple + EN completa — OK
        "code_python":     698_000,   # CodeParrot 18M — OK
        "code_other":      233_000,   # fineweb-edu-dedup — texto educativo web, parquet ✅
        "math":            186_000,   # finemath-4plus 6.7M — OK
        "science":         116_000,   # finemath-3plus 21.4M — OK
    }

    # Val = 1% del train (mínimo 500 por categoría)
    VAL_LIMITS = {k: max(500, v // 100) for k, v in TRAIN_LIMITS.items()}

    # Set compartido: los hashes de val se excluyen automáticamente del train.
    # Garantiza que ningún ejemplo aparezca en ambos splits.
    global_seen_hashes = set()

    print("\n🔵 FASE 1: Generando validación...")
    encode_dataset(
        os.path.join(DRIVE_FOLDER, "val_child.bin"),
        VAL_LIMITS,
        seen_hashes=global_seen_hashes
    )
    print(f"   Hashes acumulados tras val: {len(global_seen_hashes):,}")

    print("\n🔴 FASE 2: Generando entrenamiento...")
    encode_dataset(
        os.path.join(DRIVE_FOLDER, "train_child.bin"),
        TRAIN_LIMITS,
        seen_hashes=global_seen_hashes
    )

    print("\n" + "=" * 60)
    print("💡 Archivos generados:")
    for fname in ["train_child.bin", "val_child.bin"]:
        fpath = os.path.join(DRIVE_FOLDER, fname)
        if os.path.exists(fpath):
            size_gb = os.path.getsize(fpath) / 1e9
            tokens  = os.path.getsize(fpath) // 4  # int32 = 4 bytes
            print(f"   {fname}: {size_gb:.2f} GB — {tokens:,} tokens")
    print("=" * 60)

🔐 Iniciando sesión en Hugging Face...
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful
✅ Login exitoso.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[15:04:26] ⏳ Cargando NeoTokenizer desde neotoken_22k.pkl...
✅ NeoToken cargado: 22619 tokens en vocabulario.
[15:04:26] ✅ Tokenizer listo — vocab: 22619 tokens.

🔵 FASE 1: Generando validación...

⚡ GENERANDO DATASET: /content/drive/MyDrive/CryoDataset/val_child.bin

[15:04:26] 🎯 CATEGORÍA: NARRATIVE_ES → meta 17,450 muestras
[15:04:26] 👉 Fuente 1/2: uonlp/CulturaX (necesito 17,450)


Resolving data files:   0%|          | 0/512 [00:00<?, ?it/s]

[15:04:27]    ✅ Conectado (modo Parquet/nativo).


   CulturaX:   0%|          | 0/17450 [00:00<?, ?docs/s]

[15:05:22]    🎉 Meta cumplida (17,500/17,450). Dupes saltados: 0
[15:05:22]    ☁️  Backup a Drive OK (7,902,713 tokens hasta ahora).

[15:05:22] 🎯 CATEGORÍA: NARRATIVE_EN → meta 8,140 muestras
[15:05:22] 👉 Fuente 1/3: roneneldan/TinyStories (necesito 8,140)


...

Resolving data files:   0%|          | 0/128 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/128 [00:00<?, ?it/s]

[00:31:30]    ✅ Conectado (modo Parquet/nativo).


   finemath:   0%|          | 0/116000 [00:00<?, ?docs/s]

[00:49:54]    🎉 Meta cumplida (116,481/116,000). Dupes saltados: 666,554
[00:54:46]    ☁️  Backup a Drive OK (3,949,605,714 tokens hasta ahora).
[00:54:46] 🚀 Movimiento final a Drive...

✅ FINALIZADO
   Muestras únicas:  6,695,192
   Duplicados saltados: 666,554
   Tokens:           3,949,605,714
   Hashes en memoria: 7,605,099
   Archivo:          /content/drive/MyDrive/CryoDataset/train_child.bin

💡 Archivos generados:
   train_child.bin: 15.80 GB — 3,949,605,714 tokens
   val_child.bin: 0.16 GB — 40,385,804 tokens


### Enviarlo a Google CLOud Console

In [ ]:
# Celda 1 — Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Celda 2 — Autenticar GCP (te va a pedir login con tu cuenta de Google Cloud)
!gcloud auth login

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=SIXCxaWZlBjkVPYlwmf3S07D8gKp9J&prompt=consent&token_usage=remote&access_type=offline&code_challenge=2eNrbqXOSF409kaMBkq9bQDjsuA5Kq1B_e5yM61CKlQ&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AfrIepDIMqPVmCmz1cwCXxcbAEHMXKSDxbHU1wBowHxioKrLVev8V_i00cTDNeSTqpRYhw

You are now logged in as [zhoyelputoamoxd@gmail.com].
Your current proj

In [ ]:
!gcloud config set project brave-joy-448913-k1   # ← cambia esto

Updated property [core/project].


In [ ]:
# Celda 3 — Copiar los .bin de Drive a GCS
# -m = paralelo, mucho más rápido en archivos grandes
!gsutil -m cp \
  /content/drive/MyDrive/CryoDataset/train_child.bin \
  /content/drive/MyDrive/CryoDataset/val_child.bin \
  /content/drive/MyDrive/CryoDataset/neotoken_22k.pkl \
  gs://cryo-dataset-v4/CryoDataset/

Copying file:///content/drive/MyDrive/CryoDataset/train_child.bin [Content-Type=application/octet-stream]...
Copying file:///content/drive/MyDrive/CryoDataset/neotoken_22k.pkl [Content-Type=application/octet-stream]...
Copying file:///content/drive/MyDrive/CryoDataset/val_child.bin [Content-Type=application/octet-stream]...
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.

KeyboardInterrupt: 

# NeoTransformer


### Validamos la existencia del dataset

In [ ]:
import os
import numpy as np
from google.colab import drive
from concurrent.futures import ThreadPoolExecutor

# 1. Conectar a Google Drive
print("📂 Conectando a Google Drive...")
drive.mount('/content/drive')

# 2. Definir la ruta base
drive_folder = "/content/drive/MyDrive/CryoDataset"

# 3. Archivos a buscar (corregida la lista)
file_names = [
    "train_child_patch.bin",
    "train_child.bin",
    "val_child_patch.bin",
    "val_child.bin"
]

# Function to process a single file
def count_tokens_in_file(file_path, fname):
    try:
        if os.path.exists(file_path):
            # np.fromfile is efficient for binary data and releases GIL
            tokens = np.fromfile(file_path, dtype=np.int32)
            num_tokens = len(tokens)
            print(f"   ✅ {fname}: {num_tokens:,} tokens")
            return num_tokens
        else:
            print(f"   ❌ {fname}: No encontrado")
            return 0
    except Exception as e:
        print(f"   ⚠️ Error al leer {fname}: {e}")
        return 0

total_tokens_count = 0

print(f"\n🔍 Buscando archivos en {drive_folder} y contando tokens de manera concurrente...")

# 4. Iterar sobre los archivos y contarlos en paralelo
# Using ThreadPoolExecutor for I/O-bound tasks like reading from disk/drive
with ThreadPoolExecutor(max_workers=4) as executor: # Use up to 4 threads for concurrent I/O
    future_to_file = {
        executor.submit(count_tokens_in_file, os.path.join(drive_folder, fname), fname):
        fname for fname in file_names
    }
    for future in future_to_file:
        total_tokens_count += future.result()

print(f"\n📊 Total de tokens encontrados en todos los archivos: {total_tokens_count:,}")


📂 Conectando a Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🔍 Buscando archivos en /content/drive/MyDrive/CryoDataset y contando tokens de manera concurrente...
   ✅ val_child_patch.bin: 35,508,060 tokens
   ✅ val_child.bin: 35,893,234 tokens
   ✅ train_child.bin: 1,424,375,684 tokens
   ✅ train_child_patch.bin: 1,936,650,042 tokens

📊 Total de tokens encontrados en todos los archivos: 3,432,427,020


### NeoTransformer + CryoCCM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass, field
from torch.utils.checkpoint import checkpoint as _gradient_checkpoint
# ==============================================================================
# 0. CONFIGURACIÓN V4.2 (GOLDEN MASTER - TUNED DEFAULTS)
# ==============================================================================

@dataclass
class CryoConfig:
    model_name: str = "Cryo-CCM-V4.2-GM-206M-ctx2048"
    vocab_size: int = 22619

    # --- ARQUITECTURA CORE ---
    hidden_size: int = 768
    num_query_heads: int = 12
    num_kv_heads: int = 4           # GQA

    # Torres HRM (Dual Stream)
    n_layers_hielo: int = 15        # Razonamiento Profundo
    n_layers_trapos: int = 6        # Percepción Rápida

    block_size: int = 2048

    # --- MEMORY SYSTEM (RAG INTERNO DIFERENCIABLE) ---
    enable_memory: bool = True
    num_memory_slots: int = 64
    memory_write_heads: int = 4     # Escritura multi-head para detalle

    # --- ACT V4.2 (ADAPTIVE COMPUTE TIME) ---
    max_steps_act: int = 3
    ponder_penalty_beta: float = 0.01
    # Prior geométrico PonderNet: p(n) = λ*(1-λ)^(n-1)
    # λ=0.2 → E[pasos] = 1/λ = 5, pero con max_steps=3 el prior se trunca ~ paso 2.
    # Necesario para estabilidad del halt_head — sin él colapsa a siempre-0 o siempre-1.
    ponder_prior_lambda: float = 0.2
    enable_early_exit: bool = True

    # Uncertainty Params
    uncertainty_threshold: float = 0.35     # Ajustado post-auditoría (era 0.3)
    uncertainty_penalty_weight: float = 1.0

    # Self-Critic Params
    enable_self_critic: bool = True
    self_critic_weight: float = 0.015       # AJUSTE SEGURIDAD: Reducido de 0.05 para evitar ruido inicial

    # --- OPTIMIZACIONES ---
    use_gradient_checkpointing: bool = True
    mlp_expansion_ratio: float = 11/3

    # Training Defaults (Metadata)
    learning_rate: float = 6e-4
    warmup_steps: int = 2000
    total_steps: int = 100000
    min_lr: float = 6e-5
    dropout: float = 0.1
    use_amp: bool = True
    gradient_clip: float = 1.0

def trunc_normal_init_(tensor: torch.Tensor, std: float = 0.02):
    with torch.no_grad():
        nn.init.trunc_normal_(tensor, std=std)
    return tensor

# ==============================================================================
# 1. BLOQUES FUNDAMENTALES (RMSNorm, SwiGLU, RoPE, GQA)
# ==============================================================================

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_f = x.float()
        norm = x_f * torch.rsqrt(x_f.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(x.dtype)

class SwiGLU(nn.Module):
    def __init__(self, hidden_size: int, expansion: float = 11/3, dropout: float = 0.0):
        super().__init__()
        inter_size = int(hidden_size * expansion)
        inter_size = 256 * ((inter_size + 255) // 256)
        self.w1 = nn.Linear(hidden_size, inter_size, bias=False)
        self.w2 = nn.Linear(hidden_size, inter_size, bias=False)
        self.w3 = nn.Linear(inter_size, hidden_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 2048, theta: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, x, seq_len: int):
        # DECISIÓN PROGRAMADOR: Mantener Assert.
        # Fallar ruidosamente es mejor que corromper silenciosamente los datos posicionales.
        assert seq_len <= self.cos_cached.shape[0], \
            f"Sequence length {seq_len} exceeds RoPE cache {self.cos_cached.shape[0]}. Check your DataLoader."

        return self.cos_cached[:seq_len, :].to(x.device), self.sin_cached[:seq_len, :].to(x.device)

def apply_rotary_pos_emb(q, k, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(2)
    q1, q2 = q.chunk(2, dim=-1)
    k1, k2 = k.chunk(2, dim=-1)
    q_emb = (q * cos) + (torch.cat((-q2, q1), dim=-1) * sin)
    k_emb = (k * cos) + (torch.cat((-k2, k1), dim=-1) * sin)
    return q_emb, k_emb

class GroupedQueryAttention(nn.Module):
    def __init__(self, hidden_size: int, num_q_heads: int, num_kv_heads: int, dropout: float = 0.0):
        super().__init__()
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.num_groups = num_q_heads // num_kv_heads
        self.head_dim = hidden_size // num_q_heads

        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_size, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)
        self.dropout = dropout

    def forward(self, x, cos, sin):
        B, T, C = x.size()
        q = self.q_norm(self.q_proj(x).view(B, T, self.num_q_heads, self.head_dim))
        k = self.k_norm(self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim))
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim)

        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        if self.num_groups > 1:
            # FIX: repeat_interleave en lugar de expand().reshape() — evita tensores
            # no-contiguos con strides=0 que pueden causar corrupción silenciosa en CUDA.
            # LLaMA-3 y Mistral usan esta misma solución.
            k = k.repeat_interleave(self.num_groups, dim=2)
            v = v.repeat_interleave(self.num_groups, dim=2)

        y = F.scaled_dot_product_attention(
            q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2),
            dropout_p=self.dropout if self.training else 0.0, is_causal=True
        )
        return self.o_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class Block(nn.Module):
    def __init__(self, config: CryoConfig):
        super().__init__()
        self.ln1 = RMSNorm(config.hidden_size)
        self.attn = GroupedQueryAttention(config.hidden_size, config.num_query_heads, config.num_kv_heads, config.dropout)
        self.ln2 = RMSNorm(config.hidden_size)
        self.mlp = SwiGLU(config.hidden_size, config.mlp_expansion_ratio, config.dropout)
        self.use_checkpoint = config.use_gradient_checkpointing

    def forward(self, x, cos, sin):
        if self.use_checkpoint and self.training:
            x = x + _gradient_checkpoint(self._attn_forward, self.ln1(x), cos, sin, use_reentrant=False)
            x = x + _gradient_checkpoint(self._mlp_forward, self.ln2(x), use_reentrant=False)
        else:
            x = x + self.attn(self.ln1(x), cos, sin)
            x = x + self.mlp(self.ln2(x))
        return x

    def _attn_forward(self, x, cos, sin): return self.attn(x, cos, sin)
    def _mlp_forward(self, x): return self.mlp(x)

# ==============================================================================
# 2. SISTEMA DE MEMORIA OPTIMIZADO (RAG INTERNO)
# ==============================================================================

@dataclass
class HRMCarry:
    z_H: torch.Tensor
    z_L: torch.Tensor
    memory_bank: torch.Tensor = None  # [B, M, D]

class MemorySystem(nn.Module):
    """
    Sistema de memoria diferenciable (Neural Turing Machine style).
    - Escritura Multi-Head con ventana temporal (tokens recientes).
    - Addressing completo sobre todos los slots.
    """
    def __init__(self, config: CryoConfig):
        super().__init__()
        self.num_slots = config.num_memory_slots
        self.dim = config.hidden_size
        self.write_heads = config.memory_write_heads

        # Inicialización aprendible
        self.mem_init = nn.Parameter(torch.randn(1, self.num_slots, self.dim) * 0.02)

        # Lectura
        self.read_query = nn.Linear(self.dim, self.dim, bias=False)
        self.read_gate = nn.Linear(self.dim, 1)

        # Escritura
        self.write_query = nn.Linear(self.dim, self.dim, bias=False)
        self.write_erase = nn.Linear(self.dim, self.dim, bias=False)
        self.write_add = nn.Linear(self.dim, self.dim, bias=False)
        self.write_gate_strength = nn.Linear(self.dim, 1)

    def init_memory(self, batch_size, device):
        return self.mem_init.expand(batch_size, -1, -1).to(device).contiguous()

    def read(self, query_state, memory_bank):
        B, T, D = query_state.shape
        query = self.read_query(query_state)

        scores = torch.matmul(query, memory_bank.transpose(1, 2)) / math.sqrt(D)
        weights = F.softmax(scores, dim=-1)
        retrieved = torch.matmul(weights, memory_bank)

        read_strength = torch.sigmoid(self.read_gate(query_state))
        return retrieved * read_strength

    def write(self, write_state, memory_bank):
        """
        Escritura multi-head secuencial sobre los últimos K tokens.
        """
        B, T, D = write_state.shape

        # Seleccionar contexto rico: Últimos K tokens
        k_recent = min(T, self.write_heads)
        recent_tokens = write_state[:, -k_recent:, :]

        if k_recent < self.write_heads:
             mean_state = write_state.mean(dim=1, keepdim=True)
             padding_needed = self.write_heads - k_recent
             padding = mean_state.expand(-1, padding_needed, -1)
             recent_tokens = torch.cat([recent_tokens, padding], dim=1)

        # Generar señales
        query = self.write_query(recent_tokens)      # [B, K, D]
        erase_vec = torch.sigmoid(self.write_erase(recent_tokens))
        add_vec = torch.tanh(self.write_add(recent_tokens))

        # Addressing
        scores = torch.matmul(query, memory_bank.transpose(1, 2)) / math.sqrt(D)
        write_weights = F.softmax(scores, dim=-1)

        # Loop secuencial para actualización correcta
        new_memory = memory_bank
        for k in range(self.write_heads):
            w = write_weights[:, k:k+1, :].transpose(1, 2) # [B, M, 1]
            e = erase_vec[:, k:k+1, :]                     # [B, 1, D]
            a = add_vec[:, k:k+1, :]                       # [B, 1, D]

            erase_term = 1.0 - (w * e)
            add_term = w * a
            new_memory = new_memory * erase_term + add_term

        # Gate global
        global_state = write_state.mean(dim=1, keepdim=True)
        strength = torch.sigmoid(self.write_gate_strength(global_state))

        final_memory = memory_bank * (1 - strength) + new_memory * strength
        return final_memory

# ==============================================================================
# 3. HRM CORE V4.2 (LIMPIO Y OPTIMIZADO)
# ==============================================================================

class HierarchicalReasoningModel(nn.Module):
    def __init__(self, config: CryoConfig):
        super().__init__()
        self.config = config

        self.token_emb = nn.Embedding(config.vocab_size, config.hidden_size)
        head_dim = config.hidden_size // config.num_query_heads
        self.rope = RotaryEmbedding(head_dim, config.block_size * 2)
        self.dropout = nn.Dropout(config.dropout)

        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.halt_head = nn.Linear(config.hidden_size, 1, bias=True)

        self.token_emb.weight = self.lm_head.weight

        if config.enable_memory:
            self.memory = MemorySystem(config)
            self.mem_fusion_gate = nn.Parameter(torch.tensor(0.5))

        self.H_init = nn.Parameter(torch.randn(1, 1, config.hidden_size) * 0.02)
        self.L_init = nn.Parameter(torch.randn(1, 1, config.hidden_size) * 0.02)

        self.H_layers = nn.ModuleList([Block(config) for _ in range(config.n_layers_hielo)])
        self.L_layers = nn.ModuleList([Block(config) for _ in range(config.n_layers_trapos)])

        self.apply(self._init_weights)
        with torch.no_grad():
            self.halt_head.bias.fill_(-1.5)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            trunc_normal_init_(module.weight)
            if module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            trunc_normal_init_(module.weight)

    def init_carry(self, batch_size: int, seq_len: int, device):
        mem = None
        if self.config.enable_memory:
            mem = self.memory.init_memory(batch_size, device)

        return HRMCarry(
            z_H=self.H_init.expand(batch_size, seq_len, -1).contiguous(),
            z_L=self.L_init.expand(batch_size, seq_len, -1).contiguous(),
            memory_bank=mem
        )

    def forward_computation(self, z_H, z_L, memory_bank, input_emb, cos, sin):
        # 1. Torre L
        curr_L = z_L + input_emb
        for block in self.L_layers:
            curr_L = block(curr_L, cos, sin)

        # 2. Torre H — integración directa desde Torre L (diseño canónico HRM 2506.21734)
        # z_H_new = f_H(z_H + curr_L): suma directa es la implementación del paper original.
        curr_H = z_H + curr_L

        if self.config.enable_memory and memory_bank is not None:
            context = self.memory.read(curr_H, memory_bank)
            gate = torch.sigmoid(self.mem_fusion_gate)
            curr_H = curr_H + gate * context

        for block in self.H_layers:
            curr_H = block(curr_H, cos, sin)

        # 3. Write Memory
        new_memory = memory_bank
        if self.config.enable_memory and memory_bank is not None:
            new_memory = self.memory.write(curr_H, memory_bank)

        # 4. Heads
        logits = self.lm_head(curr_H)
        halt_logit = self.halt_head(curr_H).squeeze(-1)

        return curr_H, curr_L, new_memory, logits, halt_logit

# ==============================================================================
# 4. LOSS HEAD V4.2 (COMPLETO Y OPTIMIZADO)
# ==============================================================================

class ACTLoss(nn.Module):
    def __init__(self, model: HierarchicalReasoningModel):
        super().__init__()
        self.model = model

    def compute_entropy(self, logits):
        """Incertidumbre por token"""
        probs = F.softmax(logits, dim=-1)
        entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=-1)
        max_entropy = math.log(logits.size(-1))
        return entropy / max_entropy

    def self_critic_loss(self, all_logits, labels, p_stop):
        num_steps = len(all_logits)
        if num_steps < 2: return torch.tensor(0.0, device=labels.device)

        step_correct = []
        for logits in all_logits:
            preds = logits.argmax(dim=-1)
            correct = ((preds == labels) & (labels != -100)).float()
            step_correct.append(correct.mean().item())  # FIX: .item() evita comparación tensor vs tensor

        critic_loss = torch.tensor(0.0, device=labels.device)

        for i in range(num_steps - 1):
            curr_acc = step_correct[i]
            next_acc = step_correct[i + 1]

            if next_acc > curr_acc:
                reward = torch.tensor(next_acc - curr_acc, device=labels.device) * p_stop[i + 1].mean()
                critic_loss -= reward
            elif next_acc < curr_acc:
                penalty = torch.tensor(curr_acc - next_acc, device=labels.device) * p_stop[i + 1].mean()
                critic_loss += penalty

            if curr_acc > 0.9 and next_acc > 0.9:
                early_bonus = p_stop[i].mean() * 0.05
                critic_loss -= early_bonus

        return critic_loss / (num_steps - 1)

    def forward(self, input_ids, labels):
        B, T = input_ids.shape
        device = input_ids.device
        config = self.model.config

        input_emb = self.model.token_emb(input_ids)
        input_emb = self.model.dropout(input_emb)
        cos, sin = self.model.rope(input_emb, T)
        carry = self.model.init_carry(B, T, device)

        all_halt_probs = []
        all_losses = []      # OOM FIX: loss por step [B,T] en vez de logits [B,T,V] — 1000x menos memoria

        flat_labels = labels.view(-1)
        valid_mask = flat_labels != -100
        if valid_mask.sum() == 0:
            return {"loss_combined": torch.tensor(0.0, device=device, requires_grad=True)}

        for step in range(config.max_steps_act):
            new_z_H, new_z_L, new_mem, logits, halt_logit = self.model.forward_computation(
                carry.z_H, carry.z_L, carry.memory_bank,
                input_emb, cos, sin
            )

            carry.z_H = new_z_H
            carry.z_L = new_z_L
            carry.memory_bank = new_mem

            entropy = self.compute_entropy(logits)
            entropy_seq = entropy.mean(dim=1, keepdim=True)
            uncertainty_penalty = torch.relu(entropy_seq - config.uncertainty_threshold)
            adjusted_halt_logit = halt_logit - (uncertainty_penalty * config.uncertainty_penalty_weight)
            halt_prob = torch.sigmoid(adjusted_halt_logit)

            # OOM FIX: calcular cross-entropy ahora mismo y liberar logits grandes [B,T,V]
            # En vez de acumular 3 × [B,T,22619] tensores = ~1.1GB, acumulamos 3 × [B,T] = ~3MB
            step_loss = F.cross_entropy(
                logits.view(-1, config.vocab_size),
                flat_labels, ignore_index=-100, reduction='none'
            ).view(B, T)
            all_losses.append(step_loss)
            del logits  # liberar inmediatamente — no se necesita más

            all_halt_probs.append(halt_prob)

            # Early Exit
            if config.enable_early_exit and step > 0:
                confident = halt_prob.mean() > 0.95
                certain = entropy.mean() < config.uncertainty_threshold
                if confident and certain:
                    all_halt_probs[-1] = torch.ones_like(halt_prob)
                    break

        num_steps = len(all_losses)
        stack_halt_probs = torch.stack(all_halt_probs, dim=0)
        stack_losses = torch.stack(all_losses, dim=0)   # [S, B, T] — pequeño, sin logits

        p_continue = torch.cumprod(1.0 - stack_halt_probs[:-1], dim=0)
        p_continue = torch.cat([torch.ones(1, B, T, device=device), p_continue], dim=0)
        p_stop = stack_halt_probs * p_continue
        residual = 1.0 - p_stop.sum(dim=0)
        p_stop[-1] += residual

        weighted_loss = (p_stop * stack_losses).sum(dim=0)
        recon_loss = weighted_loss[labels != -100].mean()

        steps_range = torch.arange(1, num_steps + 1, device=device).view(-1, 1, 1)
        expected_steps = (p_stop * steps_range).sum(dim=0)

        # PonderNet geometric prior: KL(p_stop || p_prior) con p_prior ~ Geometric(λ_p)
        lambda_p = config.ponder_prior_lambda
        steps_idx = torch.arange(1, num_steps + 1, device=device).float().view(-1, 1, 1)
        log_prior = (steps_idx - 1) * math.log(1.0 - lambda_p) + math.log(lambda_p)
        p_stop_safe = p_stop.clamp(min=1e-8)
        kl_loss = (p_stop_safe * (p_stop_safe.log() - log_prior)).sum(dim=0).mean()
        ponder_loss = config.ponder_penalty_beta * kl_loss

        # Self-critic: ya no tiene acceso a logits — desactivado durante training para ahorrar VRAM.
        # El halt_head se guía solo por PonderNet KL + uncertainty penalty, que es suficiente.
        critic_loss = torch.tensor(0.0, device=device)

        total_loss = recon_loss + ponder_loss + critic_loss

        with torch.no_grad():
            # Accuracy aproximada usando losses (menor loss = mejor predicción)
            # No necesita logits — evita el OOM del bloque anterior
            avg_loss_per_token = stack_losses.mean(dim=0)  # [B, T]
            accuracy = (avg_loss_per_token[labels != -100] < math.log(config.vocab_size) * 0.5).float().mean()

        return {
            "loss_combined": total_loss,
            "lm_loss": recon_loss,
            "ponder_cost": ponder_loss,
            "critic_loss": critic_loss,
            "avg_steps": expected_steps.mean(),
            "accuracy": accuracy
        }

# ==============================================================================
# 5. WRAPPER FINAL (CON INFERENCIA CONSISTENTE)
# ==============================================================================

class CryoCCM(nn.Module):
    def __init__(self, config: CryoConfig):
        super().__init__()
        self.config = config
        self.model = HierarchicalReasoningModel(config)
        self.loss_head = ACTLoss(self.model)

    def forward(self, input_ids, labels=None):
        """
        Forward híbrido:
        - Labels != None -> Training Loss
        - Labels == None -> Inference Logits (Consistent Halting)
        """
        if labels is not None:
             return None, self.loss_head(input_ids, labels)

        # MODO INFERENCIA — torch.no_grad() interno para eficiencia
        # aunque el caller no lo especifique explícitamente
        with torch.no_grad():
            B, T = input_ids.shape
            device = input_ids.device
            config = self.config

            input_emb = self.model.token_emb(input_ids)
            input_emb = self.model.dropout(input_emb)
            cos, sin = self.model.rope(input_emb, T)
            carry = self.model.init_carry(B, T, device)

            all_logits = []
            all_halt_probs = []

            for step in range(config.max_steps_act):
                z_H, z_L, mem, logits, halt_logit = self.model.forward_computation(
                    carry.z_H, carry.z_L, carry.memory_bank,
                    input_emb, cos, sin
                )

                carry.z_H = z_H
                carry.z_L = z_L
                carry.memory_bank = mem

                # CONSISTENCIA: Aplicar misma lógica de incertidumbre que en training
                entropy = self.loss_head.compute_entropy(logits)
                entropy_seq = entropy.mean(dim=1, keepdim=True)

                uncertainty_penalty = torch.relu(entropy_seq - config.uncertainty_threshold)
                adjusted_halt_logit = halt_logit - (uncertainty_penalty * config.uncertainty_penalty_weight)
                halt_prob = torch.sigmoid(adjusted_halt_logit)

                all_logits.append(logits)
                all_halt_probs.append(halt_prob)

                if config.enable_early_exit and step > 0:
                    if halt_prob.mean() > 0.95:
                        break

            # Aggregation PonderNet
            stack_logits = torch.stack(all_logits, dim=0)
            stack_halt_probs = torch.stack(all_halt_probs, dim=0)

            p_continue = torch.cumprod(1.0 - stack_halt_probs[:-1], dim=0)
            p_continue = torch.cat([torch.ones(1, B, T, device=device), p_continue], dim=0)
            p_stop = stack_halt_probs * p_continue

            p_stop[-1] += 1.0 - p_stop.sum(dim=0)

            avg_logits = (stack_logits * p_stop.unsqueeze(-1)).sum(dim=0)

        return avg_logits, None

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable

if __name__ == "__main__":
    print("=" * 70)
    print("🧠 CRYO-CCM V4.2 GOLDEN MASTER (AUDITED)")
    print("=" * 70)

    config = CryoConfig()
    model = CryoCCM(config)

    total, trainable = model.count_parameters()

    print(f"\n📊 PARÁMETROS:")
    print(f"  • Total: {total:,} ({total/1e6:.1f}M)")

    print(f"\n✅ ESTADO FINAL:")
    print(f"  • Safety Weights: APLICADOS (Self-Critic 0.015)")
    print(f"  • Inferencia: CONSISTENTE (Training Logic)")
    print(f"  • Integridad: RoPE ASSERT ACTIVO")

    print("=" * 70)

🧠 CRYO-CCM V4.2 GOLDEN MASTER (AUDITED)

📊 PARÁMETROS:
  • Total: 189,098,116 (189.1M)

✅ ESTADO FINAL:
  • Safety Weights: APLICADOS (Self-Critic 0.015)
  • Inferencia: CONSISTENTE (Training Logic)
  • Integridad: RoPE ASSERT ACTIVO


### Entrenamiento

In [ ]:
import os
import time
import math
import glob
import shutil
import datetime
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from contextlib import nullcontext
import gc

# Importar modelo — necesario para apply_curriculum (MemorySystem) y train()
from NeoTransformer import CryoConfig, CryoCCM, MemorySystem

# ==============================================================================
# CONFIGURACIÓN
# ==============================================================================


# ==============================================================================
# PATHS — Vertex AI Workbench (Jupyter Notebook)
# ==============================================================================
# El home en Vertex AI Workbench es /home/jupyter/ — NO existe /content/.
#
# OPCIÓN A — Datos en disco local de la instancia (más rápido, sin persistencia):
#   Sube tus archivos .bin con el file browser de Jupyter a /home/jupyter/CryoDataset/
DATA_DIR       = "/home/jupyter/CryoDataset"
LOCAL_DATA_DIR = "/home/jupyter/CryoDataset"   # misma carpeta, ya están en SSD local
CHECKPOINT_DIR = "/home/jupyter/CryoDataset/checkpoints_v4_2_ctx2048"
#
# OPCIÓN B — Datos en GCS (bucket montado vía FUSE, persiste si la instancia muere):
#   Requiere activar "Mount Cloud Storage bucket" en la config de la instancia.
#   El bucket se monta automáticamente en /gcs/<nombre-del-bucket>/
#   Descomenta y reemplaza TU_BUCKET por el nombre real de tu bucket:
# DATA_DIR       = "/gcs/TU_BUCKET/CryoDataset"
# LOCAL_DATA_DIR = "/home/jupyter/tmp_data"     # caché SSD local para lecturas rápidas
# CHECKPOINT_DIR = "/gcs/TU_BUCKET/checkpoints_v4_2_ctx2048"
# ==============================================================================

# OOM FIX: Forzar allocator expandible ANTES de cualquier operación CUDA.
# Elimina fragmentación — PyTorch divide bloques grandes en vez de buscar
# bloques contiguos que no existen tras 3 pasos ACT acumulados.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Hardware
BATCH_SIZE = 2           # OOM FIX: 3 pasos ACT × vocab 22k × block 2048 fragmenta VRAM.
                         # batch=2 + grad_accum=16 = effective batch 32, igual que siempre.
GRAD_ACCUM = 16          # Effective batch = 32 — misma dinámica, sin cambios al modelo.
BLOCK_SIZE = 2048        # PUNTO DULCE: 25-30 turnos de conversación en contexto
DTYPE = 'bfloat16'       # bfloat16 > float16 en L4/A100: más estable, mismo rendimiento

# Optimizador
LEARNING_RATE = 3e-4     # Reducido vs 6e-4: bloques más largos = gradientes más ricos
MIN_LR = 3e-5            # 10% del LR máximo (regla estándar cosine)
WARMUP_STEPS = 2000      # Más warmup para bloques largos (más tokens por step)
MAX_STEPS = 60272        # Calculado: 3.95B tokens / (2*16*2048) = 60,272 steps — igual, effective batch 32 no cambia
WEIGHT_DECAY = 1e-1
BETA1 = 0.9
BETA2 = 0.95
GRAD_CLIP = 1.0

# Curriculum & Validation
PHASE_1_END = 3000       # Ajustado a los nuevos steps totales (~5% del total)
PHASE_2_END = 9000       # Ajustado (~15% del total)
EVAL_INTERVAL = 500      # Validar cada 500 steps (más frecuente, steps más lentos)
EVAL_ITERS = 50          # Cuantos batches usar para validar
PATIENCE = 10            # Cuantas validaciones sin mejora antes de declarar saturación

# Constantes de NeoToken para Domain Monitor
SCALE_CODE = 15
SCALE_MATH = 16

# ==============================================================================
# DATASET REPRODUCIBLE (MODO GOLDEN MASTER: ALL-IN)
# ==============================================================================

def prepare_data_environment():
    print(f"\n🚀 PREPARANDO DATOS...")
    print(f"   DATA_DIR:       {DATA_DIR}")
    print(f"   LOCAL_DATA_DIR: {LOCAL_DATA_DIR}")
    print(f"   CHECKPOINT_DIR: {CHECKPOINT_DIR}")

    if not os.path.exists(DATA_DIR):
        print(f"❌ Directorio de datos no encontrado: {DATA_DIR}")
        print(f"   Crea la carpeta y sube tus archivos .bin antes de correr train().")
        exit(1)

    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # Buscar archivos .bin en DATA_DIR
    remote_files = glob.glob(os.path.join(DATA_DIR, "*child*.bin"))
    if not remote_files:
        print(f"❌ No se encontraron archivos *child*.bin en {DATA_DIR}")
        exit(1)

    local_files = []
    total_size_gb = 0

    print(f"📦 Verificando {len(remote_files)} archivos...")

    for f in remote_files:
        fname = os.path.basename(f)
        dst   = os.path.join(LOCAL_DATA_DIR, fname)
        size_gb = os.path.getsize(f) / (1024**3)
        total_size_gb += size_gb

        if DATA_DIR == LOCAL_DATA_DIR:
            # Datos ya en disco local — no copiar
            print(f"   ✓ {fname} ({size_gb:.2f} GB) — en disco local")
            local_files.append(f)
        elif not os.path.exists(dst):
            print(f"   -> Copiando {fname} ({size_gb:.2f} GB) a SSD local...")
            shutil.copy(f, dst)
            local_files.append(dst)
        else:
            print(f"   ✓ {fname} ya en caché local. Saltando.")
            local_files.append(dst)

    print(f"✅ {total_size_gb:.2f} GB listos para entrenamiento.")
    return local_files


class InfiniteMemmapDataset(IterableDataset):
    """Dataset con shuffle reproducible"""
    def __init__(self, bin_files, block_size, seed=1337):
        self.bin_files = bin_files
        self.block_size = block_size
        self.seed = seed
        self.mmaps = []
        self.total_tokens = 0

        for f in bin_files:
            try:
                m = np.memmap(f, dtype=np.int32, mode='r')
                self.mmaps.append(m)
                self.total_tokens += len(m)
            except Exception as e:
                print(f"⚠️ Error cargando {f}: {e}")

        print(f"📚 Dataset cargado: {self.total_tokens:,} tokens ({len(bin_files)} archivos)")

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        worker_id = worker_info.id if worker_info else 0
        rng = np.random.default_rng(self.seed + worker_id)

        while True:
            file_idx = rng.integers(len(self.mmaps))
            mmap = self.mmaps[file_idx]
            if len(mmap) < self.block_size + 1: continue
            ix = rng.integers(len(mmap) - self.block_size - 1)
            chunk = torch.from_numpy(mmap[ix : ix + self.block_size + 1].astype(np.int64))
            yield chunk[:-1], chunk[1:]

# ==============================================================================
# UTILIDADES & SMART MONITORING
# ==============================================================================

class SmartDomainMonitor:
    """Rastrea el rendimiento por dominio durante la validación."""
    def __init__(self):
        self.metrics = {"general": [], "code": [], "math": []}

    def update(self, loss: float, batch_ids: torch.Tensor):
        if (batch_ids == SCALE_MATH).any():
            self.metrics["math"].append(loss)
        elif (batch_ids == SCALE_CODE).any():
            self.metrics["code"].append(loss)
        else:
            self.metrics["general"].append(loss)

    def get_summary(self):
        summary = {}
        for k, v in self.metrics.items():
            summary[k] = sum(v) / len(v) if v else float('inf')
        return summary

    def reset(self):
        self.metrics = {"general": [], "code": [], "math": []}

def get_lr(it, max_steps=None):
    _max = max_steps if max_steps else MAX_STEPS
    if it < WARMUP_STEPS:
        return LEARNING_RATE * it / WARMUP_STEPS
    if it > _max:
        return MIN_LR
    decay_ratio = (it - WARMUP_STEPS) / (_max - WARMUP_STEPS)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (LEARNING_RATE - MIN_LR)

def apply_curriculum(model, step):
    raw_model = model.module if hasattr(model, "module") else model
    config = raw_model.config
    device = next(raw_model.parameters()).device

    if step < PHASE_1_END:
        if config.enable_memory:
            config.enable_memory = False
            raw_model.model.memory = None
        config.max_steps_act = 1
        config.enable_self_critic = False
        phase = "FASE 1 (Bebé)"
    elif step < PHASE_2_END:
        if not config.enable_memory:
            config.enable_memory = True
            # FIX: Crear MemorySystem con write_heads=2 (Fase 2).
            # Se inicializa con pesos pequeños (*0.02 en mem_init) — spike de loss
            # es mínimo y el modelo lo absorbe en los primeros steps de esta fase.
            config.memory_write_heads = 2
            raw_model.model.memory = MemorySystem(config).to(device)
            raw_model.model.mem_fusion_gate = torch.nn.Parameter(torch.tensor(0.5, device=device))
        config.max_steps_act = 2
        config.enable_self_critic = False
        phase = "FASE 2 (Niño)"
    else:
        # FIX: En Fase 3 NO recreamos MemorySystem — conservamos los pesos aprendidos
        # en Fase 2. Solo actualizamos write_heads en config (las capas lineales
        # ya tienen dim correcta, el sistema usa los primeros 2 heads).
        # Recrear aquí causaría pérdida de todo lo aprendido en Fase 2.
        config.memory_write_heads = 4
        config.max_steps_act = 3
        config.enable_self_critic = True
        phase = "FASE 3 (Adulto)"

    return phase

@torch.no_grad()
def estimate_loss(model, ctx, loader, eval_iters, domain_monitor=None):
    """Estima la pérdida (Loss) en un subset aleatorio."""
    model.eval()
    if domain_monitor: domain_monitor.reset()

    losses = []
    iter_loader = iter(loader)

    for _ in range(eval_iters):
        try:
            X, Y = next(iter_loader)
        except StopIteration:
            iter_loader = iter(loader)
            X, Y = next(iter_loader)

        X, Y = X.to(next(model.parameters()).device), Y.to(next(model.parameters()).device)
        with ctx:
            _, metrics = model(X, labels=Y)
            loss = metrics['loss_combined']
            losses.append(loss.item())

            if domain_monitor:
                domain_monitor.update(metrics['lm_loss'].item(), X)

    model.train()
    return np.mean(losses)

# ==============================================================================
# TRAINING LOOP PRINCIPAL
# ==============================================================================

def train():
    torch.manual_seed(1337)
    np.random.seed(1337)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    ctx = torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

    print(f"🔧 Device: {device}")

    # 1. Preparar Datos (ALL IN)
    all_files = prepare_data_environment()

    # Separar train (train_child.bin) y val (val_child.bin)
    train_files = [f for f in all_files if 'train' in os.path.basename(f)]
    val_files   = [f for f in all_files if 'val'   in os.path.basename(f)]

    if not train_files:
        print("❌ No se encontró train_child.bin"); exit(1)
    if not val_files:
        print("⚠️  No se encontró val_child.bin — usando train para validación (no ideal)")
        val_files = train_files

    train_dataset = InfiniteMemmapDataset(train_files, BLOCK_SIZE, seed=1337)
    val_dataset   = InfiniteMemmapDataset(val_files,   BLOCK_SIZE, seed=9999)  # seed distinto

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

    train_iter = iter(train_loader)

    # FIX: MAX_STEPS calculado desde el dataset real — no hardcodeado.
    # Si cambiás BATCH_SIZE o GRAD_ACCUM, se ajusta automáticamente.
    computed_max_steps = train_dataset.total_tokens // (BATCH_SIZE * GRAD_ACCUM * BLOCK_SIZE)
    print(f"📐 MAX_STEPS calculado: {computed_max_steps:,} "
          f"({train_dataset.total_tokens/1e9:.2f}B tokens / {BATCH_SIZE*GRAD_ACCUM*BLOCK_SIZE:,} tok/step)")

    # 2. Modelo
    config = CryoConfig()
    model = CryoCCM(config)
    model.to(device)
    total_params, _ = model.count_parameters()
    print(f"📊 Parámetros: {total_params:,} ({total_params/1e6:.1f}M)")

    # 3. Optimizador
    scaler = torch.amp.GradScaler('cuda', enabled=(DTYPE == 'float16'))  # bfloat16 no necesita scaler
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2), weight_decay=WEIGHT_DECAY, fused=True)

    # 4. Estado de Entrenamiento
    start_iter = 0
    best_val_loss = float('inf')
    patience_counter = 0
    domain_monitor = SmartDomainMonitor()

    # Cargar Checkpoint
    ckpt_path = os.path.join(CHECKPOINT_DIR, "ckpt_latest.pt")
    if os.path.exists(ckpt_path):
        print(f"🔄 Resumiendo desde checkpoint...")
        # FIX CRÍTICO: weights_only=False para permitir cargar CryoConfig
        checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)

        start_iter = checkpoint['iter_num']

        # FIX DE ARQUITECTURA DINÁMICA:
        # Aplicamos el curriculum ANTES de cargar los pesos.
        # Esto asegura que si estamos en Fase 1 (Bebé), el modelo elimine la memoria
        # y coincida estructuralmente con el checkpoint guardado.
        print(f"   ⚙️ Ajustando arquitectura al paso {start_iter}...")
        apply_curriculum(model, start_iter)

        # Ahora sí es seguro cargar
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

        best_val_loss = checkpoint.get('best_val_loss', float('inf'))
        patience_counter = checkpoint.get('patience_counter', 0)

        # Restaurar RNG
        torch.set_rng_state(checkpoint['rng_state'].cpu())
        np.random.set_state(checkpoint['numpy_rng_state'])
        print(f"   -> Step: {start_iter} | Best Loss: {best_val_loss:.4f}")

    # --- BUCLE DE ENTRENAMIENTO ---
    model.train()
    t0 = time.time()
    run_start_time = time.time()

    print("\n" + "="*70)
    print("🔥 ENTRENAMIENTO INICIADO (GOLDEN MASTER - ALL DATA)")
    print(f"   Batch: {BATCH_SIZE} | Accum: {GRAD_ACCUM} | Effective: {BATCH_SIZE * GRAD_ACCUM}")
    print("="*70 + "\n")

    for iter_num in range(start_iter, computed_max_steps):
        # A. Curriculum
        phase = apply_curriculum(model, iter_num)
        if iter_num == start_iter or iter_num in [PHASE_1_END, PHASE_2_END]:
            print(f"\n{'='*70}")
            print(f"🎓 {phase} | Step {iter_num}")
            print(f"   Memory: {model.config.enable_memory} | ACT Steps: {model.config.max_steps_act}")
            print("="*70 + "\n")

        # B. Learning Rate
        lr = get_lr(iter_num, computed_max_steps)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # C. Forward + Backward
        optimizer.zero_grad(set_to_none=True)
        accum_loss = 0.0
        accum_metrics = {}

        for _ in range(GRAD_ACCUM):
            try:
                X, Y = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                X, Y = next(train_iter)

            X, Y = X.to(device), Y.to(device)
            with ctx:
                _, metrics = model(X, labels=Y)
                loss = metrics['loss_combined'] / GRAD_ACCUM

            # NaN DETECTION: Si la loss explota, parar inmediatamente y guardar.
            # Sin esto el training corre 23h quemando créditos sin aprender nada.
            loss_val = loss.item()
            if not math.isfinite(loss_val):
                print(f"\n💀 NaN/Inf detectado en step {iter_num}! Loss = {loss_val}")
                print("   Guardando emergency checkpoint antes de parar...")
                emergency_path = os.path.join(CHECKPOINT_DIR, f"emergency_nan_step_{iter_num}.pt")
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'iter_num': iter_num,
                    'best_val_loss': best_val_loss,
                }, emergency_path)
                print(f"   ✅ Guardado en {emergency_path}")
                print("   Causa probable: LR demasiado alto o gradientes explodiendo.")
                print("   Reducí LEARNING_RATE a 1e-4 y retomá desde el último ckpt_latest.pt")
                return
            scaler.scale(loss).backward()
            accum_loss += loss_val

            # Métricas
            for k, v in metrics.items():
                if k not in accum_metrics: accum_metrics[k] = 0
                val = v.item() if isinstance(v, torch.Tensor) else v
                accum_metrics[k] += val

        # Promediar métricas
        for k in accum_metrics: accum_metrics[k] /= GRAD_ACCUM

        # D. Step
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        # E. Logging (UX PREMIUM REIMPLANTADA)
        t1 = time.time()
        dt = t1 - t0
        t0 = t1

        if iter_num % 10 == 0:
            # Cálculos de UX
            elapsed_run = time.time() - run_start_time
            steps_run = max(1, iter_num - start_iter + 1)
            avg_dt_run = elapsed_run / steps_run
            eta_seconds = (computed_max_steps - iter_num) * avg_dt_run
            eta_str = str(datetime.timedelta(seconds=int(eta_seconds)))

            tokens_per_sec = (BATCH_SIZE * GRAD_ACCUM * BLOCK_SIZE) / avg_dt_run  # FIX: media, no step actual
            vram = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
            torch.cuda.reset_peak_memory_stats()

            # Desglose COMPLETO (LM, Ponder, Critic)
            lm = accum_metrics.get('lm_loss', 0.0)
            ponder = accum_metrics.get('ponder_cost', 0.0)
            critic = accum_metrics.get('critic_loss', 0.0) # Recuperado!

            print(
                f"Step {iter_num:6d} | "
                f"Loss {accum_loss:.3f} (LM {lm:.2f} P {ponder:.2f} C {critic:.2f}) | "
                f"Acc {accum_metrics.get('accuracy', 0):.1%} | "
                f"ACT {accum_metrics.get('avg_steps', 0):.2f}s | "
                f"{int(tokens_per_sec/1000)}k t/s | "
                f"VRAM {vram:.1f}G | ETA {eta_str}"
            )

        # F. VALIDACIÓN & BEST MODEL (SMART TRAINING)
        if iter_num > 0 and iter_num % EVAL_INTERVAL == 0:
            print(f"\n🔍 Validando (Step {iter_num})...")
            val_loss = estimate_loss(model, ctx, val_loader, EVAL_ITERS, domain_monitor)
            dom_summary = domain_monitor.get_summary()

            def fmt(v): return f"{v:.3f}" if v != float('inf') else "N/A"
            print(f"   📊 Val Loss: {val_loss:.4f} | General: {fmt(dom_summary['general'])} | Code: {fmt(dom_summary['code'])} | Math: {fmt(dom_summary['math'])}")

            # OVERFITTING DETECTION: train_loss << val_loss indica memorización
            train_loss_current = accum_loss
            gap = val_loss - train_loss_current
            if iter_num > PHASE_2_END and gap > 1.0:
                print(f"   ⚠️  OVERFITTING SIGNAL: val={val_loss:.3f} vs train={train_loss_current:.3f} (gap={gap:.2f})")
                print(f"   Si el gap sigue creciendo en próximas evaluaciones, considera parar.")

            # UNDERFITTING DETECTION: loss no baja de ~8 después del warmup indica problema
            if iter_num > WARMUP_STEPS * 2 and val_loss > 8.0:
                print(f"   ⚠️  UNDERFITTING SIGNAL: loss={val_loss:.3f} muy alta tras {iter_num} steps.")
                print(f"   Posible causa: LR muy bajo, datos corruptos, o bug en el pipeline.")

            # Lógica Best Model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
                print(f"   ⭐ ¡NUEVO RECORD! Guardando Best Model en {best_model_path}")
                torch.save(model.state_dict(), best_model_path)
            else:
                patience_counter += 1
                print(f"   ⚠️ Sin mejora. Paciencia: {patience_counter}/{PATIENCE}")

                # Early Stopping real — para el training y avisa.
                # PATIENCE=10 evaluaciones × EVAL_INTERVAL=500 steps = 5000 steps sin mejora.
                # Si llegás acá en pretraining temprano (<20k steps) probablemente
                # es un LR muy bajo o un bug. Si es >40k steps es convergencia real.
                if patience_counter >= PATIENCE:
                    print(f"   🛑 EARLY STOPPING: {PATIENCE} evaluaciones sin mejora.")
                    print(f"   Mejor val loss logrado: {best_val_loss:.4f} en este run.")
                    sat_path = os.path.join(CHECKPOINT_DIR, f"saturated_step_{iter_num}.pt")
                    torch.save(model.state_dict(), sat_path)
                    print(f"   💾 Checkpoint saturación guardado: {sat_path}")
                    print("   El best_model.pt tiene el mejor modelo del run.")
                    return

            print("-" * 50)

        # G. Checkpoint Regular
        if iter_num > 0 and iter_num % 500 == 0:
            checkpoint_data = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'iter_num': iter_num,
                'config': config,
                'rng_state': torch.get_rng_state(),
                'numpy_rng_state': np.random.get_state(),
                'best_val_loss': best_val_loss,
                'patience_counter': patience_counter
            }
            torch.save(checkpoint_data, ckpt_path)
            if iter_num % 5000 == 0:
                shutil.copy(ckpt_path, os.path.join(CHECKPOINT_DIR, f"ckpt_{iter_num}.pt"))

if __name__ == "__main__":
    # ── Google Colab ──────────────────────────────────────────────────────────
    # Si estás en Colab, descomenta estas dos líneas:
    # from google.colab import drive
    # drive.mount('/content/drive')

    # ── Vertex AI Workbench ───────────────────────────────────────────────────
    # En Vertex AI no se necesita montar nada. Los datos vienen de GCS
    # (configura GCS_BUCKET arriba) o ya están en el disco local.
    # Ejecutar con: python Training.py
    # ─────────────────────────────────────────────────────────────────────────
    train()

📂 Montando Drive...
Mounted at /content/drive
🔧 Device: cuda

🚀 PREPARANDO DATOS (MODO GOLDEN MASTER - ALL IN)...
📦 Sincronizando 4 archivos a SSD...
   -> Copiando val_child_patch.bin (0.13 GB)...
   -> Copiando train_child_patch.bin (7.21 GB)...
   -> Copiando val_child.bin (0.13 GB)...
   -> Copiando train_child.bin (5.31 GB)...
✅ 12.79 GB listos. Usando TODO para entrenamiento.
📚 Dataset cargado: 3,432,427,020 tokens (4 archivos)
📚 Dataset cargado: 3,432,427,020 tokens (4 archivos)
📊 Parámetros: 189,098,116 (189.1M)
🔄 Resumiendo desde checkpoint...
   ⚙️ Ajustando arquitectura al paso 6500...
   -> Step: 6500 | Best Loss: 1.6984

🔥 ENTRENAMIENTO INICIADO (GOLDEN MASTER - ALL DATA)
   Batch: 4 | Accum: 8 | Effective: 32


🎓 FASE 2 (Niño) | Step 6500
   Memory: True | ACT Steps: 2

Step   6500 | Loss 1.660 (LM 1.65 P 0.01 C 0.00) | Acc 62.0% | ACT 1.00s | 2k t/s | VRAM 8.5G | ETA 8 days, 7:55:43
Step   6510 | Loss 1.267 (LM 1.26 P 0.01 C 0.00) | Acc 69.2% | ACT 1.00s | 2k t/s | VRAM 

# Uso del modelo

### Llamar a los modelos

In [ ]:
# Descargar desde GCS
!gcloud auth login

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=8v8Re0DOcvEuujAFQTb7UDGkDZOgJi&prompt=consent&token_usage=remote&access_type=offline&code_challenge=y-UxU5u8aQDmRybsUWUxby7abq8x8fNvkwvexMfCLPQ&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0Aci98E_ID_ZmCTLBxxBBQGjwbHPg6wIa7oSJQoK0i8OYACP20rtRzTcR9ke5G8jRf1CWgA

You are now logged in as [zhoyelputoamoxd@gmail.com].
Your current proj

In [ ]:
# Celda 1 — descargar desde GCS
!gsutil cp gs://cryo-dataset-v4/checkpoints_v4_2_ctx2048/best_model.pt /content/
!gsutil cp gs://cryo-dataset-v4/CryoDataset/neotoken_22k.pkl /content/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://cryo-dataset-v4/checkpoints_v4_2_ctx2048/best_model.pt...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

| [1 files][721.5 MiB/721.5 MiB]                                                
Operation completed over 1 objects/721.5 MiB.                                    
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Cop

### Ejectar Base Model

In [ ]:
"""
inference.py — CryoCCM Base Model Testing  (SALVAGE EDITION)
=============================================================
Cambios respecto al original:

TOKENIZACIÓN:
  - FIX CRÍTICO: encode() ahora fuerza scale=2 (subword BPE) para prompts de código.
    El AST es útil para entrenamiento pero en inferencia produce tokens que el modelo
    no puede "continuar" como texto legible. Subword es el modo que produce outputs
    coherentes en todos los dominios.
  - FIX: encode() con adaptive=False + scale explícita evita el switch dinámico
    que partía segmentos a mitad de prompt y confundía al modelo.

GENERACIÓN:
  - FIX: repetition_penalty con ventana deslizante de 128 (era 64).
    Con contextos largos, 64 tokens de ventana causa que el modelo repita
    frases completas pasado ese punto.
  - MEJORA: parámetros óptimos por dominio (code / prose / dialogue).
    Código funciona mejor con temp más baja (0.3) porque el espacio de
    tokens válidos es más restringido.
  - MEJORA: min_p sampling añadido como alternativa a top_p. Filtra tokens
    cuya probabilidad es menor a min_p * max_prob. Más estable que nucleus
    en modelos pequeños con distribuciones planas.
  - MEJORA: early stop si el modelo genera el mismo token 5 veces seguidas
    (colapso de distribución — inútil seguir generando).
  - MEJORA: guardia ante NaN/all-inf en logits post-filtrado.

DIAGNÓSTICO:
  - Flag --debug: muestra tokens raw generados, dominio detectado, tokens/seg.
  - Flag --interactive: modo interactivo con comandos /code /prose /debug /temp.

ESTRUCTURA DE ARCHIVOS ESPERADA:
    /content/
        best_model.pt
        neotoken_22k.pkl
        NeoTransformer.py
        NeoTokenizer.py     ← versión SALVAGE con _ast_pattern_to_code
"""

import torch
import torch.nn.functional as F
import sys
import os
import time
from typing import Optional

# ── Imports del proyecto ───────────────────────────────────────────────────────
sys.path.insert(0, '/content')

# ── Configuración ──────────────────────────────────────────────────────────────
MODEL_PATH     = "/content/best_model.pt"
TOKENIZER_PATH = "/content/neotoken_22k.pkl"
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🖥️  Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# ── Cargar tokenizador ─────────────────────────────────────────────────────────
print("\n📦 Cargando tokenizador...")
tokenizer = NeoToken()
tokenizer.load(TOKENIZER_PATH)
print(f"   Vocab: {tokenizer.vocab_size} tokens")


# ── Cargar modelo ──────────────────────────────────────────────────────────────
print("\n🧠 Cargando modelo...")
config = CryoConfig()
model  = CryoCCM(config)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

if "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

# Limpiar prefijos DDP / torch.compile
cleaned = {}
for k, v in state_dict.items():
    key = k
    for prefix in ("module.", "_orig_mod."):
        if key.startswith(prefix):
            key = key[len(prefix):]
    cleaned[key] = v

missing, unexpected = model.load_state_dict(cleaned, strict=False)
if missing:
    print(f"   ⚠️  Keys faltantes ({len(missing)}): {missing[:5]}")
if unexpected:
    print(f"   ⚠️  Keys inesperadas ({len(unexpected)}): {unexpected[:5]}")

model = model.to(DEVICE)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"   Parámetros: {total_params/1e6:.1f}M")
if DEVICE == "cuda":
    print(f"   VRAM usada: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("\n✅ Modelo listo.\n")


# ── Detección de dominio ───────────────────────────────────────────────────────

def _detect_domain(prompt: str) -> str:
    """
    Detecta el dominio del prompt para aplicar parámetros óptimos.
    Retorna: 'code', 'math', 'dialogue', 'prose'
    """
    import re
    code_signals = [
        bool(re.search(r'^\s*(def |class |async def |import |from .+ import)', prompt, re.MULTILINE)),
        bool(re.search(r'^\s*(for .+ in |if .+:|while .+:|try:|except)', prompt, re.MULTILINE)),
        bool(re.search(r'(:\s*\n\s+)', prompt)),
        bool(re.search(r'(->|:=|!=|==|<=|>=)', prompt)),
    ]
    if sum(code_signals) >= 2 or code_signals[0]:
        return 'code'

    if bool(re.search(r'(\$.*\$|\\frac|\\sum|\\int)', prompt)):
        return 'math'

    if bool(re.search(r'^(Usuario|Asistente|User|Assistant|Human|AI):', prompt, re.MULTILINE)):
        return 'dialogue'

    return 'prose'


def _encode_for_inference(prompt: str, domain: str) -> list:
    """
    FIX CRÍTICO: Tokeniza el prompt forzando subword (scale=2) para inferencia.

    Problema original: adaptive=True detecta código en el prompt y lo codifica
    como tokens AST ([A:NODE:X]). El modelo entonces intenta continuar esa
    secuencia generando más AST — que al decodificarse produce la sopa de
    'obj.attr_0a+bvar_1' que veíamos.

    Solución: siempre subword BPE para inferencia. El modelo vio subword en
    todos los dominios durante training, puede continuar desde ahí en cualquier caso.
    El único caso donde usamos otra escala es LaTeX (scale=5), que raramente
    aparece en los prompts de prueba.
    """
    if domain == 'math':
        return tokenizer.encode(prompt, preferred_scale=5)
    else:
        # preferred_scale bypasea el enrutamiento dinámico completamente
        return tokenizer.encode(prompt, preferred_scale=2)


# ── Función de generación principal ───────────────────────────────────────────

def generate(
    prompt:             str,
    max_new_tokens:     int   = 120,
    temperature:        float = 0.8,
    top_k:              int   = 50,
    top_p:              float = 0.92,
    min_p:              float = 0.05,
    repetition_penalty: float = 1.15,
    repetition_window:  int   = 128,
    domain:             Optional[str] = None,
    debug:              bool  = False,
) -> str:
    """
    Genera texto a partir de un prompt.

    Args:
        prompt:             Texto inicial.
        max_new_tokens:     Tokens a generar.
        temperature:        >1 más aleatorio, <1 más determinista.
        top_k:              Top-k sampling.
        top_p:              Nucleus sampling.
        min_p:              Min-p sampling: filtra tokens < min_p * p_max.
                            Más estable que top_p para modelos pequeños.
                            Rango útil: 0.02–0.1. Poner 0.0 para desactivar.
        repetition_penalty: >1 penaliza repetición.
        repetition_window:  Ventana de tokens pasados para rep. penalty (FIX: era 64).
        domain:             'code'|'math'|'dialogue'|'prose'|None (autodetectar).
        debug:              Muestra tokens raw generados para diagnóstico.

    Returns:
        str: texto generado (sin el prompt original).
    """
    if domain is None:
        domain = _detect_domain(prompt)

    # FIX CRÍTICO: tokenizar sin AST
    input_ids = _encode_for_inference(prompt, domain)
    if not input_ids:
        return "[ERROR: prompt vacío después de tokenizar]"

    if debug:
        raw = [tokenizer.id_to_token.get(i, '[UNK]') for i in input_ids]
        print(f"   [DEBUG] Dominio: {domain}")
        print(f"   [DEBUG] Prompt tokens ({len(input_ids)}): {raw[:20]}{'...' if len(raw)>20 else ''}")

    input_tensor = torch.tensor([input_ids], dtype=torch.long, device=DEVICE)
    generated    = list(input_ids)

    last_token    = None
    repeat_streak = 0
    t0 = time.time()

    with torch.no_grad():
        for step in range(max_new_tokens):
            context = input_tensor[:, -config.block_size:]
            logits, _ = model(context)
            next_logits = logits[:, -1, :].float()

            # Repetition penalty con ventana más larga
            if repetition_penalty != 1.0:
                window = generated[-repetition_window:]
                for token_id in set(window):
                    if next_logits[0, token_id] > 0:
                        next_logits[0, token_id] /= repetition_penalty
                    else:
                        next_logits[0, token_id] *= repetition_penalty

            # Temperature
            next_logits = next_logits / max(temperature, 1e-8)

            # Top-k
            if top_k > 0:
                values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits[next_logits < values[:, -1:]] = float('-inf')

            # Top-p (nucleus)
            if top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
                cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                remove    = cum_probs - F.softmax(sorted_logits, dim=-1) > top_p
                sorted_logits[remove] = float('-inf')
                next_logits.scatter_(1, sorted_idx, sorted_logits)

            # Min-p sampling
            if min_p > 0.0:
                probs_tmp = F.softmax(next_logits, dim=-1)
                max_prob  = probs_tmp.max(dim=-1, keepdim=True).values
                next_logits[probs_tmp < min_p * max_prob] = float('-inf')

            # Guardia NaN / all-inf (filtros demasiado agresivos)
            probs = F.softmax(next_logits, dim=-1)
            if probs.isnan().any() or probs.sum() < 1e-8:
                probs = F.softmax(logits[:, -1, :].float() / max(temperature, 1e-8), dim=-1)

            next_id = torch.multinomial(probs, num_samples=1).item()

            # Stop: tokens especiales
            if next_id in (tokenizer.special_tokens.get('[PAD]', 0),
                           tokenizer.special_tokens.get('[SEP]', 3)):
                break

            # Stop: colapso de distribución (mismo token 5 veces seguidas)
            if next_id == last_token:
                repeat_streak += 1
                if repeat_streak >= 5:
                    if debug:
                        tok_name = tokenizer.id_to_token.get(next_id, '?')
                        print(f"   [DEBUG] Early stop por colapso en '{tok_name}' (id={next_id})")
                    break
            else:
                repeat_streak = 0
            last_token = next_id

            generated.append(next_id)
            input_tensor = torch.cat([
                input_tensor,
                torch.tensor([[next_id]], device=DEVICE)
            ], dim=1)

    new_tokens = generated[len(input_ids):]

    if debug:
        raw_new = [tokenizer.id_to_token.get(i, '[UNK]') for i in new_tokens]
        ast_tok = [t for t in raw_new if t.startswith('[A:')]
        elapsed = time.time() - t0
        print(f"   [DEBUG] Generados: {len(new_tokens)} tokens | "
              f"{len(ast_tok)} AST | {len(new_tokens)/elapsed:.1f} tok/s")
        if ast_tok:
            print(f"   [DEBUG] AST leak: {ast_tok[:10]}")

    return tokenizer.decode(new_tokens)


# ── Wrappers con parámetros óptimos por dominio ────────────────────────────────

def generate_code(prompt: str, max_new_tokens: int = 80, debug: bool = False) -> str:
    """
    Código: temperatura baja, min_p más alto, rep_penalty más agresivo.
    El espacio de tokens de código válido es más restringido → menos temperatura.
    """
    return generate(
        prompt, max_new_tokens=max_new_tokens,
        temperature=0.3, top_k=30, top_p=0.90, min_p=0.08,
        repetition_penalty=1.2, repetition_window=128,
        domain='code', debug=debug
    )

def generate_prose(prompt: str, max_new_tokens: int = 120, debug: bool = False) -> str:
    """Prosa: balance entre creatividad y coherencia."""
    return generate(
        prompt, max_new_tokens=max_new_tokens,
        temperature=0.8, top_k=50, top_p=0.92, min_p=0.04,
        repetition_penalty=1.15, repetition_window=128,
        domain='prose', debug=debug
    )

def generate_dialogue(prompt: str, max_new_tokens: int = 100, debug: bool = False) -> str:
    """Diálogo: temperatura media, más coherente que prosa libre."""
    return generate(
        prompt, max_new_tokens=max_new_tokens,
        temperature=0.7, top_k=40, top_p=0.90, min_p=0.05,
        repetition_penalty=1.2, repetition_window=128,
        domain='dialogue', debug=debug
    )


# ── Demo ───────────────────────────────────────────────────────────────────────

def run_demo(debug: bool = False):
    tests = [
        (
            "Completado de prosa",
            "La inteligencia artificial ha transformado la manera en que",
            generate_prose, 80
        ),
        (
            "Completado de código Python",
            "def fibonacci(n):\n    \"\"\"Retorna el n-ésimo número de Fibonacci.\"\"\"\n    if n <= 1:\n        return n\n    ",
            generate_code, 60
        ),
        (
            "Completado de diálogo",
            "Usuario: ¿Cómo funciona una red neuronal?\nAsistente:",
            generate_dialogue, 100
        ),
        (
            "Completado técnico",
            "Los transformers utilizan mecanismos de atención que permiten al modelo",
            generate_prose, 80
        ),
        (
            "Completado de lista",
            "Los principales pasos para entrenar un modelo de lenguaje son:\n1.",
            generate_prose, 70
        ),
    ]

    print("=" * 65)
    print("🧪 PRUEBAS BASE MODEL — CryoCCM V4.2  [SALVAGE EDITION]")
    print("   (completado de texto, no assistant)")
    print("=" * 65)

    for desc, prompt, gen_fn, max_tok in tests:
        print(f"\n── {desc} ──")
        print(f"PROMPT: {prompt[:80]}{'...' if len(prompt)>80 else ''}")
        print("OUTPUT:", end=" ")
        try:
            result = gen_fn(prompt, max_new_tokens=max_tok, debug=debug)
            print(result)
        except Exception as e:
            print(f"[ERROR: {e}]")
            import traceback; traceback.print_exc()

    print("\n" + "=" * 65)
    print("💡 Wrappers disponibles:")
    print('   generate_prose("texto...")          → prosa española')
    print('   generate_code("def mi_func():")     → código Python')
    print('   generate_dialogue("Usuario: ...")   → diálogo')
    print('   generate("...", debug=True)         → ver tokens raw')
    print("=" * 65)


# ── Modo interactivo ───────────────────────────────────────────────────────────

def interactive():
    print("\n🎮 MODO INTERACTIVO — escribe 'salir' para terminar")
    print("   Comandos: /temp 0.8 | /tokens 100 | /code | /prose | /dialogue | /auto | /debug")
    print("-" * 55)

    temp    = 0.8
    max_tok = 120
    debug   = False
    mode    = 'auto'

    while True:
        try:
            user_input = input("\nPrompt > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSaliendo...")
            break

        if not user_input or user_input.lower() == 'salir':
            break

        if user_input.startswith('/temp '):
            try:
                temp = float(user_input.split()[1])
                print(f"  → Temperature: {temp}")
            except ValueError:
                print("  → Uso: /temp 0.8")
            continue

        if user_input.startswith('/tokens '):
            try:
                max_tok = int(user_input.split()[1])
                print(f"  → Max tokens: {max_tok}")
            except ValueError:
                print("  → Uso: /tokens 100")
            continue

        if user_input == '/debug':
            debug = not debug
            print(f"  → Debug: {'ON 🔍' if debug else 'OFF'}")
            continue

        if user_input in ('/code', '/prose', '/dialogue', '/auto'):
            mode = user_input[1:]
            print(f"  → Modo: {mode}")
            continue

        domain = None if mode == 'auto' else mode
        print("\nGenerando...")
        try:
            result = generate(
                user_input, max_new_tokens=max_tok,
                temperature=temp, domain=domain, debug=debug
            )
            print(f"\n{user_input}{result}")
        except Exception as e:
            print(f"[ERROR: {e}]")
            import traceback; traceback.print_exc()


# ── Entry point ────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    debug_mode = "--debug" in sys.argv or "-d" in sys.argv

    if "--interactive" in sys.argv or "-i" in sys.argv:
        interactive()
    else:
        run_demo(debug=debug_mode)
        print("\n💡 Para modo interactivo: python inference.py --interactive")
        print("💡 Para debug de tokens:  python inference.py --debug")

🖥️  Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB

📦 Cargando tokenizador...
✅ NeoToken cargado: 22619 tokens en vocabulario.
   Vocab: 22619 tokens

🧠 Cargando modelo...
   Parámetros: 189.1M
   VRAM usada: 1.53 GB

✅ Modelo listo.

🧪 PRUEBAS BASE MODEL — CryoCCM V4.2  [SALVAGE EDITION]
   (completado de texto, no assistant)

── Completado de prosa ──
PROMPT: La inteligencia artificial ha transformado la manera en que
OUTPUT: ...
Según un estudio publicado en el National Institute for Deep Learning, las personas que toman el control de su computación son más propensas a desarrollar lenguajes de programación, informática y tecnologías de cómputo que pueden ser utilizados para mejorar la comunicación y la comprens

── Completado de código Python ──
PROMPT: def fibonacci(n):
    """Retorna el n-ésimo número de Fibonacci."""
    if n <= ...
OUTPUT: elif n == 0:
     

── Completado de diálogo ──
PROMPT: Usuario: ¿Cómo funciona una red neuronal?
Asistente:
OUTPUT: Una red neuronal es una 

### SFT

In [ ]:
"""
SFT_Training.py — Fine-Tuning de CryoCCM V4.2 para modo Asistente
==================================================================
LA RECETA MÉDICA COMPLETA

¿Qué hace esto?
  Toma el base model (que sabe hablar pero no sabe comportarse como asistente)
  y le enseña el formato pregunta → respuesta con ejemplos reales en español.
  Es la diferencia entre un estudiante que sabe física y un profesor que la explica.

¿Qué necesitás?
  - Google Colab Free (T4, 15GB VRAM) — ALCANZA
  - best_model.pt           ← tu modelo entrenado
  - neotoken_22k.pkl        ← tu tokenizador
  - NeoTransformer.py       ← arquitectura
  - NeoTokenizer.py         ← tokenizador (versión SALVAGE)
  - Este archivo

DOSIS:
  - Dataset: ~3000-5000 ejemplos de conversación en español (se descarga solo)
  - Épocas: 3 (suficiente para aprender el formato sin sobreajustar)
  - Tiempo estimado en T4 Free: 1.5 - 2.5 horas
  - LR: 3e-5 (10x menor que pretraining — cirugía fina, no a mazazos)
  - Batch effective: 8 (pequeño por el tamaño del contexto SFT)

INSTRUCCIONES COLAB:
  1. Subí best_model.pt y neotoken_22k.pkl a /content/
  2. Copiá NeoTransformer.py, NeoTokenizer.py y este archivo a /content/
  3. Ejecutá: !python SFT_Training.py
  4. Al terminar tenés sft_model.pt listo para usar con Inference.py
     (cambiá MODEL_PATH = "/content/sft_model.pt" en Inference.py)

FORMATO DE ENTRENAMIENTO:
  Cada ejemplo se convierte en:
    <|user|> pregunta del usuario
    <|assistant|> respuesta del asistente <|end|>

  El modelo aprende a predecir SOLO los tokens de la respuesta
  (la pregunta se enmascara con -100, igual que en Alpaca/Llama).
"""

import os
import sys
import math
import time
import json
import datetime
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext

sys.path.insert(0, '/content')

# ==============================================================================
# CONFIGURACIÓN — DOSIS CALCULADA PARA T4 FREE (15GB VRAM)
# ==============================================================================

MODEL_PATH      = "/content/best_model.pt"
TOKENIZER_PATH  = "/content/neotoken_22k.pkl"
OUTPUT_PATH     = "/content/sft_model.pt"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

# Hiperparámetros SFT
# Regla de oro: LR de SFT = LR de pretraining / 10 como mínimo.
# LR muy alto destruye lo aprendido. LR muy bajo no enseña nada.
LEARNING_RATE   = 3e-5       # cirugía fina
MIN_LR          = 3e-6       # 10% del LR
WARMUP_STEPS    = 100        # warmup corto — ya tiene base, no necesita mucho
EPOCHS          = 3          # 3 pasadas completas al dataset
BATCH_SIZE      = 1          # 1 ejemplo a la vez — contextos SFT son más largos
GRAD_ACCUM      = 8          # effective batch = 8
MAX_SEQ_LEN     = 512        # longitud máxima por ejemplo (suficiente para Q&A)
WEIGHT_DECAY    = 0.01
GRAD_CLIP       = 1.0
EVAL_EVERY      = 100        # validar cada 100 steps
DTYPE           = 'bfloat16'

# ACT fijo en 2 para SFT — más rápido, no necesitamos máximo razonamiento
# para aprender el formato de conversación
SFT_MAX_ACT_STEPS = 2

# ==============================================================================
# DATASET — DESCARGA Y PREPARA AUTOMÁTICAMENTE
# ==============================================================================

def _extract_example(item):
    """
    Extrae (user, assistant) de un item sin importar el esquema de keys.
    Soporta: Alpaca, Chat, prompt/completion, text###, question/answer.
    """
    user_msg = None
    asst_msg = None

    # Esquema 1: instruction / output (Alpaca)
    if 'instruction' in item and 'output' in item:
        instr = (item.get('instruction') or '').strip()
        inp   = (item.get('input') or '').strip()
        out   = (item.get('output') or '').strip()
        if instr and out:
            user_msg = f"{instr}\n{inp}".strip() if inp else instr
            asst_msg = out

    # Esquema 2: prompt / completion
    elif 'prompt' in item and 'completion' in item:
        user_msg = (item.get('prompt') or '').strip()
        asst_msg = (item.get('completion') or '').strip()

    # Esquema 3: messages lista (formato chat)
    elif 'messages' in item:
        msgs = item.get('messages') or []
        for m in msgs:
            role    = (m.get('role') or m.get('from') or '').lower()
            content = (m.get('content') or m.get('value') or '').strip()
            if role in ('user', 'human') and user_msg is None:
                user_msg = content
            elif role in ('assistant', 'gpt', 'bot') and asst_msg is None and user_msg:
                asst_msg = content

    # Esquema 4: question / answer
    elif 'question' in item and 'answer' in item:
        user_msg = (item.get('question') or '').strip()
        asst_msg = (item.get('answer') or '').strip()

    # Esquema 5: text plano con ### markers
    elif 'text' in item:
        text = (item.get('text') or '')
        if '### Response:' in text:
            parts    = text.split('### Response:', 1)
            asst_msg = parts[1].strip()
            pre      = parts[0]
            if '### Instruction:' in pre:
                user_msg = pre.split('### Instruction:', 1)[1].strip()
                if '### Input:' in user_msg:
                    p2       = user_msg.split('### Input:', 1)
                    extra    = p2[1].strip()
                    user_msg = f"{p2[0].strip()}\n{extra}".strip() if extra else p2[0].strip()
            else:
                user_msg = pre.strip()

    return user_msg, asst_msg


def download_sft_dataset():
    """
    Descarga dataset SFT probando múltiples fuentes con detección automática de keys.
    Fallback a dataset sintético si no hay internet o todos fallan.
    """
    try:
        from datasets import load_dataset
        print("📥 Descargando dataset SFT en español...")

        candidates = [
            ("somosnlp/somos-alpaca-es", "train", None),
            ("yahma/alpaca-cleaned",     "train", None),
            ("tatsu-lab/alpaca",         "train", None),
            ("HuggingFaceH4/no_robots",  "train", None),
        ]

        for ds_name, split, cfg in candidates:
            try:
                print(f"   Probando {ds_name}...")
                ds = load_dataset(ds_name, cfg, split=split) if cfg else \
                     load_dataset(ds_name, split=split)

                first = ds[0]
                print(f"   Keys reales: {list(first.keys())}")

                examples = []
                for item in ds:
                    user_msg, asst_msg = _extract_example(item)
                    if not user_msg or not asst_msg:
                        continue
                    if len(asst_msg) < 15 or len(asst_msg) > 1000:
                        continue
                    if len(user_msg) < 5:
                        continue
                    examples.append({'user': user_msg, 'assistant': asst_msg})
                    if len(examples) >= 4500:
                        break

                if len(examples) >= 100:
                    print(f"   ✅ {len(examples)} ejemplos válidos de {ds_name}")
                    return examples
                else:
                    print(f"   ⚠️ Solo {len(examples)} útiles en {ds_name}, siguiente...")

            except Exception as e:
                print(f"   ⚠️ {ds_name}: {type(e).__name__}: {e}")
                continue

        raise Exception("Todos los datasets remotos fallaron")

    except Exception as e:
        print(f"   ⚠️ ({e}) — usando dataset sintético...")
        return _generate_synthetic_dataset()


def _generate_synthetic_dataset():
    """
    Dataset sintético de emergencia — 500 ejemplos básicos.
    Suficiente para que el modelo aprenda el formato asistente
    aunque no mejore mucho el conocimiento.
    Úsalo solo si no hay conexión a internet.
    """
    examples = [
        # Saludos y meta
        {"user": "Hola, ¿cómo estás?",
         "assistant": "¡Hola! Estoy listo para ayudarte. ¿En qué puedo asistirte hoy?"},
        {"user": "¿Qué puedes hacer?",
         "assistant": "Puedo ayudarte con preguntas, explicar conceptos, escribir textos, analizar información y mantener conversaciones sobre una amplia variedad de temas."},
        {"user": "¿Quién eres?",
         "assistant": "Soy un asistente de inteligencia artificial diseñado para ayudarte con tus preguntas y tareas."},

        # Ciencia y tecnología
        {"user": "¿Qué es la inteligencia artificial?",
         "assistant": "La inteligencia artificial es una rama de la informática que desarrolla sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como reconocer patrones, tomar decisiones y aprender de la experiencia."},
        {"user": "¿Cómo funciona una red neuronal?",
         "assistant": "Una red neuronal es un sistema de capas de nodos interconectados inspirado en el cerebro humano. Recibe datos de entrada, los procesa a través de capas intermedias aplicando pesos y funciones de activación, y produce una salida. Durante el entrenamiento, ajusta los pesos para minimizar el error en sus predicciones."},
        {"user": "¿Qué es el machine learning?",
         "assistant": "El machine learning o aprendizaje automático es un subcampo de la inteligencia artificial donde los sistemas aprenden patrones a partir de datos sin ser programados explícitamente. En lugar de seguir reglas fijas, el modelo ajusta sus parámetros internos basándose en ejemplos."},
        {"user": "¿Qué es Python?",
         "assistant": "Python es un lenguaje de programación de alto nivel, interpretado y de propósito general. Es conocido por su sintaxis clara y legible, lo que lo hace ideal para principiantes y expertos. Se usa ampliamente en ciencia de datos, inteligencia artificial, desarrollo web y automatización."},
        {"user": "¿Cómo se hace un bucle for en Python?",
         "assistant": "En Python, un bucle for se escribe así:\n\nfor elemento in iterable:\n    # código a ejecutar\n\nPor ejemplo, para imprimir números del 1 al 5:\n\nfor i in range(1, 6):\n    print(i)"},
        {"user": "¿Qué es una función en programación?",
         "assistant": "Una función es un bloque de código reutilizable que realiza una tarea específica. Se define una vez y se puede llamar múltiples veces. En Python:\n\ndef nombre_funcion(parametros):\n    # código\n    return resultado"},

        # Historia y cultura
        {"user": "¿Cuándo fue la Segunda Guerra Mundial?",
         "assistant": "La Segunda Guerra Mundial turó de 1939 a 1945. Comenzó el 1 de septiembre de 1939 con la invasión alemana de Polonia y terminó el 2 de septiembre de 1945 con la rendición de Japón."},
        {"user": "¿Quién fue Simón Bolívar?",
         "assistant": "Simón Bolívar fue un líder militar y político venezolano nacido en 1783, conocido como el Libertador de América. Lideró los movimientos de independencia de Venezuela, Colombia, Ecuador, Perú y Bolivia, que lleva su nombre."},
        {"user": "¿Cuál es la capital de Francia?",
         "assistant": "La capital de Francia es París."},
        {"user": "¿Cuántos países hay en América del Sur?",
         "assistant": "América del Sur tiene 12 países: Argentina, Bolivia, Brasil, Chile, Colombia, Ecuador, Guyana, Paraguay, Perú, Surinam, Uruguay y Venezuela."},

        # Matemáticas
        {"user": "¿Cuánto es 15 por 23?",
         "assistant": "15 × 23 = 345.\n\nPara calcularlo: 15 × 20 = 300, más 15 × 3 = 45. Total: 300 + 45 = 345."},
        {"user": "¿Qué es el teorema de Pitágoras?",
         "assistant": "El teorema de Pitágoras establece que en un triángulo rectángulo, el cuadrado de la hipotenusa (el lado más largo, opuesto al ángulo recto) es igual a la suma de los cuadrados de los otros dos lados. Se expresa como: a² + b² = c², donde c es la hipotenusa."},
        {"user": "¿Qué es una derivada?",
         "assistant": "Una derivada mide la tasa de cambio instantánea de una función respecto a su variable. Geométricamente, representa la pendiente de la recta tangente a la curva en un punto. Por ejemplo, si f(x) = x², entonces f'(x) = 2x."},

        # Salud y bienestar
        {"user": "¿Cuántas horas de sueño necesita un adulto?",
         "assistant": "Los adultos generalmente necesitan entre 7 y 9 horas de sueño por noche según la mayoría de organizaciones de salud. Dormir menos puede afectar la concentración, el sistema inmune y el estado de ánimo."},
        {"user": "¿Qué beneficios tiene el ejercicio físico?",
         "assistant": "El ejercicio físico regular tiene numerosos beneficios: mejora la salud cardiovascular, fortalece músculos y huesos, reduce el estrés y la ansiedad, mejora el sueño, ayuda a mantener un peso saludable y puede prevenir enfermedades crónicas como la diabetes tipo 2."},

        # Escritura y redacción
        {"user": "¿Cómo se escribe un correo formal?",
         "assistant": "Un correo formal debe incluir:\n1. Asunto claro y conciso\n2. Saludo formal (Estimado/a Sr./Sra. o A quien corresponda)\n3. Introducción breve\n4. Cuerpo del mensaje con el tema principal\n5. Cierre educado (Atentamente, Cordialmente)\n6. Firma con nombre y datos de contacto"},
        {"user": "¿Qué es un ensayo?",
         "assistant": "Un ensayo es un texto en prosa que desarrolla una idea o argumento desde la perspectiva personal del autor. Tiene estructura libre pero generalmente incluye introducción, desarrollo y conclusión. Se distingue por la reflexión crítica y el estilo propio del escritor."},

        # Filosofía y reflexión
        {"user": "¿Qué es la consciencia?",
         "assistant": "La consciencia es la capacidad de tener experiencias subjetivas y ser consciente de uno mismo y del entorno. Es uno de los mayores misterios de la filosofía y la neurociencia. Implica tanto la percepción del mundo externo como la capacidad de reflexionar sobre los propios pensamientos."},
        {"user": "¿Cuál es el sentido de la vida?",
         "assistant": "Esta es una de las preguntas más profundas de la filosofía. Diferentes tradiciones dan respuestas distintas: para el existencialismo, cada persona crea su propio sentido; para el utilitarismo, es maximizar el bienestar; para muchas religiones, cumplir un propósito divino. No hay una respuesta única, y explorar esa pregunta es parte de lo que nos hace humanos."},

        # Cocina
        {"user": "¿Cómo se hace el arroz?",
         "assistant": "Para hacer arroz básico: lava el arroz hasta que el agua salga clara. En una olla, pon 2 tazas de agua por cada taza de arroz con sal al gusto. Cuando hierva, agrega el arroz, baja el fuego al mínimo, tapa y cocina 18 minutos. Apaga y deja reposar 5 minutos antes de servir."},
        {"user": "¿Qué ingredientes lleva una tortilla española?",
         "assistant": "La tortilla española tradicional lleva: huevos, patatas (papas), aceite de oliva y sal. Algunos añaden cebolla, aunque es un tema de debate entre los puristas. Se cocina la patata en aceite, se mezcla con los huevos batidos y se cuaja en sartén por ambos lados."},
    ]

    # Expandir con variaciones para llegar a ~300 ejemplos
    base = examples.copy()
    expanded = list(base)

    reformulations = [
        ("¿Puedes explicarme qué es la inteligencia artificial?",
         "La inteligencia artificial (IA) es la simulación de procesos de inteligencia humana por sistemas informáticos. Incluye el aprendizaje, el razonamiento y la autocorrección. Sus aplicaciones van desde asistentes virtuales hasta diagnósticos médicos."),
        ("Dame un ejemplo de código Python para calcular el factorial",
         "Aquí tienes una función para calcular el factorial:\n\ndef factorial(n):\n    if n == 0 or n == 1:\n        return 1\n    return n * factorial(n - 1)\n\n# Ejemplo:\nprint(factorial(5))  # Output: 120"),
        ("¿Qué es el cambio climático?",
         "El cambio climático es la variación a largo plazo de los patrones de temperatura y precipitación en la Tierra. Actualmente se refiere principalmente al calentamiento global causado por la emisión de gases de efecto invernadero como el CO2, resultado de actividades humanas como quemar combustibles fósiles."),
        ("Explícame la fotosíntesis",
         "La fotosíntesis es el proceso mediante el cual las plantas convierten luz solar, agua y dióxido de carbono en glucosa y oxígeno. Ocurre principalmente en los cloroplastos usando clorofila. La ecuación simplificada es: 6CO2 + 6H2O + luz → C6H12O6 + 6O2."),
        ("¿Cómo puedo mejorar mi memoria?",
         "Para mejorar la memoria: duerme bien (el sueño consolida recuerdos), haz ejercicio regular, practica la técnica de repetición espaciada para estudiar, usa mnemotecnia, reduce el estrés, mantén una dieta equilibrada y ejercita la mente con lecturas y rompecabezas."),
        ("¿Qué es la gravedad?",
         "La gravedad es una fuerza fundamental de la naturaleza que atrae los cuerpos con masa entre sí. Newton la describió como proporcional al producto de las masas e inversamente proporcional al cuadrado de la distancia. Einstein la redefinió como la curvatura del espacio-tiempo causada por la masa."),
        ("¿Cómo funciona internet?",
         "Internet es una red global de computadoras interconectadas que se comunican mediante protocolos estándar (TCP/IP). Cuando accedés a una web, tu dispositivo envía una solicitud a un servidor que responde con los datos. Los datos viajan en paquetes por cables físicos, fibra óptica y conexiones inalámbricas."),
        ("Dame consejos para aprender un idioma nuevo",
         "Para aprender un idioma: practica todos los días aunque sea 15 minutos, usa la inmersión (películas, música, podcasts en ese idioma), aprende vocabulario con contexto, habla con nativos aunque cometas errores, usa aplicaciones como Duolingo para vocabulario básico y no tengas miedo de equivocarte."),
    ]

    for user, assistant in reformulations:
        expanded.append({"user": user, "assistant": assistant})

    print(f"   ✅ Dataset sintético generado: {len(expanded)} ejemplos")
    return expanded


# ==============================================================================
# TOKENIZACIÓN SFT
# ==============================================================================

# Tokens especiales de formato conversacional
# Usamos formato simple que el tokenizador puede manejar como subwords
USER_PREFIX      = "Usuario: "
ASSISTANT_PREFIX = "\nAsistente: "
END_TOKEN        = "\n"

class SFTDataset(Dataset):
    """
    Convierte ejemplos conversacionales en tensores para SFT.

    Formato de cada ejemplo:
        "Usuario: {pregunta}\nAsistente: {respuesta}\n"

    La loss se calcula SOLO sobre los tokens de la respuesta.
    La pregunta se enmascara con -100 (ignorada en cross-entropy).
    Esto es crucial — si entrenamos sobre la pregunta, el modelo
    aprende a predecir preguntas en vez de respuestas.
    """
    def __init__(self, examples, tokenizer, max_seq_len=512):
        self.tokenizer   = tokenizer
        self.max_seq_len = max_seq_len
        self.data        = []

        skipped = 0
        for ex in examples:
            user_text = USER_PREFIX + ex['user'].strip()
            asst_text = ASSISTANT_PREFIX + ex['assistant'].strip() + END_TOKEN

            # Tokenizar por separado para saber dónde termina la pregunta
            user_ids = tokenizer.encode(user_text,      preferred_scale=2)
            asst_ids = tokenizer.encode(asst_text,      preferred_scale=2)

            full_ids = user_ids + asst_ids

            # Truncar si es muy largo
            if len(full_ids) > max_seq_len:
                # Priorizar la respuesta: truncar la pregunta si hace falta
                max_user = max_seq_len - len(asst_ids) - 10
                if max_user < 10:
                    skipped += 1
                    continue
                user_ids = user_ids[:max_user]
                full_ids = user_ids + asst_ids

            if len(full_ids) < 10:
                skipped += 1
                continue

            # Input: todos los tokens menos el último
            # Labels: -100 en la pregunta, tokens reales en la respuesta
            input_ids = full_ids[:-1]
            labels    = [-100] * len(user_ids) + asst_ids[1:]

            # Alinear longitudes
            min_len   = min(len(input_ids), len(labels))
            input_ids = input_ids[:min_len]
            labels    = labels[:min_len]

            if min_len < 5:
                skipped += 1
                continue

            self.data.append((
                torch.tensor(input_ids, dtype=torch.long),
                torch.tensor(labels,    dtype=torch.long)
            ))

        print(f"   📚 Dataset SFT: {len(self.data)} ejemplos válidos "
              f"({skipped} descartados por longitud)")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    """Padding dinámico — cada batch tiene su longitud máxima."""
    inputs, labels = zip(*batch)
    max_len = max(x.size(0) for x in inputs)

    padded_inputs = torch.zeros(len(inputs), max_len, dtype=torch.long)
    padded_labels = torch.full((len(inputs), max_len), -100, dtype=torch.long)

    for i, (inp, lbl) in enumerate(zip(inputs, labels)):
        padded_inputs[i, :inp.size(0)] = inp
        padded_labels[i, :lbl.size(0)] = lbl

    return padded_inputs, padded_labels


# ==============================================================================
# LEARNING RATE SCHEDULE
# ==============================================================================

def get_sft_lr(step, total_steps):
    """Cosine decay con warmup corto."""
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / WARMUP_STEPS
    if step >= total_steps:
        return MIN_LR
    decay = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    cosine = 0.5 * (1.0 + math.cos(math.pi * decay))
    return MIN_LR + cosine * (LEARNING_RATE - MIN_LR)


# ==============================================================================
# FINE-TUNING LOOP
# ==============================================================================

def finetune():
    print("=" * 65)
    print("💊 SFT — FINE-TUNING CryoCCM V4.2")
    print("   Convirtiendo base model → asistente conversacional")
    print("=" * 65)

    torch.manual_seed(42)
    np.random.seed(42)

    ctx = (torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)
           if DEVICE == "cuda" else nullcontext())

    # ── 1. Cargar tokenizador ──────────────────────────────────────────────────
    print("\n📦 Cargando tokenizador...")
    tokenizer = NeoToken()
    tokenizer.load(TOKENIZER_PATH)
    print(f"   Vocab: {tokenizer.vocab_size} tokens")

    # ── 2. Cargar modelo base ──────────────────────────────────────────────────
    print("\n🧠 Cargando base model...")
    config = CryoConfig()

    # SFT con ACT reducido — más rápido y suficiente para aprender formato
    config.max_steps_act             = SFT_MAX_ACT_STEPS
    config.ponder_penalty_beta       = 0.005  # suave — no queremos que ACT domine
    config.self_critic_weight        = 0.01
    config.uncertainty_threshold     = 0.35
    config.use_gradient_checkpointing = True   # necesario para no quedarnos sin VRAM

    model = CryoCCM(config)

    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    cleaned = {}
    for k, v in state_dict.items():
        key = k
        for prefix in ("module.", "_orig_mod."):
            if key.startswith(prefix):
                key = key[len(prefix):]
        cleaned[key] = v

    missing, unexpected = model.load_state_dict(cleaned, strict=False)
    if missing:
        print(f"   ⚠️  Keys faltantes: {len(missing)}")

    model = model.to(DEVICE)
    model.train()

    total_params = sum(p.numel() for p in model.parameters())
    print(f"   Parámetros: {total_params/1e6:.1f}M")
    if DEVICE == "cuda":
        print(f"   VRAM base: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    # ── 3. Dataset ────────────────────────────────────────────────────────────
    print("\n📥 Preparando dataset SFT...")
    all_examples = download_sft_dataset()

    # Split 90/10 train/val
    np.random.shuffle(all_examples)
    split        = int(len(all_examples) * 0.9)
    train_exs    = all_examples[:split]
    val_exs      = all_examples[split:]

    print(f"   Split: {len(train_exs)} train | {len(val_exs)} val")

    train_ds = SFTDataset(train_exs, tokenizer, MAX_SEQ_LEN)
    val_ds   = SFTDataset(val_exs,   tokenizer, MAX_SEQ_LEN)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_fn, num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=0
    )

    # ── 4. Optimizador ────────────────────────────────────────────────────────
    # Para SFT no necesitamos fused optimizer ni configuración especial —
    # el modelo ya tiene estructura estable, solo ajustamos levemente.
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        betas=(0.9, 0.95),
        weight_decay=WEIGHT_DECAY,
        eps=1e-8
    )

    scaler = torch.amp.GradScaler('cuda', enabled=False)  # bfloat16 no necesita scaler

    steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM))
    total_steps     = steps_per_epoch * EPOCHS

    print(f"\n📐 Plan de entrenamiento:")
    print(f"   Epochs:          {EPOCHS}")
    print(f"   Steps/epoch:     {steps_per_epoch}")
    print(f"   Total steps:     {total_steps}")
    print(f"   Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
    print(f"   LR:              {LEARNING_RATE} → {MIN_LR}")
    print(f"   ACT steps:       {SFT_MAX_ACT_STEPS} (reducido para velocidad)")

    # Estimación de tiempo (basado en ~0.8 seg/step en T4 con ACT=2)
    est_min = total_steps * 0.8 / 60
    print(f"   ⏱️  Tiempo estimado T4 Free: {est_min:.0f}-{est_min*1.5:.0f} minutos")

    # ── 5. Training Loop ──────────────────────────────────────────────────────
    print("\n" + "=" * 65)
    print("🔥 INICIANDO SFT")
    print("=" * 65 + "\n")

    global_step   = 0
    best_val_loss = float('inf')
    run_start     = time.time()

    for epoch in range(EPOCHS):
        print(f"\n{'='*50}")
        print(f"📚 ÉPOCA {epoch+1}/{EPOCHS}")
        print(f"{'='*50}")

        model.train()
        train_iter    = iter(train_loader)
        epoch_loss    = 0.0
        epoch_steps   = 0
        accum_loss    = 0.0
        optimizer.zero_grad(set_to_none=True)

        batch_count = 0
        while True:
            # Acumular gradientes
            micro_loss = 0.0
            valid_micros = 0

            for _ in range(GRAD_ACCUM):
                try:
                    X, Y = next(train_iter)
                except StopIteration:
                    train_iter = None
                    break

                X, Y = X.to(DEVICE), Y.to(DEVICE)

                with ctx:
                    _, metrics = model(X, labels=Y)
                    loss = metrics['loss_combined'] / GRAD_ACCUM

                loss_val = loss.item()
                if not math.isfinite(loss_val):
                    print(f"\n💀 NaN en step {global_step}! Saltando batch.")
                    optimizer.zero_grad(set_to_none=True)
                    micro_loss = 0.0
                    valid_micros = 0
                    break

                loss.backward()
                micro_loss  += loss_val
                valid_micros += 1

            if train_iter is None:
                break  # época terminada

            if valid_micros == 0:
                continue

            # LR schedule
            lr = get_sft_lr(global_step, total_steps)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            accum_loss  += micro_loss
            epoch_loss  += micro_loss
            epoch_steps += 1
            global_step += 1
            batch_count += 1

            # Log cada 10 steps
            if global_step % 10 == 0:
                elapsed   = time.time() - run_start
                steps_run = max(1, global_step)
                eta_secs  = (total_steps - global_step) * (elapsed / steps_run)
                eta_str   = str(datetime.timedelta(seconds=int(eta_secs)))
                vram      = torch.cuda.max_memory_allocated()/1e9 if DEVICE=="cuda" else 0
                torch.cuda.reset_peak_memory_stats() if DEVICE=="cuda" else None

                print(f"  Step {global_step:4d}/{total_steps} | "
                      f"Loss {accum_loss/10:.3f} | "
                      f"LR {lr:.2e} | "
                      f"VRAM {vram:.1f}G | "
                      f"ETA {eta_str}")
                accum_loss = 0.0

            # Validación periódica
            if global_step % EVAL_EVERY == 0:
                val_loss = _estimate_val_loss(model, ctx, val_loader)
                avg_train = epoch_loss / max(1, epoch_steps)

                print(f"\n  🔍 Validación step {global_step}:")
                print(f"     Train loss: {avg_train:.4f} | Val loss: {val_loss:.4f}")

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(model.state_dict(), OUTPUT_PATH)
                    print(f"     ⭐ Nuevo mejor modelo guardado → {OUTPUT_PATH}")
                else:
                    print(f"     (mejor val loss hasta ahora: {best_val_loss:.4f})")
                print()

                model.train()

        avg_epoch_loss = epoch_loss / max(1, epoch_steps)
        print(f"\n  ✅ Época {epoch+1} completa | Loss promedio: {avg_epoch_loss:.4f}")

    # ── 6. Guardado final ─────────────────────────────────────────────────────
    final_path = OUTPUT_PATH.replace('.pt', '_final.pt')
    torch.save(model.state_dict(), final_path)

    total_time = time.time() - run_start
    print(f"\n{'='*65}")
    print(f"🎉 SFT COMPLETADO")
    print(f"   Tiempo total: {str(datetime.timedelta(seconds=int(total_time)))}")
    print(f"   Mejor val loss: {best_val_loss:.4f}")
    print(f"   Mejor modelo:  {OUTPUT_PATH}")
    print(f"   Modelo final:  {final_path}")
    print(f"\n💊 DOSIS APLICADA:")
    print(f"   {EPOCHS} épocas × {steps_per_epoch} steps × effective batch {BATCH_SIZE*GRAD_ACCUM}")
    print(f"   LR: {LEARNING_RATE} → {MIN_LR} (cosine decay)")
    print(f"\n🔧 PARA USAR EN INFERENCE:")
    print(f"   Cambiá MODEL_PATH = '{OUTPUT_PATH}' en Inference.py")
    print(f"{'='*65}")


@torch.no_grad()
def _estimate_val_loss(model, ctx, val_loader, max_batches=30):
    """Estima val loss en un subset para no tardar demasiado."""
    model.eval()
    losses = []
    for i, (X, Y) in enumerate(val_loader):
        if i >= max_batches:
            break
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        with ctx:
            _, metrics = model(X, labels=Y)
            loss_val = metrics.get('lm_loss', metrics['loss_combined'])
            if math.isfinite(loss_val.item()):
                losses.append(loss_val.item())
    model.train()
    return np.mean(losses) if losses else float('inf')


# ==============================================================================
# ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    print(f"🖥️  Device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
        print(f"   VRAM disponible: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    elif DEVICE == "cpu":
        print("   ⚠️  Sin GPU — el training va a ser muy lento.")
        print("   Abrí esto en Google Colab con runtime GPU.")

    finetune()

🖥️  Device: cuda
   GPU: Tesla T4
   VRAM disponible: 15.6 GB
💊 SFT — FINE-TUNING CryoCCM V4.2
   Convirtiendo base model → asistente conversacional

📦 Cargando tokenizador...
✅ NeoToken cargado: 22619 tokens en vocabulario.
   Vocab: 22619 tokens

🧠 Cargando base model...
   Parámetros: 189.1M
   VRAM base: 1.52 GB

📥 Preparando dataset SFT...
📥 Descargando dataset SFT en español...
   Probando somosnlp/somos-alpaca-es...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002-9c03ba7ec0a116(…):   0%|          | 0.00/328M [00:00<?, ?B/s]

data/train-00001-of-00002-6da406345ec5a0(…):   0%|          | 0.00/325M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

   Keys reales: ['text', 'inputs', 'prediction', 'prediction_agent', 'annotation', 'annotation_agent', 'vectors', 'multi_label', 'explanation', 'id', 'metadata', 'status', 'event_timestamp', 'metrics']
   ⚠️ Solo 0 útiles en somosnlp/somos-alpaca-es, siguiente...
   Probando yahma/alpaca-cleaned...


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

   Keys reales: ['output', 'input', 'instruction']
   ✅ 4500 ejemplos válidos de yahma/alpaca-cleaned
   Split: 4050 train | 450 val
   📚 Dataset SFT: 3945 ejemplos válidos (105 descartados por longitud)
   📚 Dataset SFT: 442 ejemplos válidos (8 descartados por longitud)

📐 Plan de entrenamiento:
   Epochs:          3
   Steps/epoch:     494
   Total steps:     1482
   Effective batch: 8
   LR:              3e-05 → 3e-06
   ACT steps:       2 (reducido para velocidad)
   ⏱️  Tiempo estimado T4 Free: 20-30 minutos

🔥 INICIANDO SFT


📚 ÉPOCA 1/3
  Step   10/1482 | Loss 1.415 | LR 3.00e-06 | VRAM 5.1G | ETA 1:46:34
  Step   20/1482 | Loss 1.238 | LR 6.00e-06 | VRAM 5.1G | ETA 1:46:39
  Step   30/1482 | Loss 1.063 | LR 9.00e-06 | VRAM 5.1G | ETA 1:44:34
  Step   40/1482 | Loss 1.019 | LR 1.20e-05 | VRAM 5.1G | ETA 1:44:05
  Step   50/1482 | Loss 1.030 | LR 1.50e-05 | VRAM 5.1G | ETA 1:43:42
  Step   60/1482 | Loss 0.942 | LR 1.80e-05 | VRAM 5.1G | ETA 1:43:02
  Step   70/1482 | Loss 1.006 

# RIO ZIPH - MODELO FLUVIAL

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
import torch.optim as optim
from datasets import load_dataset
from collections import Counter, defaultdict # Added defaultdict here
import re
import random
import time
from tqdm import tqdm
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}\n")

# =============================================
# 1. TOKENIZER BPE + Zipf-aware (mejora sin romper esencia)
# =============================================
class RioZipfTokenizer:
    def __init__(self, max_vocab=20000):
        self.special_tokens = {'[PAD]':0, '[UNK]':1, '[BOS]':2, '[EOS]':3, '[USER]':4, '[ASSISTANT]':5, '[SYSTEM]':6}
        self.vocab = self.special_tokens.copy()
        self.id_to_token = {}
        self.merges = []
        self.max_vocab = max_vocab

    def build_from_corpus(self, texts):
        print("Construyendo tokenizer BPE + Zipf...")
        all_text = " ".join(texts[:25000]).lower()
        all_text = re.sub(r'[^a-záéíóúñü\s.,!?¿¡]', ' ', all_text)
        words = all_text.split()
        word_freq = Counter(words)

        # Inicial BPE
        vocab_tokenized = {}
        for word, freq in word_freq.most_common(14000):
            chars = tuple(list(word) + ['</w>'])
            vocab_tokenized[chars] = freq

        # Aprender merges
        num_merges = min(self.max_vocab - len(self.special_tokens) - 1000, 8500)
        for _ in tqdm(range(num_merges), desc="BPE merges"):
            pairs = defaultdict(int) # defaultdict correctly imported now
            for tokens, freq in vocab_tokenized.items():
                for i in range(len(tokens)-1):
                    pairs[(tokens[i], tokens[i+1])] += freq
            if not pairs: break
            best = max(pairs, key=pairs.get)
            self.merges.append(best)
            # Aplicar merge
            new_vocab = {}
            pattern = re.compile(r'(?<!\S)' + re.escape(' '.join(best)) + r'(?!\S)')
            for tokens, freq in vocab_tokenized.items():
                new_str = pattern.sub(''.join(best), ' '.join(tokens))
                new_vocab[tuple(new_str.split())] = freq
            vocab_tokenized = new_vocab

        # Vocab final
        idx = len(self.special_tokens)
        all_tokens = set()
        for tokens in vocab_tokenized:
            all_tokens.update(tokens)
        for tok in sorted(all_tokens):
            if tok not in self.vocab and idx < self.max_vocab:
                self.vocab[tok] = idx
                idx += 1

        self.id_to_token = {v: k for k, v in self.vocab.items()}
        print(f"Tokenizer BPE listo → {len(self.vocab)} tokens\n")

    def encode(self, text):
        text = re.sub(r'\s+', ' ', text.lower()).strip()
        words = text.split()
        tokens = []
        for word in words:
            wt = list(word) + ['</w>']
            for merge in self.merges:
                i = 0
                while i < len(wt)-1:
                    if (wt[i], wt[i+1]) == merge:
                        wt[i] = merge[0] + merge[1]
                        del wt[i+1]
                    else:
                        i += 1
            tokens.extend(wt)
        ids = [self.vocab.get(t, self.vocab['[UNK]']) for t in tokens]
        return [self.vocab['[BOS]']] + ids + [self.vocab['[EOS]']]

    def decode(self, ids):
        text = ''.join([self.id_to_token.get(i, '[UNK]') for i in ids])
        text = text.replace('</w>', ' ')
        text = re.sub(r'\s+', ' ', text).strip()
        return text


# =============================================
# 2. MODELO - RÍOZIPF (fiel a nuestra teoría)
# =============================================
class RioZipf(nn.Module):
    def __init__(self, vocab_size, dim=320, num_canales=4, num_charcas=28):
        super().__init__()
        self.dim = dim
        self.num_canales = num_canales

        self.embedding = nn.Embedding(vocab_size, dim)
        self.gating = nn.Sequential(
            nn.Linear(dim + 64, 256),
            nn.ReLU(),
            nn.Linear(256, num_canales + 2)   # +1 principal +1 incertidumbre
        )
        self.charcas = nn.Parameter(torch.zeros(num_charcas, dim) * 0.02)
        self.canales = nn.ModuleList([Canal(dim, deep=(i==0)) for i in range(num_canales)])
        self.output_proj = nn.Linear(dim, vocab_size)
        self.register_buffer('h_principal', torch.zeros(dim))

    def reset_state(self):
        self.h_principal.zero_()

    def forward(self, input_ids, reset_state=True):
        if input_ids.dim() == 1:
            input_ids = input_ids.unsqueeze(0)
        input_ids = input_ids.to(device)

        B, T = input_ids.shape
        all_logits = []

        for b in range(B):
            if reset_state:
                self.reset_state()

            h = self.h_principal.clone()
            recent = []

            for t in range(T):
                emb = self.embedding(input_ids[b, t]).squeeze(0)
                recent.append(input_ids[b, t].item())

                # Zipf features (placeholder for now)
                zipf_feat = torch.zeros(64, device=device)

                # Ensure h is expanded to match dimensions for concatenation if it's 1D
                h_concat = h.clone().detach() # Detach to prevent gradient issues if h is used elsewhere
                if h_concat.dim() == 0: # If h is a scalar
                  h_concat = h_concat.unsqueeze(0)
                if h_concat.dim() == 1 and h_concat.shape[0] != self.dim:
                    h_concat = h_concat.expand(self.dim) # Expand if h is 1D but needs to match self.dim

                gating_input = torch.cat([h_concat, zipf_feat], dim=-1)

                gate_probs = F.softmax(self.gating(gating_input), dim=0)

                p_main = gate_probs[0]
                p_canales = gate_probs[1:-1]
                uncertainty = gate_probs[-1]

                # Río principal (flujo hacia Z)
                alpha = 0.93 if p_main > 0.6 else 0.77
                steps = 1 + int(uncertainty.item() * 2)   # adaptive compute ligero
                for _ in range(steps):
                    h = alpha * h + (1 - alpha) * emb

                # Bifurcaciones hacia Y (canales)
                h_canales = []
                for i in range(self.num_canales):
                    # Removed random.random() condition for p_canales activation to ensure consistent flow
                    if p_canales[i].item() > 0.5: # Activate based on a threshold
                        h_c = self.canales[i](h, emb)
                        h_canales.append(h_c)
                        # Escribir en charca
                        with torch.no_grad():
                            idx = (t + i) % self.charcas.shape[0]
                            self.charcas.data[idx] = 0.85 * self.charcas.data[idx] + 0.15 * h_c

                # Leer charcas
                retrieved = torch.zeros(self.dim, device=device)
                if random.random() < 0.35:
                    sim = F.cosine_similarity(h.unsqueeze(0), self.charcas, dim=1)
                    if sim.max() > 0.25:
                        retrieved = self.charcas[sim.argmax()] * 0.18

                # Mezcla final
                h_canales_sum = sum(h_canales) if h_canales else torch.zeros_like(h) # Ensure h_canales_sum is a tensor
                h_final = 0.60 * h + 0.25 * h_canales_sum + 0.15 * retrieved

                logit = self.output_proj(h_final)
                all_logits.append(logit)

        return torch.stack(all_logits).unsqueeze(0)


class Canal(nn.Module):
    def __init__(self, dim=320, deep=False):
        super().__init__()
        self.deep = deep
        self.proj = nn.Linear(dim, dim)
        if deep:
            self.extra = nn.Sequential(nn.Linear(dim, dim*2), nn.GELU(), nn.Linear(dim*2, dim))
        self.norm = nn.LayerNorm(dim)

    def forward(self, h, input_emb, alpha=0.87):
        x = self.proj(input_emb)
        if self.deep:
            x = x + self.extra(x)
        h_new = alpha * h + (1 - alpha) * x
        return self.norm(h_new)


# =============================================
# 4. DATASET + ENTRENAMIENTO CON BUEN LOGGING
# =============================================
class InfiniteDataset(IterableDataset):
    def __init__(self, texts, tokenizer, block_size=128, min_len_factor=1.5):
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.min_valid_len = block_size + 1 # Minimum length for a single example
        self.min_len_factor = min_len_factor

        # Pre-process and filter texts in __init__
        print("Pre-procesando y filtrando textos para InfiniteDataset...")
        self.processed_samples = []
        for text in tqdm(texts, desc="Codificando textos"): # Use tqdm for progress
            ids = self.tokenizer.encode(text)
            # Only keep texts that are significantly longer than a single block
            # to allow for some variability in starting positions
            if len(ids) >= self.block_size + int(self.block_size * min_len_factor):
                self.processed_samples.append(ids)

        # Shuffle the processed samples once
        random.shuffle(self.processed_samples)
        self.num_samples = len(self.processed_samples)
        print(f"Dataset listo con {self.num_samples} textos válidos pre-procesados.")

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        worker_id = worker_info.id if worker_info else 0
        # Use a separate RNG for each worker to ensure diverse samples
        rng = random.Random(worker_id + int(time.time()))

        # Create a shuffled list of indices to iterate through the processed samples
        indices = list(range(self.num_samples))
        rng.shuffle(indices)

        sample_idx = 0
        while True:
            if self.num_samples == 0: # Handle empty dataset gracefully
                yield torch.empty(self.block_size), torch.empty(self.block_size)
                continue

            # Iterate through the pre-processed and shuffled samples
            current_ids = self.processed_samples[indices[sample_idx]]
            sample_idx = (sample_idx + 1) % self.num_samples
            if sample_idx == 0: # Reshuffle indices after one full pass
                rng.shuffle(indices)

            # Randomly select a starting point within the current text
            # The minimum length ensures there's always a full block_size + 1 sequence
            if len(current_ids) < self.min_valid_len: # Should not happen with pre-filtering, but as a safeguard
                continue

            start = rng.randint(0, len(current_ids) - self.min_valid_len)
            chunk = current_ids[start : start + self.min_valid_len]

            yield torch.tensor(chunk[:-1], dtype=torch.long), torch.tensor(chunk[1:], dtype=torch.long)

def train_model(model, tokenizer, epochs=8, batch_size=14, grad_accum=6, learning_rate=4e-4, warmup_steps=100, max_steps=10000):
    print("Cargando datos reales (CulturaX es)... (esto puede tomar un tiempo para pre-procesar)")
    texts = []
    try:
        # Load more data for better representation and more effective filtering
        ds = load_dataset("uonlp/CulturaX", "es", split="train", streaming=True)
        for i, ex in enumerate(ds):
            if i >= 500000: break # Increased data size for better sampling
            texts.append(ex['text'])
        print(f"Se cargaron {len(texts)} textos de CulturaX.")
    except Exception as e:
        print(f"FALLO al cargar CulturaX: {e}. Usando fallback sintético...")
        fallback = "el río guayas fluye por guayaquil. las personas hablan español con calidez. " * 50000 # Increased size
        texts = [fallback]
        print(f"Usando {len(texts)} textos sintéticos.")

    # Create dataset AFTER tokenizer is built
    dataset = InfiniteDataset(texts, tokenizer, block_size=model.charcas.shape[0]) # Use charcas size for block_size as a proxy if no other config
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=2, pin_memory=(device.type=="cuda"))

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    model = model.to(device)
    model.train()

    start_time = time.time()
    global_step = 0

    # Learning rate scheduler (linear warmup + cosine decay)
    def get_lr(step):
        if step < warmup_steps:
            return learning_rate * (step / warmup_steps)
        if step > max_steps:
            return learning_rate * 0.1 # Minimum LR
        # Cosine decay
        progress = (step - warmup_steps) / (max_steps - warmup_steps)
        return learning_rate * (0.5 * (1 + math.cos(math.pi * progress)))

    total_train_steps = len(dataset) // (batch_size * grad_accum) * epochs if len(dataset) > 0 else max_steps # Fallback if dataset is empty
    print(f"Total de pasos de entrenamiento estimados: {total_train_steps}")

    for epoch in range(epochs):
        accum_loss = 0.0
        optimizer.zero_grad()
        pbar = tqdm(enumerate(loader), total=total_train_steps // epochs, desc=f"Epoch {epoch+1}/{epochs}") # Correct total for tqdm

        for batch_idx, (input_ids, targets) in pbar:
            input_ids = input_ids.to(device)
            targets = targets.to(device)

            logits = model(input_ids, reset_state=True)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=tokenizer.vocab['[PAD]']) # Use PAD as ignore_index
            loss = loss / grad_accum
            loss.backward()

            accum_loss += loss.item() * grad_accum

            if (batch_idx + 1) % grad_accum == 0:
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                optimizer.step()
                optimizer.zero_grad()
                global_step += 1

                current_lr = get_lr(global_step)
                for param_group in optimizer.param_groups:
                    param_group['lr'] = current_lr

                if global_step % 1 == 0: # Log every step
                    elapsed = time.time() - start_time
                    tokens_processed = global_step * batch_size * grad_accum * model.charcas.shape[0] # Use block_size from model or dataset
                    tps = tokens_processed / elapsed if elapsed > 0 else 0

                    pbar.set_postfix({
                        'Loss': f"{accum_loss/grad_accum:.4f}",
                        'LR': f"{current_lr:.2e}",
                        'Tokens/s': f"{tps:.0f}",
                        'Step': global_step
                    })
                accum_loss = 0.0

        print(f"Epoch {epoch+1} completada.\n")

    torch.save({'model_state': model.state_dict(), 'vocab': tokenizer.vocab}, "riozipf_v1.0_fiel.pth")
    print("Modelo guardado como riozipf_v1.0_fiel.pth")
    return model


# =============================================
# EJECUCIÓN
# =============================================
if __name__ == "__main__":
    tokenizer = RioZipfTokenizer(max_vocab=22000)

    print("Construyendo vocabulario...")
    sample_texts = []
    try:
        ds = load_dataset("uonlp/CulturaX", "es", split="train", streaming=True)
        for i, ex in enumerate(ds):
            if i >= 20000: break
            sample_texts.append(ex['text'])
    except Exception as e: # Catch any exception during dataset loading
        print(f"FALLO al cargar CulturaX para construir vocabulario: {e}. Usando fallback sintético...")
        sample_texts = ["el río guayas fluye por guayaquil. las personas hablan español con calidez."] * 15000

    tokenizer.build_from_corpus(sample_texts)

    # Use a dynamic block_size for the model based on a reasonable chunk size, e.g., 128 or 256.
    # The previous code used model.charcas.shape[0] which is 28, too small for block_size.
    # Let's define a fixed block_size for the model and dataset here.
    model_block_size = 128
    model = RioZipf(vocab_size=len(tokenizer.vocab), num_charcas=model_block_size)

    print("\nIniciando entrenamiento RíoZipf v1.0 - Fiel a nuestra teoría...\n")
    # Pass model_block_size to InfiniteDataset for proper filtering.
    # Adjusted max_steps to be more reasonable for a typical training run.
    model = train_model(model, tokenizer, epochs=8, batch_size=14, grad_accum=6, max_steps=5000)


Dispositivo: cuda
GPU: Tesla T4

Construyendo vocabulario...


Resolving data files:   0%|          | 0/512 [00:00<?, ?it/s]

# New Tokenizer

In [ ]:
"""NeoToken V2 — tokenizador lossless y validado.

Objetivos de esta revisión
--------------------------
1. El texto original siempre puede recuperarse con ``decode(encode(text))``.
2. El código Python NO se sustituye automáticamente por una representación AST
   destructiva. El AST queda disponible únicamente como característica auxiliar
   o como modo legacy explícitamente habilitado.
3. Texto, código, JSON y matemáticas usan por defecto el mismo canal subword con
   fallback completo a bytes. La escala LaTeX puede activarse, pero conserva el
   texto exacto y debe usarse de forma consistente en training e inferencia.
4. El vocabulario se valida al cargar: IDs contiguos, mapas biyectivos, tokens
   especiales coherentes y byte fallback completo.
5. El BPE greedy usa la longitud máxima real del vocabulario, no un límite fijo
   de 20 caracteres.
6. La caché es LRU, limitada y opcional. No crece sin control durante la creación
   masiva de datasets.
7. Los IDs desconocidos y [UNK] nunca desaparecen silenciosamente al decodificar.

Compatibilidad
--------------
Mantiene la interfaz principal de la versión anterior::

    tokenizer = NeoToken()
    tokenizer.load("neotoken_22k.pkl")
    ids = tokenizer.encode(text, adaptive=True)
    text = tokenizer.decode(ids)

Los archivos pickle solo deben cargarse si son de confianza: ``pickle`` puede
 ejecutar código arbitrario al deserializar.
"""

from __future__ import annotations

import ast
import os
import pickle
import re
import warnings
from collections import Counter, OrderedDict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple


@dataclass
class TokenScale:
    """Describe una escala almacenada en el vocabulario."""

    level: int
    granularity: str
    vocab_size: int
    compression_ratio: float


class PatternMemory:
    """Estructura legacy conservada para poder cargar pickles anteriores.

    No participa en la tokenización del LM. Una memoria de patrones mutable en el
    tokenizador haría que la representación dependiera del historial de llamadas.
    """

    def __init__(self, max_patterns: int = 10_000):
        self.max_patterns = max_patterns
        self.patterns: Counter = Counter()
        self.context_cache: Dict[Tuple[int, ...], str] = {}

    def observe(self, tokens: Sequence[int], context: str) -> None:
        pattern = tuple(tokens[:10])
        self.patterns[pattern] += 1
        if self.patterns[pattern] > 10:
            self.context_cache[pattern] = context

    def get_frequent_patterns(self, top_k: int = 100) -> List[Tuple]:
        return self.patterns.most_common(top_k)

    def evolve(self) -> None:
        if len(self.patterns) > self.max_patterns:
            self.patterns = Counter(dict(self.patterns.most_common(self.max_patterns)))


class NeoToken:
    """Tokenizador multiescala con ruta principal lossless.

    Parámetros importantes:
        enable_ast_mode:
            Conservado por compatibilidad. El AST nunca se usa automáticamente.
        enable_latex_mode:
            Si es True y ``adaptive=True``, solo los segmentos delimitados como
            matemáticas se tokenizan con escala 5. El resto sigue usando subword.
        allow_lossy_ast:
            Permite solicitar explícitamente ``preferred_scale=4``. Debe quedar
            en False para pretraining/SFT normal.
        cache_size:
            Máximo de segmentos subword en caché LRU. Use 0 al construir datasets.
        add_scale_markers:
            Conserva los tokens [SCALE_X]. Para un tokenizer nuevo de una sola
            ruta puede ponerse en False, pero debe ser igual en train/inferencia.
    """

    DEFAULT_SPECIAL_TOKENS = {
        "[PAD]": 0,
        "[UNK]": 1,
        "[CLS]": 2,
        "[SEP]": 3,
        "[MASK]": 4,
        "[CODE_START]": 5,
        "[CODE_END]": 6,
        "[STRUCT_START]": 7,
        "[STRUCT_END]": 8,
        "[MATH_START]": 9,
        "[MATH_END]": 10,
        "[SCALE_0]": 11,
        "[SCALE_1]": 12,
        "[SCALE_2]": 13,
        "[SCALE_3]": 14,
        "[SCALE_4]": 15,
        "[SCALE_5]": 16,
    }

    def __init__(
        self,
        target_vocab_size: int = 50_000,
        enable_ast_mode: bool = False,
        enable_memory: bool = False,
        *,
        enable_latex_mode: bool = False,
        allow_lossy_ast: bool = False,
        cache_size: int = 4_096,
        add_scale_markers: bool = True,
    ) -> None:
        self.target_vocab_size = int(target_vocab_size)
        self.enable_ast = bool(enable_ast_mode)
        self.enable_memory = bool(enable_memory)
        self.enable_latex_mode = bool(enable_latex_mode)
        self.allow_lossy_ast = bool(allow_lossy_ast)
        self.cache_size = max(0, int(cache_size))
        self.add_scale_markers = bool(add_scale_markers)

        self.scales = {
            0: TokenScale(0, "byte", 256, 1.0),
            1: TokenScale(1, "char", 512, 1.0),
            2: TokenScale(2, "subword", 25_000, 3.5),
            3: TokenScale(3, "word", 12_000, 5.0),
            4: TokenScale(4, "ast_node_lossy", 5_000, 8.0),
            5: TokenScale(5, "latex_math_lossless", 8_000, 6.0),
        }
        self.special_tokens: Dict[str, int] = dict(self.DEFAULT_SPECIAL_TOKENS)

        # Solo se conserva para compatibilidad de serialización.
        self.pattern_memory = PatternMemory() if enable_memory else None
        self.token_to_id: Dict[str, int] = {}
        self.id_to_token: Dict[int, str] = {}
        self.vocabs: Dict[int, dict] = {}
        self._is_trained = False

        # Índices derivados. Nunca se persisten.
        self._subword_prefixes: set[str] = set()
        self._max_subword_length = 0
        self._bpe_cache: OrderedDict[str, Tuple[int, ...]] = OrderedDict()
        self._special_token_names: set[str] = set(self.special_tokens)

    # ------------------------------------------------------------------
    # Propiedades y configuración
    # ------------------------------------------------------------------

    @property
    def vocab_size(self) -> int:
        """Dimensión necesaria para el embedding: ``max_id + 1``.

        Usar ``len(token_to_id)`` oculta vocabularios con huecos o IDs fuera de
        rango. La validación estricta rechaza esos casos, pero esta propiedad sigue
        siendo correcta incluso antes de validar.
        """

        return max(self.token_to_id.values(), default=-1) + 1

    def __len__(self) -> int:
        return self.vocab_size

    def __repr__(self) -> str:
        status = "cargado" if self._is_trained else "sin cargar"
        return (
            f"NeoToken(vocab={self.vocab_size}, latex={self.enable_latex_mode}, "
            f"lossy_ast={self.allow_lossy_ast}, status={status})"
        )

    def get_scale_name(self, scale_id: int) -> str:
        scale = self.scales.get(scale_id)
        return scale.granularity if scale is not None else "unknown"

    def set_cache_size(self, cache_size: int) -> None:
        self.cache_size = max(0, int(cache_size))
        while len(self._bpe_cache) > self.cache_size:
            self._bpe_cache.popitem(last=False)
        if self.cache_size == 0:
            self.clear_cache()

    def clear_cache(self) -> None:
        self._bpe_cache.clear()

    # ------------------------------------------------------------------
    # Persistencia y validación
    # ------------------------------------------------------------------

    def load(
        self,
        path: str,
        *,
        strict: bool = True,
        repair_id_map: bool = True,
        require_full_byte_fallback: bool = True,
    ) -> None:
        """Carga un pickle de confianza, reconstruye índices y valida el vocabulario."""

        if not os.path.exists(path):
            raise FileNotFoundError(f"Archivo de modelo no encontrado: {path}")

        with open(path, "rb") as file:
            state = pickle.load(file)
        if not isinstance(state, dict):
            raise TypeError(f"Estado inválido en {path}: se esperaba dict, llegó {type(state)!r}")

        # La configuración que altera la representación (LaTeX y marcadores)
        # se restaura desde el archivo V2. Para pickles legacy se conservan los
        # defaults del objeto actual. La caché y el permiso AST son opciones de
        # runtime y nunca deben activarse implícitamente desde un pickle.
        legacy_defaults = {
            "enable_latex_mode": self.enable_latex_mode,
            "add_scale_markers": self.add_scale_markers,
        }
        runtime_cache_size = self.cache_size
        runtime_allow_lossy_ast = self.allow_lossy_ast
        self.__dict__.update(state)
        for key, value in legacy_defaults.items():
            if key not in state:
                setattr(self, key, value)
        self.cache_size = runtime_cache_size
        self.allow_lossy_ast = runtime_allow_lossy_ast

        self._restore_legacy_types()
        if repair_id_map:
            self._repair_id_map()
        self.validate_vocab(
            strict=strict,
            require_full_byte_fallback=require_full_byte_fallback,
        )
        self._is_trained = True
        self._build_runtime_indexes()
        print(
            f"✅ NeoToken cargado y validado: {self.vocab_size:,} IDs, "
            f"BPE máx. {self._max_subword_length} caracteres."
        )

    def save(self, path: str) -> None:
        """Guarda solo estado estable; cachés e índices se reconstruyen al cargar."""

        if not self.token_to_id:
            raise ValueError("No se puede guardar un vocabulario vacío.")
        self.validate_vocab(strict=True, require_full_byte_fallback=True)

        state = {
            "format_version": 2,
            "target_vocab_size": self.target_vocab_size,
            "enable_ast": self.enable_ast,
            "enable_memory": self.enable_memory,
            "enable_latex_mode": self.enable_latex_mode,
            "allow_lossy_ast": self.allow_lossy_ast,
            "add_scale_markers": self.add_scale_markers,
            "scales": {
                key: (
                    value.level,
                    value.granularity,
                    value.vocab_size,
                    value.compression_ratio,
                )
                for key, value in self.scales.items()
            },
            "vocabs": self.vocabs,
            "special_tokens": self.special_tokens,
            "token_to_id": self.token_to_id,
            "id_to_token": self.id_to_token,
            "pattern_memory": (
                self.pattern_memory.__dict__ if self.pattern_memory is not None else None
            ),
        }
        with open(path, "wb") as file:
            pickle.dump(state, file, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"💾 NeoToken guardado en {path}")

    def _restore_legacy_types(self) -> None:
        raw_scales = getattr(self, "scales", {})
        rebuilt_scales: Dict[int, TokenScale] = {}
        for raw_key, raw_value in raw_scales.items():
            key = int(raw_key)
            if isinstance(raw_value, TokenScale):
                rebuilt_scales[key] = raw_value
            elif isinstance(raw_value, (tuple, list)) and len(raw_value) == 4:
                rebuilt_scales[key] = TokenScale(*raw_value)
            elif isinstance(raw_value, dict):
                rebuilt_scales[key] = TokenScale(**raw_value)
            else:
                raise TypeError(
                    f"Tipo inesperado para scales[{raw_key!r}]: {type(raw_value)!r}"
                )
        self.scales = rebuilt_scales or self.scales

        pattern_memory = getattr(self, "pattern_memory", None)
        if isinstance(pattern_memory, dict):
            rebuilt_memory = PatternMemory()
            rebuilt_memory.__dict__.update(pattern_memory)
            self.pattern_memory = rebuilt_memory
        elif pattern_memory is not None and not isinstance(pattern_memory, PatternMemory):
            warnings.warn(
                f"pattern_memory legacy de tipo {type(pattern_memory)!r}; se desactiva.",
                RuntimeWarning,
            )
            self.pattern_memory = None

        self.token_to_id = {
            str(token): int(token_id)
            for token, token_id in getattr(self, "token_to_id", {}).items()
        }
        self.id_to_token = {
            int(token_id): str(token)
            for token_id, token in getattr(self, "id_to_token", {}).items()
        }
        self.special_tokens = {
            str(token): int(token_id)
            for token, token_id in getattr(
                self, "special_tokens", self.DEFAULT_SPECIAL_TOKENS
            ).items()
        }

    def _repair_id_map(self) -> None:
        """Reconstruye id_to_token únicamente cuando token_to_id no tiene colisiones."""

        reverse: Dict[int, str] = {}
        for token, token_id in self.token_to_id.items():
            if token_id in reverse and reverse[token_id] != token:
                raise ValueError(
                    f"Colisión de ID {token_id}: {reverse[token_id]!r} y {token!r}."
                )
            reverse[token_id] = token
        if reverse != self.id_to_token:
            warnings.warn(
                "id_to_token no coincidía con token_to_id; fue reconstruido.",
                RuntimeWarning,
            )
            self.id_to_token = reverse

    def validate_vocab(
        self,
        *,
        strict: bool = True,
        require_full_byte_fallback: bool = True,
    ) -> Dict[str, object]:
        """Valida invariantes necesarias para embeddings y roundtrip lossless."""

        errors: List[str] = []
        warnings_list: List[str] = []

        if not self.token_to_id:
            errors.append("token_to_id está vacío")
        if len(set(self.token_to_id.values())) != len(self.token_to_id):
            errors.append("token_to_id contiene IDs duplicados")

        ids = sorted(self.token_to_id.values())
        if ids and ids != list(range(ids[-1] + 1)):
            missing = sorted(set(range(ids[-1] + 1)).difference(ids))
            errors.append(f"los IDs no son contiguos; faltan {missing[:20]}")

        for token, token_id in self.token_to_id.items():
            if self.id_to_token.get(token_id) != token:
                errors.append(f"mapa inverso inconsistente para {token!r} -> {token_id}")
                if len(errors) >= 25:
                    break

        for token, token_id in self.special_tokens.items():
            actual = self.token_to_id.get(token)
            if actual is None:
                errors.append(f"token especial ausente: {token}")
            elif actual != token_id:
                errors.append(
                    f"ID especial incoherente para {token}: esperado {token_id}, actual {actual}"
                )

        missing_bytes = [
            value
            for value in range(256)
            if f"[B:{value:02x}]" not in self.token_to_id
        ]
        if missing_bytes:
            message = (
                f"faltan {len(missing_bytes)} tokens byte; primeros: "
                f"{[f'{value:02x}' for value in missing_bytes[:16]]}"
            )
            if require_full_byte_fallback:
                errors.append(message)
            else:
                warnings_list.append(message)

        report = {
            "vocab_size": self.vocab_size,
            "num_tokens": len(self.token_to_id),
            "errors": errors,
            "warnings": warnings_list,
            "valid": not errors,
        }

        for message in warnings_list:
            warnings.warn(message, RuntimeWarning)
        if errors:
            message = "Vocabulario inválido:\n- " + "\n- ".join(errors)
            if strict:
                raise ValueError(message)
            warnings.warn(message, RuntimeWarning)
        return report

    def _build_runtime_indexes(self) -> None:
        self._special_token_names = set(self.special_tokens)
        self._subword_prefixes = set()
        self._max_subword_length = 0
        self._bpe_cache = OrderedDict()

        for token in self.token_to_id:
            payload = self._payload(token, "[S:")
            if payload is None:
                continue
            self._max_subword_length = max(self._max_subword_length, len(payload))
            for length in range(1, len(payload) + 1):
                self._subword_prefixes.add(payload[:length])

    # ------------------------------------------------------------------
    # API de codificación
    # ------------------------------------------------------------------

    def encode(
        self,
        text: str,
        preferred_scale: Optional[int] = None,
        adaptive: bool = True,
        *,
        allow_lossy: Optional[bool] = None,
    ) -> List[int]:
        """Codifica texto.

        La ruta por defecto es lossless. El código no activa AST. Para obtener
        features AST use ``extract_ast_features``; para reproducir el modo legacy
        debe solicitar explícitamente escala 4 y ``allow_lossy=True``.
        """

        if not self._is_trained:
            raise ValueError("NeoToken no ha sido cargado o entrenado.")
        if not isinstance(text, str):
            raise TypeError(f"text debe ser str, llegó {type(text)!r}")
        if text == "":
            return []

        lossy_allowed = self.allow_lossy_ast if allow_lossy is None else allow_lossy

        if preferred_scale is not None:
            if preferred_scale == 4 and not lossy_allowed:
                raise ValueError(
                    "La escala AST es destructiva. Use extract_ast_features() o "
                    "pase allow_lossy=True de forma explícita."
                )
            return self._dispatch_encoder(text, preferred_scale)

        if adaptive and self.enable_latex_mode:
            return self._encode_dynamic_stream(text)

        # Ruta estable recomendada para texto, código, JSON y matemáticas.
        return self._encode_subwords(text)

    def _dispatch_encoder(self, text: str, scale: int) -> List[int]:
        encoders = {
            0: self._encode_bytes,
            1: self._encode_chars,
            2: self._encode_subwords,
            3: self._encode_words,
            4: self._encode_ast_lossy,
            5: self._encode_science_latex,
        }
        try:
            encoder = encoders[int(scale)]
        except (KeyError, ValueError):
            raise ValueError(f"Escala inválida: {scale!r}; se esperaba 0..5") from None
        return encoder(text)

    def _encode_dynamic_stream(self, text: str) -> List[int]:
        """Enrutamiento lossless: BPE general y LaTeX solo en delimitadores matemáticos.

        No existe detección automática de código. Esa detección era destructiva y
        producía una distribución distinta entre training e inferencia.
        """

        result: List[int] = []
        for is_math, segment in self._split_math_segments(text):
            if not segment:
                continue
            if is_math:
                result.extend(self._encode_science_latex(segment))
            else:
                result.extend(self._encode_subwords(segment))
        return result

    @staticmethod
    def _split_math_segments(text: str) -> Iterable[Tuple[bool, str]]:
        # Soporta $$...$$, $...$, \[...\] y \(...\). El lookbehind simple evita
        # interpretar \$ como inicio de matemática.
        pattern = re.compile(
            r"(?<!\\)(\$\$.*?(?<!\\)\$\$|\$(?:\\.|[^$\n])*?(?<!\\)\$|"
            r"\\\[.*?\\\]|\\\(.*?\\\))",
            flags=re.DOTALL,
        )
        cursor = 0
        for match in pattern.finditer(text):
            if match.start() > cursor:
                yield False, text[cursor : match.start()]
            yield True, match.group(0)
            cursor = match.end()
        if cursor < len(text):
            yield False, text[cursor:]

    # ------------------------------------------------------------------
    # Decodificación
    # ------------------------------------------------------------------

    def decode(
        self,
        ids: Sequence[int],
        *,
        strict: bool = False,
        skip_special_tokens: bool = True,
        replacement: str = "�",
        legacy_ast_decode: bool = False,
    ) -> str:
        """Decodifica IDs sin ocultar corrupción.

        ``strict=True`` lanza una excepción ante ID desconocido, [UNK], byte
        inválido o token no reconocido. En modo tolerante se agrega ``replacement``.
        Los AST legacy se muestran como ``<AST:...>`` salvo que se solicite
        explícitamente la reconstrucción aproximada.
        """

        result: List[str] = []
        byte_buffer = bytearray()
        self._ast_decode_state = {"func_count": 0, "var_count": 0, "indent": 0}

        def flush_bytes() -> None:
            nonlocal byte_buffer
            if not byte_buffer:
                return
            try:
                result.append(byte_buffer.decode("utf-8", errors="strict"))
            except UnicodeDecodeError as exc:
                if strict:
                    raise ValueError(f"Secuencia UTF-8 inválida durante decode: {exc}") from exc
                result.append(byte_buffer.decode("utf-8", errors="replace"))
            byte_buffer = bytearray()

        for position, raw_id in enumerate(ids):
            try:
                token_id = int(raw_id)
            except (TypeError, ValueError) as exc:
                if strict:
                    raise TypeError(f"ID inválido en posición {position}: {raw_id!r}") from exc
                flush_bytes()
                result.append(replacement)
                continue

            token = self.id_to_token.get(token_id)
            if token is None:
                if strict:
                    raise KeyError(f"ID desconocido {token_id} en posición {position}")
                flush_bytes()
                result.append(replacement)
                continue

            if token == "[UNK]":
                if strict:
                    raise ValueError(f"[UNK] encontrado en posición {position}")
                flush_bytes()
                result.append(replacement)
                continue

            if token.startswith("[SCALE_"):
                flush_bytes()
                if not skip_special_tokens:
                    result.append(token)
                continue

            if token in self._special_token_names:
                flush_bytes()
                if not skip_special_tokens:
                    result.append(token)
                continue

            payload = self._payload(token, "[B:")
            if payload is not None:
                try:
                    raw = bytes.fromhex(payload)
                except ValueError as exc:
                    if strict:
                        raise ValueError(f"Token byte inválido {token!r}") from exc
                    flush_bytes()
                    result.append(replacement)
                    continue
                if len(raw) != 1:
                    if strict:
                        raise ValueError(f"Token byte debe contener un byte: {token!r}")
                    flush_bytes()
                    result.append(replacement)
                    continue
                byte_buffer.extend(raw)
                continue

            flush_bytes()

            payload = self._payload(token, "[C:")
            if payload is not None:
                result.append(" " if payload == "▁" else payload)
                continue

            payload = self._payload(token, "[S:")
            if payload is not None:
                result.append(payload.replace("▁", " "))
                continue

            payload = self._payload(token, "[W:")
            if payload is not None:
                result.append(payload)
                continue

            payload = self._payload(token, "[L:")
            if payload is not None:
                result.append(payload)
                continue

            payload = self._payload(token, "[A:")
            if payload is not None:
                if legacy_ast_decode:
                    result.append(self._legacy_ast_pattern_to_code(payload))
                else:
                    result.append(f"<AST:{payload}>")
                continue

            if strict:
                raise ValueError(f"Formato de token desconocido: {token!r}")
            result.append(replacement)

        flush_bytes()
        return "".join(result)

    # ------------------------------------------------------------------
    # Encoders lossless
    # ------------------------------------------------------------------

    def _scale_prefix(self, level: int) -> List[int]:
        if not self.add_scale_markers:
            return []
        token = f"[SCALE_{level}]"
        token_id = self.special_tokens.get(token)
        if token_id is None:
            raise ValueError(f"Marcador de escala ausente: {token}")
        return [token_id]

    def _get_safe_char_tokens(self, char: str) -> List[int]:
        lookup_char = " " if char == "▁" else char
        char_token = f"[C:{lookup_char}]"
        if char_token in self.token_to_id:
            return [self.token_to_id[char_token]]
        return self._bytes_for_text(lookup_char)

    def _bytes_for_text(self, text: str) -> List[int]:
        result: List[int] = []
        for value in text.encode("utf-8"):
            token = f"[B:{value:02x}]"
            token_id = self.token_to_id.get(token)
            if token_id is None:
                raise ValueError(
                    f"Byte fallback incompleto: falta {token}. "
                    "Ejecute validate_vocab(require_full_byte_fallback=True)."
                )
            result.append(token_id)
        return result

    def _encode_bytes(self, text: str) -> List[int]:
        return self._scale_prefix(0) + self._bytes_for_text(text)

    def _encode_chars(self, text: str) -> List[int]:
        result = self._scale_prefix(1)
        for char in text:
            result.extend(self._get_safe_char_tokens(char))
        return result

    def _encode_subwords(self, text: str) -> List[int]:
        cached = self._cache_get(text)
        if cached is not None:
            return self._scale_prefix(2) + list(cached)

        inner: List[int] = []
        normalized = text.replace(" ", "▁")
        index = 0

        while index < len(normalized):
            first_char = normalized[index]
            if first_char not in self._subword_prefixes:
                inner.extend(self._get_safe_char_tokens(first_char))
                index += 1
                continue

            found = False
            max_length = min(self._max_subword_length, len(normalized) - index)
            for length in range(max_length, 0, -1):
                candidate = normalized[index : index + length]
                if candidate not in self._subword_prefixes:
                    continue
                token_id = self.token_to_id.get(f"[S:{candidate}]")
                if token_id is not None:
                    inner.append(token_id)
                    index += length
                    found = True
                    break

            if not found:
                inner.extend(self._get_safe_char_tokens(first_char))
                index += 1

        self._cache_put(text, inner)
        return self._scale_prefix(2) + inner

    def _encode_words(self, text: str) -> List[int]:
        result = self._scale_prefix(3)
        for match in re.finditer(r"\w+|\s+|[^\w\s]", text, flags=re.UNICODE):
            fragment = match.group(0)
            token_id = self.token_to_id.get(f"[W:{fragment}]")
            if token_id is not None:
                result.append(token_id)
                continue
            # Fallback subword conserva mejor la compresión que char-only.
            subword_ids = self._encode_subwords(fragment)
            if self.add_scale_markers and subword_ids:
                subword_ids = subword_ids[1:]
            result.extend(subword_ids)
        return result

    def _encode_science_latex(self, text: str) -> List[int]:
        """Escala LaTeX lossless con fallback subword/byte.

        El regex solo propone fragmentos; ningún fragmento se elimina. Por defecto
        esta escala se usa únicamente para segmentos matemáticos delimitados.
        """

        result = self._scale_prefix(5)
        pattern = re.compile(
            r"\\[a-zA-Z]+|\\.|\d+(?:\.\d+)?|[a-zA-Z]+|\s+|.",
            flags=re.DOTALL,
        )
        for match in pattern.finditer(text):
            fragment = match.group(0)
            token_id = self.token_to_id.get(f"[L:{fragment}]")
            if token_id is not None:
                result.append(token_id)
                continue
            # No convertir todo a chars: reutilizar BPE cuando no exista token LaTeX.
            subword_ids = self._encode_subwords(fragment)
            if self.add_scale_markers and subword_ids:
                subword_ids = subword_ids[1:]
            result.extend(subword_ids)
        return result

    # ------------------------------------------------------------------
    # AST: auxiliar, nunca ruta automática del LM
    # ------------------------------------------------------------------

    def extract_ast_features(self, text: str) -> List[str]:
        """Extrae features AST sin sustituir el código original."""

        try:
            tree = ast.parse(text)
        except SyntaxError as exc:
            raise ValueError(f"Código Python inválido: {exc}") from exc

        features: List[str] = []
        for node in ast.walk(tree):
            features.append(f"NODE:{type(node).__name__}")
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                features.append(f"FUNC:{len(node.args.args)}args")
            elif isinstance(node, (ast.For, ast.AsyncFor)):
                features.append("LOOP:for")
            elif isinstance(node, ast.While):
                features.append("LOOP:while")
            elif isinstance(node, ast.If):
                features.append("COND:if")
        return features

    # Alias legacy usado por versiones anteriores.
    def _extract_ast_patterns(self, tree: ast.AST) -> List[str]:
        source_features: List[str] = []
        for node in ast.walk(tree):
            source_features.append(f"NODE:{type(node).__name__}")
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                source_features.append(f"FUNC:{len(node.args.args)}args")
            elif isinstance(node, (ast.For, ast.AsyncFor)):
                source_features.append("LOOP:for")
            elif isinstance(node, ast.While):
                source_features.append("LOOP:while")
            elif isinstance(node, ast.If):
                source_features.append("COND:if")
        return source_features

    def _encode_ast_lossy(self, text: str) -> List[int]:
        """Modo legacy destructivo. Solo accesible con ``allow_lossy=True``."""

        try:
            features = self.extract_ast_features(text)
        except ValueError:
            # El código incompleto es común durante generación; conservarlo como BPE.
            return self._encode_subwords(text)

        result = self._scale_prefix(4)
        unk_id = self.special_tokens["[UNK]"]
        result.extend(self.token_to_id.get(f"[A:{feature}]", unk_id) for feature in features)
        return result

    # Alias legacy para código externo que llamaba directamente este método.
    def _encode_ast(self, text: str) -> List[int]:
        if not self.allow_lossy_ast:
            raise ValueError(
                "_encode_ast es destructivo y está deshabilitado. "
                "Use extract_ast_features() o active allow_lossy_ast explícitamente."
            )
        return self._encode_ast_lossy(text)

    def _legacy_ast_pattern_to_code(self, pattern: str) -> str:
        """Reconstrucción aproximada solo para inspeccionar checkpoints antiguos."""

        state = self._ast_decode_state
        indent = "    " * state["indent"]
        if pattern.startswith("FUNC:"):
            match = re.search(r"FUNC:(\d+)args", pattern)
            nargs = int(match.group(1)) if match else 1
            name = f"func_{state['func_count']}"
            state["func_count"] += 1
            state["indent"] += 1
            args = ", ".join(f"arg_{index}" for index in range(nargs))
            return f"\n{indent}def {name}({args}):\n"
        if pattern == "LOOP:for":
            state["indent"] += 1
            return f"\n{indent}for item in iterable:\n"
        if pattern == "LOOP:while":
            state["indent"] += 1
            return f"\n{indent}while condition:\n"
        if pattern == "COND:if":
            state["indent"] += 1
            return f"\n{indent}if condition:\n"
        if pattern == "NODE:Return":
            return f"{'    ' * state['indent']}return result\n"
        if pattern == "NODE:Pass":
            return f"{'    ' * state['indent']}pass\n"
        return ""

    # ------------------------------------------------------------------
    # Diagnóstico
    # ------------------------------------------------------------------

    def assert_roundtrip(
        self,
        texts: Iterable[str],
        *,
        adaptive: bool = True,
        preferred_scale: Optional[int] = None,
    ) -> None:
        """Falla con un diff compacto si cualquier texto no hace roundtrip exacto."""

        for index, text in enumerate(texts):
            ids = self.encode(text, adaptive=adaptive, preferred_scale=preferred_scale)
            decoded = self.decode(ids, strict=True)
            if decoded != text:
                mismatch = next(
                    (
                        position
                        for position, (left, right) in enumerate(zip(text, decoded))
                        if left != right
                    ),
                    min(len(text), len(decoded)),
                )
                raise AssertionError(
                    f"Roundtrip falló en muestra {index}, posición {mismatch}:\n"
                    f"original={text[max(0, mismatch-30):mismatch+30]!r}\n"
                    f"decoded ={decoded[max(0, mismatch-30):mismatch+30]!r}"
                )

    def token_debug(self, ids: Sequence[int]) -> List[str]:
        return [self.id_to_token.get(int(token_id), f"<INVALID_ID:{token_id}>") for token_id in ids]

    # ------------------------------------------------------------------
    # Helpers internos
    # ------------------------------------------------------------------

    @staticmethod
    def _payload(token: str, prefix: str) -> Optional[str]:
        if token.startswith(prefix) and token.endswith("]"):
            return token[len(prefix) : -1]
        return None

    def _cache_get(self, text: str) -> Optional[Tuple[int, ...]]:
        if self.cache_size <= 0:
            return None
        value = self._bpe_cache.get(text)
        if value is not None:
            self._bpe_cache.move_to_end(text)
        return value

    def _cache_put(self, text: str, ids: Sequence[int]) -> None:
        if self.cache_size <= 0:
            return
        self._bpe_cache[text] = tuple(ids)
        self._bpe_cache.move_to_end(text)
        while len(self._bpe_cache) > self.cache_size:
            self._bpe_cache.popitem(last=False)


if __name__ == "__main__":
    print("NeoToken V2 cargado. Use NeoToken().load('neotoken_22k.pkl').")
